In [40]:
import pandas as pd
import numpy as np
import os

In [41]:
Semi_processed = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/'
Processed = '/storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/'

In [42]:
!ls /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/

AnatomicalEntity_ChemicalEntity_Monarch.csv
BiologicalProcess_AnatomicalEntity_Monarch.csv
BiologicalProcess_CellularComponent_Monarch.csv
BiologicalProcess_MolecularActivity_Monarch.csv
BiologicalProcess_MolecularEntity_Monarch.csv
BiologicalProcess_Protein_Monarch.csv
CellularComponent_BiologicalProcess_Monarch.csv
CellularComponent_MolecularActivity_Monarch.csv
Disease_AnatomicalEntity_Monarch.csv
Disease_BiologicalProcess_Monarch.csv
Disease_CellularComponent_Monarch.csv
Disease_ChemicalEntity_Monarch.csv
Disease_MolecularActivity_Monarch.csv
Disease_MolecularEntity_Monarch.csv
Disease_Phenotype_human_MONARCH.csv
GENE_GENE_HUMAN_HUMAN_MONARCH.csv
Gene_Human_Anatomy_MONARCH.csv
Gene_Human_BiologicalProcess_MONARCH.csv
Gene_Human_CellularComponent_MONARCH.csv
Gene_Human_MolecularActivity_MONARCH.csv
Gene_Human_Phenotype_MONARCH.csv
Human_AnatomicalEntity_BiologicalProcess_Monarch.csv
Human_AnatomicalEntity_CellularComponent_Monarch.csv
Hum_AnatomicalEntity_AnatomicalEntity_MONARCH.cs

In [43]:
!ls /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Yeast

Gene_Yeast_Anatomy_MONARCH.csv
Gene_Yeast_BiologicalProcess_MONARCH.csv
Gene_Yeast_CellularComponent_MONARCH.csv
Gene_Yeast_MolecularActivity_MONARCH.csv


In [44]:
import os
import re
import numpy as np
import pandas as pd
from glob import glob

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : root folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"

# ── Derived input paths (do not edit below this line) ────────────────────────
ENS2NCBI_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
PUBCHEM_PKL_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.paraquet")
PUBCHEM_DB_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv")
DO_PATH           = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
MONDO_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/DO/MONDO_ID_Name_coll_from_Monarch.csv")
UNIPROT_MAP_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/UNIPROT_NCBI_ALL_MAPPED.csv")
UNIPROT_REC_PATH  = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/Uniprot_ID_Recname.csv")
MONARCH_PKL_PATH  = os.path.join(BASE_PATH, "monarchkg/monarch-kg-denormalized-edges.pkl")
UBERON_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/uberon/Uberon_ID_Name_non_dup.csv")

OUT_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/"
# ── Output sub-directories (one per species) ──────────────────────────────────
OUT_HUMAN  = os.path.join(OUT_PATH, "Human/")
OUT_MOUSE  = os.path.join(OUT_PATH, "Mouse/")
OUT_CELE   = os.path.join(OUT_PATH, "Celegans/")
OUT_DROSO  = os.path.join(OUT_PATH, "Drosophila/")
OUT_ZEBRA  = os.path.join(OUT_PATH, "Zebrafish/")

OUT_YEAST  = os.path.join(OUT_PATH, "Yeast/")

for d in [OUT_HUMAN, OUT_MOUSE, OUT_CELE, OUT_DROSO, OUT_ZEBRA, OUT_YEAST]:
    os.makedirs(d, exist_ok=True)

print("Paths configured successfully.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output root : {OUT_PATH}")

Paths configured successfully.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output root : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/


In [45]:
PR_Onto = pd.read_csv(f'{BASE_PATH}databases_for_mapping/protein_onto/promapping.txt', sep = '\t',header =None)
# PR_Onto[1] = PR_Onto[1].str.replace('UniProtKB:', '', regex=False)
PR_Onto_dict = dict(zip(PR_Onto[0],PR_Onto[1]))
# PR_Onto_dict

In [46]:
# ── 1a. ENSEMBL <-> NCBI gene symbol crosswalk ───────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))  # {symbol: ENS_ID}
del ENS_2NCBI
print(f"Symbol -> Ensembl: {len(NCBI_2ENS__dict):,}")

Symbol -> Ensembl: 40,940


In [47]:
# ── 1b. NCBI Human gene_info ──────────────────────────────────────────────────
# Monarch uses gene symbols (strings) as gene IDs, not GeneIDs (integers).
# NCBI_ALL_GENEname_dict: Symbol -> description
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID'] = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict   = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))

# HGNC ID -> Symbol (used for HGNC-prefixed gene IDs in Monarch)
def extract_hgnc(value):
    match = re.search(r'HGNC:(HGNC:\d+)', str(value))
    return match.group(1) if match else None
NCBI_ALL_GENE['HGNC'] = NCBI_ALL_GENE['dbXrefs'].apply(extract_hgnc)
NCBI_ALL_GENE_HGNC = NCBI_ALL_GENE.dropna(subset=['HGNC'])
NCBI_ALL_GENE_HGNC_dict = dict(zip(NCBI_ALL_GENE_HGNC['HGNC'], NCBI_ALL_GENE_HGNC['Symbol']))

print(f"NCBI Human genes: {len(NCBI_ALL_GENEname_dict):,}")
print(f"HGNC -> Symbol:   {len(NCBI_ALL_GENE_HGNC_dict):,}")

NCBI Human genes: 193,352
HGNC -> Symbol:   44,076


In [48]:
from io import StringIO

filename = f'{BASE_PATH}databases_for_mapping/pubchem/CID-MeSH'
with open(filename, 'r') as f:
    lines = f.readlines()

fixed_lines = []
for line in lines:
    # Remove any trailing newline for processing.
    line = line.rstrip('\n')
    parts = line.split('\t')
    # If there are more than 2 parts, merge all fields after the first.
    if len(parts) > 2:
        # Merge parts[1:] with a single space between them.
        fixed_line = parts[0] + "\t" + " ".join(parts[1:])
        fixed_lines.append(fixed_line)
    else:
        fixed_lines.append(line)

# Join the fixed lines back into a single string.
data_fixed = "\n".join(fixed_lines)

# Now read the corrected data with pandas.
mesh_2_Pubchem = pd.read_csv(StringIO(data_fixed), sep='\t', header = None)
mesh_2_Pubchem_dict = dict(zip(mesh_2_Pubchem[1],mesh_2_Pubchem[0]))
mesh_2_Pubchem_dict # 'Acetylcarnitine': 7045767,
# mesh_2_Pubchem

{'Acetylcarnitine': 7045767,
 'monoisopropanolamine': 4,
 'Dinitrochlorobenzene': 6,
 '9-ethyladenine': 7,
 '2,3-dihydroxy-3-methylpentanoic acid': 8,
 'inositol 1-phosphate': 107737,
 'ethylene dichloride': 11,
 '1,2,4-trichlorobenzene': 13,
 'Dihydrotestosterone': 44403751,
 '6-aminohexanoic acid cyclic dimer': 16,
 '2,3-Diketogulonic Acid': 53477844,
 '2,3-dihydroxybenzoic acid': 54675818,
 '2,3-dihydroxyphenylpropionic acid': 20,
 'alpha-aceto-alpha-hydroxybutyrate': 21,
 'C(alpha)-formylglycine': 443424,
 'chloroacetaldehyde': 85157,
 'Ethylene Chlorohydrin': 34,
 '3-isopropylmalic acid': 36,
 'keto-pantoyllactone': 39,
 'alpha-hydroxyglutarate': 5460200,
 'tartronic acid': 45,
 'alpha-keto-beta-methylvaleric acid': 4298878,
 'alpha-ketoglutaramate': 152239,
 'alpha-ketoisovalerate': 5204641,
 'provitamin C': 159793,
 'Ketoglutaric Acids': 90475487,
 'alpha-ketobutyric acid': 3593277,
 '2-phosphoglycerate': 59,
 '2,3-Diphosphoglycerate': 9548671,
 '2,5-dichloro-2,5-cyclohexadiene-

### MONDO

In [17]:
MONDO = pd.read_csv(f'{BASE_PATH}databases_for_mapping/DO/MONDO_ID_Name_coll_from_Monarch.csv')
MONDO
MONDO_dict = dict(zip(MONDO['NAME'],MONDO['ID']))
# MONDO_dict

In [26]:
DO_All_File = pd.read_csv(f'{BASE_PATH}databases_for_mapping/DO/All_DO_ref_files.csv')
DOID_disname_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
DOID_disAltID_dict = dict(zip(DO_All_File['label'], DO_All_File['xrefs']))

DO_All_File
# DOID_disAltID_dict
# DOID_disname_dict

,id,label,xrefs,subClassOf
0,DOID:0001816,angiosarcoma,ICDO:9120/3|MESH:D006394|NCI:C3088|NCI:C9275|S...,vascular cancer
1,DOID:0002116,pterygium,UMLS_CUI:C0033999,corneal disease
2,DOID:0014667,disease of metabolism,ICD10CM:E88.9|ICD9CM:277.9|MESH:D008659|NCI:C3...,disease
3,DOID:0040001,shrimp allergy,NaN,crustacean allergy
4,DOID:0040002,aspirin allergy,SNOMEDCT_US_2023_03_01:293586001|UMLS_CUI:C000...,drug allergy
...,...,...,...,...
11799,DOID:2848,melancholic depression,NaN,NaN
11800,DOID:3773,chordoid glioma,NaN,NaN
11801,DOID:5209,struma ovarii,NaN,NaN
11802,DOID:6496,extraskeletal myxoid chondrosarcoma,NaN,NaN


In [34]:
pubchem =pd.read_csv(f'{BASE_PATH}databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv')
pubchem.head()
pubchem_CHEBI = pubchem[~pubchem['CHEBI_ID'].isna()]
pubchem_CHEBI
Chebi2PC_dict = dict(zip(pubchem_CHEBI['CHEBI_ID'], pubchem_CHEBI['ID']))
# Extract the first three key-value pairs
first_three = dict(list(Chebi2PC_dict.items())[:3])
first_three

/tmp/ipykernel_2678861/1621929337.py:1: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem =pd.read_csv(f'{BASE_PATH}databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv')


{'CHEBI:63707': 56588024, 'CHEBI:202475': 56589567, 'CHEBI:202941': 56589577}

In [13]:
sgd = pd.read_csv(f'{BASE_PATH}databases_for_mapping/sgd/SGD_SystematicName_to_SGDID.csv')
sgd_dict = dict(zip(sgd['sgd_id'],sgd['gene_name']))
# sgd_dict

sgd = pd.read_csv(
    f'{BASE_PATH}databases_for_mapping/sgd/SGD_features.tab',
    sep='\t',
    header=None,
    usecols=[3, 4, 5],          # systematic_name, gene_name, description
    names=['Systematic_name', 'Gene_name', 'Description'],
    dtype=str
)
sgd = sgd[~sgd['Description'].isna()]
# Drop rows without a systematic name (YIL064W-style)
# sgd = sgd[sgd['Systematic_name'].str.match(r'^Y[A-Z]{2}\d+[CW]', na=False)]
sgd
yeast_sys2name_dict = dict(zip(sgd['Systematic_name'], sgd['Gene_name']))
yeast_sys2desc_dict = dict(zip(sgd['Systematic_name'], sgd['Description']))
yeast_sys2desc_dict_2 = dict(zip(sgd['Gene_name'], sgd['Description']))

In [206]:
sgd = pd.read_csv(
    f'{BASE_PATH}databases_for_mapping/SGD_features.tab',
    sep='\t',
    header=None,
    # usecols=[3, 4, 5],          # systematic_name, gene_name, description
    # names=['Systematic_name', 'Gene_name', 'Description'],
    dtype=str
)
sgd

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,S000350094,ORF,Verified,YDL204W-A,NaN,NaN,chromosome 4,NaN,4,94133,94285,W,NaN,2024-05-28,2024-05-28,"Protein of unknown function, translated but no..."
1,S000350094,CDS,NaN,NaN,NaN,NaN,YDL204W-A,NaN,4,94133,94285,W,NaN,2024-05-28,2024-05-28,NaN
2,S000350095,ORF,Verified,YFR035W-A,NaN,NaN,chromosome 6,NaN,6,226260,226550,W,NaN,2024-05-28,2024-05-28,Protein of unknown function; added to genome a...
3,S000350095,CDS,NaN,NaN,NaN,NaN,YFR035W-A,NaN,6,226260,226550,W,NaN,2024-05-28,2024-05-28,NaN
4,S000001326,ORF,Verified,YIL064W,EFM4,SEE1,chromosome 9,NaN,9,242027,242716,W,NaN,2024-05-28,2024-05-28,Lysine methyltransferase; involved in the dime...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16454,S000350097,CDS,NaN,NaN,NaN,NaN,YMR106W-A,NaN,13,480924,481187,W,NaN,2024-05-28,2024-05-28,NaN
16455,S000350098,ORF,Verified,YNL040C-A,NaN,NaN,chromosome 14,NaN,14,552558,552478,C,NaN,2024-05-28,2024-05-28,Protein of unknown function; deletion confers ...
16456,S000350098,CDS,NaN,NaN,NaN,NaN,YNL040C-A,NaN,14,552558,552478,C,NaN,2024-05-28,2024-05-28,NaN
16457,S000350099,ORF,Verified,YNL155C-A,NaN,NaN,chromosome 14,NaN,14,342135,341911,C,NaN,2024-05-28,2024-05-28,Protein of unknown function; well conserved ac...


In [49]:
# ── 2a. PubChem CID -> IUPAC name and SMILES ──────────────────────────────────
Pubchem = pd.read_parquet(PUBCHEM_PKL_PATH)
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
del Pubchem
print("PubChem CID dicts loaded")

PubChem CID dicts loaded


In [ ]:
Pubchem

In [49]:
DESIRED_COLS = [
    'Head', 'Relation', 'Tail', 'Head_type', 'Relation_type', 'Tail_type', 'Relation_Source',
    'KG_Source', 'Head_Detail_name', 'Tail_Detail_name','Head_IUPAC_name', 'Tail_IUPAC_name',
    'Head_taxon_name', 'Tail_taxon_name',
    'Head_ID_IS', 'Tail_ID_IS', 'Head_name', 'Tail_name',
    'Head_Orignal', 'Species_Species'
]

def select_cols(df):
    return df[[c for c in DESIRED_COLS if c in df.columns]]


In [50]:
wanted_cols = [
    'Head',
    'Relation',
    'Tail',
    'Head_type',
    'Tail_type',
    'Relation_type',
    'Relation_Source',
    'KG_Source',
    'Head_ID_IS',
    'Tail_ID_IS',
    'Head_IUPAC_name',
    'Head_Detail_name',
    'Tail_Detail_name'
    'Tail_IUPAC_name',
]



In [51]:

def save(df, filename):
    """Save to OUT_PATH and print confirmation."""
    path = os.path.join(Processed, filename)
    df.to_csv(path, index=None)
    print(f"Saved {len(df):,} rows -> {path}")


print("Helper functions defined.")

Helper functions defined.


In [52]:
DESIRED_COLS = [
    'Head',
    'Relation',
    'Tail',
    'Head_type',
    'Relation_type',
    'Tail_type',
    'Relation_Source',
    'KG_Source',
    'Head_Detail_name',
    'Tail_Detail_name',
    'Head_IUPAC_name',
    'Tail_IUPAC_name',
    'Head_taxon_name',
    'Tail_taxon_name',
    'Head_ID_IS',
    'Tail_ID_IS',
    'Head_name',
    'Tail_name',
    'Head_Orignal',
    'Species'
]


def select_cols(df):
    return df[[c for c in DESIRED_COLS if c in df.columns]]

In [53]:
OUT_PATH

'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/'

# Human

In [73]:
Human_Ana_Chemi = pd.read_csv(
    f'{Semi_processed}Human/AnatomicalEntity_ChemicalEntity_Monarch.csv'
)

# Rename columns
Human_Ana_Chemi = Human_Ana_Chemi.rename(columns={
    'Head_name': 'Head_Detail_name',
    'Tail_name': 'Tail_Detail_name',
    'Head_type_clean': 'Head_type',
    'Tail_type_clean': 'Tail_type'
})

# Remove duplicate columns
Human_Ana_Chemi = Human_Ana_Chemi.loc[
    :,
    ~Human_Ana_Chemi.columns.duplicated()
]

# Map PubChem CID → IUPAC
Human_Ana_Chemi['Tail_IUPAC_name'] = (
    Human_Ana_Chemi['Tail']
    .astype(str)
    .map(Pubchem_IUPAC_CID_Dict)
)

# Keep only desired columns that exist
Human_Ana_Chemi = select_cols(Human_Ana_Chemi)
Human_Ana_Chemi['Tail_ID_IS'] = 'Pubchem_ID'
Human_Ana_Chemi['Species'] = 'Homo sapiens'
Human_Ana_Chemi

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Tail_IUPAC_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species_Species,Species
0,UBERON:0001752,AnatomicalEntity_ChemicalEntity,14781,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,enamel,hydroxylapatite,pentacalcium;hydroxide;triphosphate,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens
1,UBERON:0001786,AnatomicalEntity_ChemicalEntity,5280899,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,fovea centralis,zeaxanthin,"(1R)-4-[(1E,3E,5E,7E,9E,11E,13E,15E,17E)-18-[(...",NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens
2,UBERON:0001786,AnatomicalEntity_ChemicalEntity,5281243,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,fovea centralis,lutein,"(1R)-4-[(1E,3E,5E,7E,9E,11E,13E,15E,17E)-18-[(...",NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens
3,UBERON:0001866,AnatomicalEntity_ChemicalEntity,638072,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,sebum,squalene,"(6E,10E,14E,18E)-2,6,10,15,19,23-hexamethyltet...",NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens
4,UBERON:0001970,AnatomicalEntity_ChemicalEntity,5280352,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,bile,bilirubin IXalpha,3-[2-[[3-(2-carboxyethyl)-5-[(Z)-(3-ethenyl-4-...,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens
5,UBERON:0002242,AnatomicalEntity_ChemicalEntity,446715,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,nucleus pulposus,keratan sulfate,"[(2R,3S,4S,5R,6R)-4-[(2S,3R,4R,5S,6R)-3-acetam...",NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens
6,UBERON:0002280,AnatomicalEntity_ChemicalEntity,10112,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,otolith,calcium carbonate,calcium;carbonate,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens
7,UBERON:0034948,AnatomicalEntity_ChemicalEntity,280,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,carbon dioxide in respiratory system,carbon dioxide,,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens
8,UBERON:4000115,AnatomicalEntity_ChemicalEntity,14781,AnatomicalEntity,related_to,ChemicalEntity,infores:uberon,mineralized bone tissue,hydroxylapatite,pentacalcium;hydroxide;triphosphate,NaN,NaN,UBERON,Pubchem_ID,NaN,Homo sapiens


In [74]:
save(Human_Ana_Chemi, 'Human/Human_Anatomy_Chemical.csv')

Saved 9 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/Human_Anatomy_Chemical.csv


In [ ]:
## 

In [111]:
BiologicalProcess_AnatomicalEntity = pd.read_csv(f'{Semi_processed}Human/BiologicalProcess_AnatomicalEntity_Monarch.csv')

BiologicalProcess_AnatomicalEntity = BiologicalProcess_AnatomicalEntity.rename(columns={
    'Head_name': 'Head_Detail_name',
    'Tail_name': 'Tail_Detail_name',
    'Head_type_clean': 'Head_type',
    'Tail_type_clean': 'Tail_type'
})

# Remove duplicate columns
BiologicalProcess_AnatomicalEntity = BiologicalProcess_AnatomicalEntity.loc[
    :,
    ~BiologicalProcess_AnatomicalEntity.columns.duplicated()
]
# Keep only desired columns that exist
BiologicalProcess_AnatomicalEntity = select_cols(BiologicalProcess_AnatomicalEntity)
BiologicalProcess_AnatomicalEntity['Head_ID_IS'] = 'Quick_GO'
BiologicalProcess_AnatomicalEntity['Tail_ID_IS'] = 'UBERON_ID'
BiologicalProcess_AnatomicalEntity['Species'] = 'Homo sapiens'


BiologicalProcess_AnatomicalEntity

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species_Species,Species
0,GO:0060525,BiologicalProcess_AnatomicalEntity,UBERON:0004179,BiologicalProcess,related_to,AnatomicalEntity,infores:go,prostate glandular acinus development,prostate glandular acinus,NaN,NaN,Quick_GO,UBERON_ID,NaN,Homo sapiens
1,GO:0060526,BiologicalProcess_AnatomicalEntity,UBERON:0004179,BiologicalProcess,related_to,AnatomicalEntity,infores:go,prostate glandular acinus morphogenesis,prostate glandular acinus,NaN,NaN,Quick_GO,UBERON_ID,NaN,Homo sapiens
2,GO:0060532,BiologicalProcess_AnatomicalEntity,UBERON:0001956,BiologicalProcess,related_to,AnatomicalEntity,infores:go,bronchus cartilage development,cartilage of bronchus,NaN,NaN,Quick_GO,UBERON_ID,NaN,Homo sapiens
3,GO:0060533,BiologicalProcess_AnatomicalEntity,UBERON:0001956,BiologicalProcess,related_to,AnatomicalEntity,infores:go,bronchus cartilage morphogenesis,cartilage of bronchus,NaN,NaN,Quick_GO,UBERON_ID,NaN,Homo sapiens
4,GO:0060534,BiologicalProcess_AnatomicalEntity,UBERON:0003604,BiologicalProcess,related_to,AnatomicalEntity,infores:go,trachea cartilage development,trachea cartilage,NaN,NaN,Quick_GO,UBERON_ID,NaN,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1295,GO:0060483,BiologicalProcess_AnatomicalEntity,UBERON:0004884,BiologicalProcess,related_to,AnatomicalEntity,infores:go,lobar bronchus mesenchyme development,lobar bronchus mesenchyme,NaN,NaN,Quick_GO,UBERON_ID,NaN,Homo sapiens
1296,GO:0060485,BiologicalProcess_AnatomicalEntity,UBERON:0003104,BiologicalProcess,related_to,AnatomicalEntity,infores:go,mesenchyme development,mesenchyme,NaN,NaN,Quick_GO,UBERON_ID,NaN,Homo sapiens
1297,GO:0060512,BiologicalProcess_AnatomicalEntity,UBERON:0002367,BiologicalProcess,related_to,AnatomicalEntity,infores:go,prostate gland morphogenesis,prostate gland,NaN,NaN,Quick_GO,UBERON_ID,NaN,Homo sapiens
1298,GO:0060513,BiologicalProcess_AnatomicalEntity,UBERON:0003820,BiologicalProcess,related_to,AnatomicalEntity,infores:go,prostatic bud formation,prostate bud,NaN,NaN,Quick_GO,UBERON_ID,NaN,Homo sapiens


In [112]:
save(BiologicalProcess_AnatomicalEntity, 'Human/BiologicalProcess_AnatomicalEntity.csv')

Saved 1,300 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/BiologicalProcess_AnatomicalEntity.csv


In [ ]:
## 

In [113]:
BiologicalProcess_CellularComponent = pd.read_csv(f'{Semi_processed}Human/BiologicalProcess_CellularComponent_Monarch.csv')

BiologicalProcess_CellularComponent = BiologicalProcess_CellularComponent.rename(columns={
    'Head_name': 'Head_Detail_name',
    'Tail_name': 'Tail_Detail_name',
    'Head_type_clean': 'Head_type',
    'Tail_type_clean': 'Tail_type'
})

# Remove duplicate columns
BiologicalProcess_CellularComponent = BiologicalProcess_CellularComponent.loc[
    :,
    ~BiologicalProcess_CellularComponent.columns.duplicated()
]
# Keep only desired columns that exist
BiologicalProcess_CellularComponent = select_cols(BiologicalProcess_CellularComponent)
BiologicalProcess_CellularComponent['Tail_ID_IS'] = 'Quick_GO'
BiologicalProcess_CellularComponent['Head_ID_IS'] = 'Quick_GO'
BiologicalProcess_CellularComponent['Species'] = 'Homo sapiens'

BiologicalProcess_CellularComponent

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species_Species,Species
0,GO:0060830,BiologicalProcess_CellularComponent,GO:0005929,BiologicalProcess,related_to,CellularComponent,infores:go,ciliary receptor clustering involved in smooth...,cilium,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
1,GO:0060988,BiologicalProcess_CellularComponent,GO:0060987,BiologicalProcess,related_to,CellularComponent,infores:go,lipid tube assembly,lipid tube,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
2,GO:0060997,BiologicalProcess_CellularComponent,GO:0043197,BiologicalProcess,related_to,CellularComponent,infores:go,dendritic spine morphogenesis,dendritic spine,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
3,GO:0061024,BiologicalProcess_CellularComponent,GO:0016020,BiologicalProcess,related_to,CellularComponent,infores:go,membrane organization,membrane,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
4,GO:0061025,BiologicalProcess_CellularComponent,GO:0016020,BiologicalProcess,related_to,CellularComponent,infores:go,membrane fusion,membrane,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1484,GO:0060121,BiologicalProcess_CellularComponent,GO:0032420,BiologicalProcess,related_to,CellularComponent,infores:go,vestibular receptor cell stereocilium organiza...,stereocilium,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
1485,GO:0060151,BiologicalProcess_CellularComponent,GO:0005777,BiologicalProcess,related_to,CellularComponent,infores:go,peroxisome localization,peroxisome,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
1486,GO:0060155,BiologicalProcess_CellularComponent,GO:0042827,BiologicalProcess,related_to,CellularComponent,infores:go,platelet dense granule organization,platelet dense granule,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
1487,GO:0060271,BiologicalProcess_CellularComponent,GO:0005929,BiologicalProcess,related_to,CellularComponent,infores:go,cilium assembly,cilium,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens


In [114]:
save(BiologicalProcess_CellularComponent, 'Human/BiologicalProcess_CellularComponent.csv')

Saved 1,489 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/BiologicalProcess_CellularComponent.csv


In [ ]:
##

In [115]:
BiologicalProcess_MolecularActivity = pd.read_csv(f'{Semi_processed}Human/BiologicalProcess_MolecularActivity_Monarch.csv')
BiologicalProcess_MolecularActivity = BiologicalProcess_MolecularActivity.rename(columns={
    'Head_name': 'Head_Detail_name',
    'Tail_name': 'Tail_Detail_name',
    'Head_type_clean': 'Head_type',
    'Tail_type_clean': 'Tail_type'
})

# Remove duplicate columns
BiologicalProcess_MolecularActivity = BiologicalProcess_MolecularActivity.loc[
    :,
    ~BiologicalProcess_MolecularActivity.columns.duplicated()
]
# Keep only desired columns that exist
BiologicalProcess_MolecularActivity = select_cols(BiologicalProcess_MolecularActivity)
BiologicalProcess_MolecularActivity['Tail_ID_IS'] = 'Quick_GO'
BiologicalProcess_MolecularActivity['Head_ID_IS'] = 'Quick_GO'
BiologicalProcess_MolecularActivity['Relation'] = 'BiologicalProcess_MolecularFunction'
BiologicalProcess_MolecularActivity['Tail_type'] = 'MolecularFunction'
BiologicalProcess_MolecularActivity['Species'] = 'Homo sapiens'

BiologicalProcess_MolecularActivity

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species_Species,Species
0,GO:0060549,BiologicalProcess_MolecularFunction,GO:0042132,BiologicalProcess,related_to,MolecularFunction,infores:go,"regulation of fructose 1,6-bisphosphate 1-phos...","fructose 1,6-bisphosphate 1-phosphatase activity",NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
1,GO:0060550,BiologicalProcess_MolecularFunction,GO:0042132,BiologicalProcess,related_to,MolecularFunction,infores:go,"positive regulation of fructose 1,6-bisphospha...","fructose 1,6-bisphosphate 1-phosphatase activity",NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
2,GO:0060558,BiologicalProcess_MolecularFunction,GO:0004498,BiologicalProcess,related_to,MolecularFunction,infores:go,regulation of calcidiol 1-monooxygenase activity,calcidiol 1-monooxygenase activity,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
3,GO:0060559,BiologicalProcess_MolecularFunction,GO:0004498,BiologicalProcess,related_to,MolecularFunction,infores:go,positive regulation of calcidiol 1-monooxygena...,calcidiol 1-monooxygenase activity,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
4,GO:0060584,BiologicalProcess_MolecularFunction,GO:0004666,BiologicalProcess,related_to,MolecularFunction,infores:go,regulation of prostaglandin-endoperoxide synth...,prostaglandin-endoperoxide synthase activity,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
829,GO:0060314,BiologicalProcess_MolecularFunction,GO:0005219,BiologicalProcess,related_to,MolecularFunction,infores:go,regulation of ryanodine-sensitive calcium-rele...,ryanodine-sensitive calcium-release channel ac...,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
830,GO:0060315,BiologicalProcess_MolecularFunction,GO:0005219,BiologicalProcess,related_to,MolecularFunction,infores:go,negative regulation of ryanodine-sensitive cal...,ryanodine-sensitive calcium-release channel ac...,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
831,GO:0060316,BiologicalProcess_MolecularFunction,GO:0005219,BiologicalProcess,related_to,MolecularFunction,infores:go,positive regulation of ryanodine-sensitive cal...,ryanodine-sensitive calcium-release channel ac...,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
832,GO:0060380,BiologicalProcess_MolecularFunction,GO:0043047,BiologicalProcess,related_to,MolecularFunction,infores:go,regulation of single-stranded telomeric DNA bi...,single-stranded telomeric DNA binding,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens


In [116]:
save(BiologicalProcess_MolecularActivity, 'Human/BiologicalProcess_MolecularFunction.csv')

Saved 834 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/BiologicalProcess_MolecularFunction.csv


In [117]:
##

In [126]:
# BiologicalProcess_MolecularEntity = pd.read_csv(f'{Semi_processed}Human/BiologicalProcess_MolecularEntity_Monarch.csv')
# BiologicalProcess_MolecularEntity

In [ ]:
##

In [120]:
# BiologicalProcess_Protein = pd.read_csv(f'{Semi_processed}Human/BiologicalProcess_Protein_Monarch.csv')  # Very less in number only 27 from that also3-4 nont converting to uniprot and 4-6 or other species

# BiologicalProcess_Protein = BiologicalProcess_Protein.rename(columns={
#     'Head_name': 'Head_Detail_name',
#     'Tail_name': 'Tail_Detail_name',
#     'Head_type_clean': 'Head_type',
#     'Tail_type_clean': 'Tail_type'
# })

# # Remove duplicate columns
# BiologicalProcess_Protein = BiologicalProcess_Protein.loc[
#     :,
#     ~BiologicalProcess_Protein.columns.duplicated()
# ]
# # Keep only desired columns that exist

# BiologicalProcess_Protein['Tail'] = BiologicalProcess_Protein['Tail'].map(PR_Onto_dict)
# BiologicalProcess_Protein = select_cols(BiologicalProcess_Protein)
# BiologicalProcess_Protein['Tail_ID_IS'] = 'Quick_GO'
# BiologicalProcess_Protein['Species'] = 'Uniprot_ID'

# BiologicalProcess_Protein

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species_Species,Species
0,GO:0061847,BiologicalProcess_Protein,UniProtKB:Q9PU41,BiologicalProcess,related_to,Protein,infores:go,response to cholecystokinin,cholecystokinin,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID
1,GO:0070162,BiologicalProcess_Protein,UniProtKB:Q60994,BiologicalProcess,related_to,Protein,infores:go,adiponectin secretion,adiponectin,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID
2,GO:0070459,BiologicalProcess_Protein,UniProtKB:P14676,BiologicalProcess,related_to,Protein,infores:go,prolactin secretion,prolactin,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID
3,GO:0099191,BiologicalProcess_Protein,UniProtKB:P25429,BiologicalProcess,related_to,Protein,infores:go,trans-synaptic signaling by BDNF,brain-derived neurotrophic factor,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID
4,GO:1901142,BiologicalProcess_Protein,ZFIN:ZDB-GENE-980526-110,BiologicalProcess,related_to,Protein,infores:go,insulin metabolic process,insulin gene translation product,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID
5,GO:1901143,BiologicalProcess_Protein,ZFIN:ZDB-GENE-980526-110,BiologicalProcess,related_to,Protein,infores:go,insulin catabolic process,insulin gene translation product,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID
6,GO:1904391,BiologicalProcess_Protein,UniProtKB:Q02011,BiologicalProcess,related_to,Protein,infores:go,response to ciliary neurotrophic factor,ciliary neurotrophic factor,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID
7,GO:0002001,BiologicalProcess_Protein,UniProtKB:Q9TSZ1,BiologicalProcess,related_to,Protein,infores:go,renin secretion into blood stream,renin,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID
8,GO:0002415,BiologicalProcess_Protein,UniProtKB:P15083,BiologicalProcess,related_to,Protein,infores:go,immunoglobulin transcytosis in epithelial cell...,polymeric immunoglobulin receptor,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID
9,GO:0003050,BiologicalProcess_Protein,UniProtKB:P18908,BiologicalProcess,related_to,Protein,infores:go,regulation of systemic arterial blood pressure...,atrial natriuretic factor,NaN,NaN,GO,Quick_GO,NaN,Uniprot_ID


In [124]:
CellularComponent_BiologicalProcess = pd.read_csv(f'{Semi_processed}Human/CellularComponent_BiologicalProcess_Monarch.csv')
CellularComponent_BiologicalProcess = CellularComponent_BiologicalProcess.rename(columns={
    'Head_name': 'Head_Detail_name',
    'Tail_name': 'Tail_Detail_name',
    'Head_type_clean': 'Head_type',
    'Tail_type_clean': 'Tail_type'
})

# Remove duplicate columns
CellularComponent_BiologicalProcess = CellularComponent_BiologicalProcess.loc[
    :,
    ~CellularComponent_BiologicalProcess.columns.duplicated()
]
# Keep only desired columns that exist
CellularComponent_BiologicalProcess = select_cols(CellularComponent_BiologicalProcess)
CellularComponent_BiologicalProcess['Tail_ID_IS'] = 'Quick_GO'
CellularComponent_BiologicalProcess['Head_ID_IS'] = 'Quick_GO'

CellularComponent_BiologicalProcess['Species'] = 'Homo sapiens'
CellularComponent_BiologicalProcess

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species_Species,Species
0,GO:0061694,CellularComponent_BiologicalProcess,GO:0019634,CellularComponent,related_to,BiologicalProcess,infores:go,alpha-D-ribose 1-methylphosphonate 5-triphosph...,organic phosphonate metabolic process,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
1,GO:0061702,CellularComponent_BiologicalProcess,GO:0050729,CellularComponent,related_to,BiologicalProcess,infores:go,canonical inflammasome complex,positive regulation of inflammatory response,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
2,GO:0061773,CellularComponent_BiologicalProcess,GO:0000183,CellularComponent,related_to,BiologicalProcess,infores:go,eNoSc complex,rDNA heterochromatin formation,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
3,GO:0061773,CellularComponent_BiologicalProcess,GO:0042149,CellularComponent,related_to,BiologicalProcess,infores:go,eNoSc complex,cellular response to glucose starvation,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
4,GO:0061773,CellularComponent_BiologicalProcess,GO:0045892,CellularComponent,related_to,BiologicalProcess,infores:go,eNoSc complex,negative regulation of DNA-templated transcrip...,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
318,GO:0046536,CellularComponent_BiologicalProcess,GO:0007549,CellularComponent,related_to,BiologicalProcess,infores:go,dosage compensation complex,sex-chromosome dosage compensation,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
319,GO:0046816,CellularComponent_BiologicalProcess,GO:0046794,CellularComponent,related_to,BiologicalProcess,infores:go,virion transport vesicle,transport of virus,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
320,GO:0055037,CellularComponent_BiologicalProcess,GO:0032456,CellularComponent,related_to,BiologicalProcess,infores:go,recycling endosome,endocytic recycling,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens
321,GO:0060076,CellularComponent_BiologicalProcess,GO:0098976,CellularComponent,related_to,BiologicalProcess,infores:go,excitatory synapse,excitatory chemical synaptic transmission,NaN,NaN,Quick_GO,Quick_GO,NaN,Homo sapiens


In [127]:
save(CellularComponent_BiologicalProcess, 'Human/CellularComponent_BiologicalProcess.csv')

Saved 323 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/CellularComponent_BiologicalProcess.csv


In [130]:
##

In [133]:
CellularComponent_MolecularActivity = pd.read_csv(f'{Semi_processed}Human/CellularComponent_MolecularActivity_Monarch.csv')
CellularComponent_MolecularActivity = CellularComponent_MolecularActivity.rename(columns={
    'Head_name': 'Head_Detail_name',
    'Tail_name': 'Tail_Detail_name',
    'Head_type_clean': 'Head_type',
    'Tail_type_clean': 'Tail_type'
})

# Remove duplicate columns
CellularComponent_MolecularActivity = CellularComponent_MolecularActivity.loc[
    :,
    ~CellularComponent_MolecularActivity.columns.duplicated()
]
# Keep only desired columns that exist
CellularComponent_MolecularActivity = select_cols(CellularComponent_MolecularActivity)
CellularComponent_MolecularActivity['Tail_ID_IS'] = 'Quick_GO'
CellularComponent_MolecularActivity['Head_ID_IS'] = 'Quick_GO'

CellularComponent_MolecularActivity['Species'] = 'Homo sapiens'
CellularComponent_MolecularActivity

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,GO:0061671,CellularComponent_MolecularActivity,GO:0097177,CellularComponent,related_to,MolecularActivity,infores:go,Cbp3p-Cbp6 complex,mitochondrial ribosome binding,NaN,NaN,Quick_GO,Quick_GO,Homo sapiens
1,GO:0061672,CellularComponent_MolecularActivity,GO:0036374,CellularComponent,related_to,MolecularActivity,infores:go,glutathione hydrolase complex,glutathione hydrolase activity,NaN,NaN,Quick_GO,Quick_GO,Homo sapiens
2,GO:0061694,CellularComponent_MolecularActivity,GO:0061693,CellularComponent,related_to,MolecularActivity,infores:go,alpha-D-ribose 1-methylphosphonate 5-triphosph...,alpha-D-ribose 1-methylphosphonate 5-triphosph...,NaN,NaN,Quick_GO,Quick_GO,Homo sapiens
3,GO:0061695,CellularComponent_MolecularActivity,GO:0016772,CellularComponent,related_to,MolecularActivity,infores:go,"transferase complex, transferring phosphorus-c...","transferase activity, transferring phosphorus-...",NaN,NaN,Quick_GO,Quick_GO,Homo sapiens
4,GO:0061696,CellularComponent_MolecularActivity,GO:0005179,CellularComponent,related_to,MolecularActivity,infores:go,pituitary gonadotropin complex,hormone activity,NaN,NaN,Quick_GO,Quick_GO,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
487,GO:0048179,CellularComponent_MolecularActivity,GO:0017002,CellularComponent,related_to,MolecularActivity,infores:go,activin receptor complex,activin receptor activity,NaN,NaN,Quick_GO,Quick_GO,Homo sapiens
488,GO:0048269,CellularComponent_MolecularActivity,GO:0004478,CellularComponent,related_to,MolecularActivity,infores:go,methionine adenosyltransferase complex,methionine adenosyltransferase activity,NaN,NaN,Quick_GO,Quick_GO,Homo sapiens
489,GO:0048476,CellularComponent_MolecularActivity,GO:0008821,CellularComponent,related_to,MolecularActivity,infores:go,Holliday junction resolvase complex,crossover junction DNA endonuclease activity,NaN,NaN,Quick_GO,Quick_GO,Homo sapiens
490,GO:0048492,CellularComponent_MolecularActivity,GO:0016984,CellularComponent,related_to,MolecularActivity,infores:go,ribulose bisphosphate carboxylase complex,ribulose-bisphosphate carboxylase activity,NaN,NaN,Quick_GO,Quick_GO,Homo sapiens


In [134]:
save(CellularComponent_MolecularActivity, 'Human/CellularComponent_MolecularActivity.csv')

Saved 492 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/CellularComponent_MolecularActivity.csv


In [19]:
# MONDO

In [29]:
Disease_AnatomicalEntity = pd.read_csv(f'{Semi_processed}Human/Disease_AnatomicalEntity_Monarch.csv')
# Disease_AnatomicalEntity['Head_Alt'] = Disease_AnatomicalEntity['Head_name'].map(MONDO_dict)
Disease_AnatomicalEntity['Head_DO_name'] = (
    Disease_AnatomicalEntity['Head_name']
    .map(DOID_disname_dict)
    .fillna(Disease_AnatomicalEntity['Head_name'].map(MONDO_dict))
    .fillna(Disease_AnatomicalEntity['Head'])
)
Disease_AnatomicalEntity[['Head', 'Head_DO_name']] = Disease_AnatomicalEntity[['Head_DO_name', 'Head']]

Disease_AnatomicalEntity['Head_ID_IS'] = np.where(
    Disease_AnatomicalEntity['Head'].isna(),
    None,
    np.where(
        Disease_AnatomicalEntity['Head'].str.startswith('DOID'),
        'DOID',
        'MESH'
    )
)

Disease_AnatomicalEntity

,Head,Tail,Relation_type,Relation_Source,KGSource,Head_name,Head_type,Head_ID_IS,Head_taxon,Head_taxon_label,...,Tail_ID_IS,Tail_taxon,Tail_taxon_label,Head_taxon_name,Tail_taxon_name,Head_type_clean,Tail_type_clean,Relation,Species_Species,Head_DO_name
0,MONDO:0024876,UBERON:0000304,disease_has_location,infores:mondo,MonarchKG,tendon sheath disorder,Disease,MESH,NaN,NaN,...,UBERON,NaN,NaN,NaN,NaN,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0024876
1,MONDO:0024877,UBERON:0002411,disease_has_location,infores:mondo,MonarchKG,clitoris neoplasm,Disease,MESH,NaN,NaN,...,UBERON,NaN,NaN,NaN,NaN,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0024877
2,MONDO:0024888,UBERON:0003074,disease_has_location,infores:mondo,MonarchKG,mesonephric neoplasm,Disease,MESH,NaN,NaN,...,UBERON,NaN,NaN,NaN,NaN,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0024888
3,MONDO:0025514,UBERON:0003532,disease_has_location,infores:mondo,MonarchKG,livedoid vasculopathy,Disease,MESH,NaN,NaN,...,UBERON,NaN,NaN,NaN,NaN,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0025514
4,MONDO:0033839,UBERON:0001684,disease_has_location,infores:mondo,MonarchKG,osteoradionecrosis of the mandible,Disease,MESH,NaN,NaN,...,UBERON,NaN,NaN,NaN,NaN,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0033839
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1011,MONDO:0024635,UBERON:0002108,disease_has_location,infores:mondo,MonarchKG,small intestine disorder,Disease,MESH,NaN,NaN,...,UBERON,NaN,NaN,NaN,NaN,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0024635
1012,MONDO:0024636,UBERON:0005983,disease_has_location,infores:mondo,MonarchKG,inflammation of heart layer,Disease,MESH,NaN,NaN,...,UBERON,NaN,NaN,NaN,NaN,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0024636
1013,MONDO:0024643,UBERON:0002349,disease_has_location,infores:mondo,MonarchKG,myocardial disorder,Disease,MESH,NaN,NaN,...,UBERON,NaN,NaN,NaN,NaN,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0024643
1014,MONDO:0024645,UBERON:0003693,disease_has_location,infores:mondo,MonarchKG,retroperitoneal neoplasm,Disease,MESH,NaN,NaN,...,UBERON,NaN,NaN,NaN,NaN,Disease,AnatomicalEntity,Disease_AnatomicalEntity,NaN,MONDO:0024645


In [54]:
Monarch_BP_CE = pd.read_csv(f'{Semi_processed}Human/Human_BiologicalProcess_ChemicalEntity_Monarch.csv')
# Monarch_BP_CE.columns = Monarch_BP_CE.columns.str.lower()

Monarch_BP_CE['head_id_is'] = 'Quick_GO'
# Monarch_BP_CE['tail_id_is'] = 'PubChem'
Monarch_BP_CE['relation'] = 'BiologicalProcess_ChemicalEntity'
Monarch_BP_CE['head_type'] = 'BiologicalProcess'
Monarch_BP_CE['tail_type'] = 'ChemicalEntity'

Monarch_BP_CE['Tail_ID'] = Monarch_BP_CE['Tail'].map(Chebi2PC_dict)

Monarch_BP_CE = Monarch_BP_CE[~Monarch_BP_CE['Tail_ID'].isna()]
Monarch_BP_CE['Tail_ID'] = Monarch_BP_CE['Tail_ID'].astype(str).str.replace(r'\.0$', '', regex=True)
Monarch_BP_CE[['Tail', 'Tail_ID']] = Monarch_BP_CE[['Tail_ID', 'Tail']]
Monarch_BP_CE
Monarch_BP_CE

print(f"Monarch BP→CE: {len(Monarch_BP_CE):,} rows")
Monarch_BP_CE

Monarch BP→CE: 2,662 rows


,Head,Tail,Relation_type,Relation_Source,KGSource,Head_name,Head_type,Head_ID_IS,Head_taxon,Head_taxon_label,...,Tail_taxon_name,Head_type_clean,Tail_type_clean,Relation,Species_Species,head_id_is,relation,head_type,tail_type,Tail_ID
2,GO:0061370,6013,related_to,infores:go,MonarchKG,testosterone biosynthetic process,BiologicalProcess,GO,NaN,NaN,...,NaN,BiologicalProcess,ChemicalEntity,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:17347
3,GO:0061528,5460542,related_to,infores:go,MonarchKG,aspartate secretion,BiologicalProcess,GO,NaN,NaN,...,NaN,BiologicalProcess,ChemicalEntity,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:35391
6,GO:0061538,6971009,related_to,infores:go,MonarchKG,"histamine secretion, neurotransmission",BiologicalProcess,GO,NaN,NaN,...,NaN,BiologicalProcess,ChemicalEntity,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:57595
7,GO:0061539,21680226,related_to,infores:go,MonarchKG,octopamine secretion,BiologicalProcess,GO,NaN,NaN,...,NaN,BiologicalProcess,ChemicalEntity,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:58025
8,GO:0061545,5249538,related_to,infores:go,MonarchKG,tyramine secretion,BiologicalProcess,GO,NaN,NaN,...,NaN,BiologicalProcess,ChemicalEntity,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:327995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4545,GO:0055075,813,related_to,infores:go,MonarchKG,potassium ion homeostasis,BiologicalProcess,GO,NaN,NaN,...,NaN,BiologicalProcess,ChemicalEntity,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:29103
4546,GO:0055078,923,related_to,infores:go,MonarchKG,sodium ion homeostasis,BiologicalProcess,GO,NaN,NaN,...,NaN,BiologicalProcess,ChemicalEntity,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:29101
4553,GO:0055092,1107,related_to,infores:go,MonarchKG,sterol homeostasis,BiologicalProcess,GO,NaN,NaN,...,NaN,BiologicalProcess,ChemicalEntity,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:15889
4555,GO:0055129,6971047,related_to,infores:go,MonarchKG,L-proline biosynthetic process,BiologicalProcess,GO,NaN,NaN,...,NaN,BiologicalProcess,ChemicalEntity,BiologicalProcess_ChemicalEntity,NaN,Quick_GO,BiologicalProcess_ChemicalEntity,BiologicalProcess,ChemicalEntity,CHEBI:60039


In [55]:
save(Monarch_BP_CE, 'Human/BiologicalProcess_ChemicalEntity.csv')

Saved 2,662 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/BiologicalProcess_ChemicalEntity.csv


In [56]:
#

In [58]:
Human_Chemical_Chemical_Monarch = pd.read_csv(f'{Semi_processed}Human/Human_Chemical_Chemical_Monarch.csv')

save(Human_Chemical_Chemical_Monarch, 'Human/ChemicalEntity_ChemicalEntity.csv')
Human_Chemical_Chemical_Monarch

Saved 11,407 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/ChemicalEntity_ChemicalEntity.csv


,Head,Tail,Relation_type,Relation_Source,KGSource,Head_name,Head_type,Head_ID_IS,Head_taxon,Head_taxon_label,...,Tail_taxon,Tail_taxon_label,Head_taxon_name,Tail_taxon_name,Head_type_clean,Tail_type_clean,Relation,Species_Species,Head_ID,Tail_ID
0,135398636,135823789,related_to,infores:chebi,MonarchKG,"guanosine 3',5'-bis(diphosphate)(5-)",ChemicalEntity,Pubchem,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77828,CHEBI:58215
1,135398636,135398637,related_to,infores:chebi,MonarchKG,"guanosine 3',5'-bis(diphosphate)(5-)",ChemicalEntity,Pubchem,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77828,CHEBI:17633
2,25201902,10100906,related_to,infores:chebi,MonarchKG,delphinidin 3-O-beta-D-glucoside-5-O-beta-D-gl...,ChemicalEntity,Pubchem,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77838,CHEBI:55455
3,86289381,54740357,related_to,infores:chebi,MonarchKG,stipitatate(1-),ChemicalEntity,Pubchem,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77842,CHEBI:57587
4,86289381,20501,related_to,infores:chebi,MonarchKG,stipitatate(1-),ChemicalEntity,Pubchem,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77842,CHEBI:15957
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11402,86289372,128861,related_to,infores:chebi,MonarchKG,cyanidin 3-O-(2-O-beta-D-glucuronosyl)-beta-D-...,ChemicalEntity,Pubchem,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77824,CHEBI:27843
11403,86289372,50909829,related_to,infores:chebi,MonarchKG,cyanidin 3-O-(2-O-beta-D-glucuronosyl)-beta-D-...,ChemicalEntity,Pubchem,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77824,CHEBI:61317
11404,86289374,46878438,related_to,infores:chebi,MonarchKG,luteolin 7-O-[(beta-D-glucosyluronate)-(1->2)-...,ChemicalEntity,Pubchem,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77825,CHEBI:58678
11405,86289375,45266663,related_to,infores:chebi,MonarchKG,mugineate(1-),ChemicalEntity,Pubchem,NaN,NaN,...,NaN,NaN,Homo sapiens,Homo sapiens,ChemicalEntity,ChemicalEntity,ChemicalEntity_ChemicalEntity,NaN,CHEBI:77826,CHEBI:58505


In [ ]:
#

In [62]:
Monarch_Disease_Anatomy = pd.read_csv(f'{Semi_processed}Human/Disease_AnatomicalEntity_Monarch.csv')
Monarch_Disease_Anatomy


save(Monarch_Disease_Anatomy, 'Human/Disease_AnatomicalEntity_Monarch.csv')

Saved 1,016 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Human/Disease_AnatomicalEntity_Monarch.csv


In [59]:
Semi_processed

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/'

In [ ]:
'/storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Human/Disease_AnatomicalEntity_Monarch.csv'

# Yeast

In [140]:
Gene_Yeast_Anatomy = pd.read_csv(
    f'{Semi_processed}Yeast/Gene_Yeast_Anatomy_MONARCH.csv'
)
Gene_Yeast_Anatomy

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Tail_name,Species_Species


In [188]:
Gene_Yeast_BiologicalProcess = pd.read_csv(
    f'{Semi_processed}Yeast/Gene_Yeast_BiologicalProcess_MONARCH.csv'
)
Gene_Yeast_BiologicalProcess['Head'] = Gene_Yeast_BiologicalProcess['Head'].str.replace('SGD:', '', regex=False)

Gene_Yeast_BiologicalProcess['Head'] = Gene_Yeast_BiologicalProcess['Head'].map(sgd_dict).fillna(Gene_Yeast_BiologicalProcess['Head_name'])
Gene_Yeast_BiologicalProcess = Gene_Yeast_BiologicalProcess[~Gene_Yeast_BiologicalProcess['Head'].isna()]
Gene_Yeast_BiologicalProcess['Head_Detail_name'] = Gene_Yeast_BiologicalProcess['Head'].map(yeast_sys2desc_dict).fillna(Gene_Yeast_BiologicalProcess['Head_name'])


Gene_Yeast_BiologicalProcess = Gene_Yeast_BiologicalProcess.rename(columns={
    'Tail_name': 'Tail_Detail_name',
})
# Remove duplicate columns
Gene_Yeast_BiologicalProcess = Gene_Yeast_BiologicalProcess.loc[
    :,
    ~Gene_Yeast_BiologicalProcess.columns.duplicated()
]
# Keep only desired columns that exist
Gene_Yeast_BiologicalProcess = select_cols(Gene_Yeast_BiologicalProcess)
Gene_Yeast_BiologicalProcess['Species'] = 'Saccharomyces cerevisiae'
Gene_Yeast_BiologicalProcess

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,YGR149W,Gene_BiologicalProcess,GO:0090640,Gene,acts_upstream_of_or_within,BiologicalProcess,infores:sgd,glycerophosphocholine acyltransferase,phosphatidylcholine biosynthesis from sn-glyce...,Saccharomyces cerevisiae S288C,NaN,SGD,GO,GPC1,Saccharomyces cerevisiae
1,YOR175C,Gene_BiologicalProcess,GO:0090640,Gene,acts_upstream_of_or_within,BiologicalProcess,infores:sgd,LCA1|LPT1|lysophospholipid acyltransferase|SLC4,phosphatidylcholine biosynthesis from sn-glyce...,Saccharomyces cerevisiae S288C,NaN,SGD,GO,ALE1,Saccharomyces cerevisiae
2,YGR149W,Gene_BiologicalProcess,GO:0036151,Gene,acts_upstream_of_or_within,BiologicalProcess,infores:sgd,glycerophosphocholine acyltransferase,phosphatidylcholine acyl-chain remodeling,Saccharomyces cerevisiae S288C,NaN,SGD,GO,GPC1,Saccharomyces cerevisiae
3,YML030W,Gene_BiologicalProcess,GO:0033617,Gene,acts_upstream_of_or_within,BiologicalProcess,infores:sgd,AIM31|respiratory supercomplex assembly factor...,mitochondrial cytochrome c oxidase assembly,Saccharomyces cerevisiae S288C,NaN,SGD,GO,RCF1,Saccharomyces cerevisiae
4,YNL032W,Gene_BiologicalProcess,GO:0071543,Gene,actively_involved_in,BiologicalProcess,infores:sgd,OCA3|putative tyrosine protein phosphatase SIW14,diphosphoinositol polyphosphate metabolic process,Saccharomyces cerevisiae S288C,NaN,SGD,GO,SIW14,Saccharomyces cerevisiae
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38887,Q0120,Gene_BiologicalProcess,GO:0006122,Gene,actively_involved_in,BiologicalProcess,infores:go-central,intron-encoded RNA maturase bI4,"mitochondrial electron transport, ubiquinol to...",Saccharomyces cerevisiae S288C,NaN,SGD,GO,BI4,Saccharomyces cerevisiae
38888,YCR093W,Gene_BiologicalProcess,GO:0000288,Gene,actively_involved_in,BiologicalProcess,infores:go-central,CCR4-NOT core subunit CDC39|NOT1|ROS1|SMD6,"nuclear-transcribed mRNA catabolic process, de...",Saccharomyces cerevisiae S288C,NaN,SGD,GO,CDC39,Saccharomyces cerevisiae
38889,YGR009C,Gene_BiologicalProcess,GO:0006906,Gene,actively_involved_in,BiologicalProcess,infores:go-central,HSS7,vesicle fusion,Saccharomyces cerevisiae S288C,NaN,SGD,GO,SEC9,Saccharomyces cerevisiae
38890,YDR144C,Gene_BiologicalProcess,GO:0031505,Gene,actively_involved_in,BiologicalProcess,infores:go-central,aspartyl protease|YPS2,fungal-type cell wall organization,Saccharomyces cerevisiae S288C,NaN,SGD,GO,MKC7,Saccharomyces cerevisiae


In [189]:
save(Gene_Yeast_BiologicalProcess, 'Yeast/Gene_Yeast_BiologicalProcess.csv')

Saved 38,892 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Yeast/Gene_Yeast_BiologicalProcess.csv


In [ ]:
###

In [190]:
Gene_Yeast_CellularComponent = pd.read_csv(
    f'{Semi_processed}Yeast/Gene_Yeast_CellularComponent_MONARCH.csv'
)

Gene_Yeast_CellularComponent['Head'] = (
    Gene_Yeast_CellularComponent['Head']
    .str.replace('SGD:', '', regex=False)
)

Gene_Yeast_CellularComponent['Head'] = (
    Gene_Yeast_CellularComponent['Head']
    .map(sgd_dict)
    .fillna(Gene_Yeast_CellularComponent['Head_name'])
)

Gene_Yeast_CellularComponent = Gene_Yeast_CellularComponent[
    ~Gene_Yeast_CellularComponent['Head'].isna()
]

Gene_Yeast_CellularComponent['Head_Detail_name'] = (
    Gene_Yeast_CellularComponent['Head']
    .map(yeast_sys2desc_dict).fillna(Gene_Yeast_CellularComponent['Head_name'])
)

Gene_Yeast_CellularComponent = Gene_Yeast_CellularComponent.rename(columns={
    'Tail_name': 'Tail_Detail_name',
})

# Remove duplicate columns
Gene_Yeast_CellularComponent = Gene_Yeast_CellularComponent.loc[
    :,
    ~Gene_Yeast_CellularComponent.columns.duplicated()
]

# Keep only desired columns that exist
Gene_Yeast_CellularComponent = select_cols(Gene_Yeast_CellularComponent)

Gene_Yeast_CellularComponent['Species'] = 'Saccharomyces cerevisiae'

Gene_Yeast_CellularComponent

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,YIL031W,Gene_CellularComponent,GO:0000785,Gene,located_in,CellularComponent,infores:sgd,SMT4|SUMO protease ULP2,chromatin,Saccharomyces cerevisiae S288C,NaN,SGD,GO,ULP2,Saccharomyces cerevisiae
1,YFR016C,Gene_CellularComponent,GO:0005934,Gene,located_in,CellularComponent,infores:sgd,AIP5,cellular bud tip,Saccharomyces cerevisiae S288C,NaN,SGD,GO,AIP5,Saccharomyces cerevisiae
2,YFR016C,Gene_CellularComponent,GO:0005935,Gene,located_in,CellularComponent,infores:sgd,AIP5,cellular bud neck,Saccharomyces cerevisiae S288C,NaN,SGD,GO,AIP5,Saccharomyces cerevisiae
3,YNL278W,Gene_CellularComponent,GO:0005935,Gene,located_in,CellularComponent,infores:sgd,CAF120,cellular bud neck,Saccharomyces cerevisiae S288C,NaN,SGD,GO,CAF120,Saccharomyces cerevisiae
4,YGL242C,Gene_CellularComponent,GO:0005737,Gene,is_active_in,CellularComponent,infores:sgd,ankyrin repeat-containing protein ANK1,cytoplasm,Saccharomyces cerevisiae S288C,NaN,SGD,GO,ANK1,Saccharomyces cerevisiae
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60083,YKL220C,Gene_CellularComponent,GO:0005886,Gene,is_active_in,CellularComponent,infores:go-central,ferric/cupric-chelate reductase,plasma membrane,Saccharomyces cerevisiae S288C,NaN,SGD,GO,FRE2,Saccharomyces cerevisiae
60084,YLR029C,Gene_CellularComponent,GO:0022625,Gene,part_of,CellularComponent,infores:go-central,60S ribosomal protein eL15 RPL15A|eL15|L13A|L1...,cytosolic large ribosomal subunit,Saccharomyces cerevisiae S288C,NaN,SGD,GO,RPL15A,Saccharomyces cerevisiae
60085,YOR018W,Gene_CellularComponent,GO:0005886,Gene,is_active_in,CellularComponent,infores:go-central,ART4,plasma membrane,Saccharomyces cerevisiae S288C,NaN,SGD,GO,ROD1,Saccharomyces cerevisiae
60086,YMR213W,Gene_CellularComponent,GO:0000974,Gene,part_of,CellularComponent,infores:go-central,NTC85,Prp19 complex,Saccharomyces cerevisiae S288C,NaN,SGD,GO,CEF1,Saccharomyces cerevisiae


In [191]:
save(Gene_Yeast_CellularComponent, 'Yeast/Gene_Yeast_CellularComponent.csv')

Saved 60,088 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Yeast/Gene_Yeast_CellularComponent.csv


In [ ]:
##

In [210]:
Gene_Yeast_MolecularActivity = pd.read_csv(
    f'{Semi_processed}Yeast/Gene_Yeast_MolecularActivity_MONARCH.csv'
)
Gene_Yeast_MolecularActivity['Head'] = (
    Gene_Yeast_MolecularActivity['Head']
    .str.replace('SGD:', '', regex=False)
)

Gene_Yeast_MolecularActivity['Head'] = (
    Gene_Yeast_MolecularActivity['Head']
    .map(sgd_dict)
    .fillna(Gene_Yeast_MolecularActivity['Head_name'])
)

Gene_Yeast_MolecularActivity = Gene_Yeast_MolecularActivity[
    ~Gene_Yeast_MolecularActivity['Head'].isna()
]

Gene_Yeast_MolecularActivity['Head_Detail_name'] = (
    Gene_Yeast_MolecularActivity['Head']
    .map(yeast_sys2desc_dict)
    .fillna(Gene_Yeast_MolecularActivity['Head_name'])
)

Gene_Yeast_MolecularActivity = Gene_Yeast_MolecularActivity.rename(columns={
    'Tail_name': 'Tail_Detail_name',
})

# Remove duplicate columns
Gene_Yeast_MolecularActivity = Gene_Yeast_MolecularActivity.loc[
    :,
    ~Gene_Yeast_MolecularActivity.columns.duplicated()
]

# Keep only desired columns that exist
Gene_Yeast_MolecularActivity = select_cols(Gene_Yeast_MolecularActivity)

Gene_Yeast_MolecularActivity['Species'] = 'Saccharomyces cerevisiae'
Gene_Yeast_MolecularActivity['Relation'] = 'Gene_MolecularFunction'
Gene_Yeast_MolecularActivity['Tail_type'] = 'MolecularFunction'

Gene_Yeast_MolecularActivity

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,YNL032W,Gene_MolecularFunction,GO:0052845,Gene,enables,MolecularFunction,infores:sgd,OCA3|putative tyrosine protein phosphatase SIW14,"inositol-5-diphosphate-1,2,3,4,6-pentakisphosp...",Saccharomyces cerevisiae S288C,NaN,SGD,GO,SIW14,Saccharomyces cerevisiae
1,YGR178C,Gene_MolecularFunction,GO:0044877,Gene,enables,MolecularFunction,infores:sgd,MRS16,protein-containing complex binding,Saccharomyces cerevisiae S288C,NaN,SGD,GO,PBP1,Saccharomyces cerevisiae
2,YDL110C,Gene_MolecularFunction,GO:0098772,Gene,enables,MolecularFunction,infores:sgd,ADC17,molecular function regulator activity,Saccharomyces cerevisiae S288C,NaN,SGD,GO,TMA17,Saccharomyces cerevisiae
3,YDR291W,Gene_MolecularFunction,GO:0043138,Gene,enables,MolecularFunction,infores:sgd,ATP-dependent 3'-5' DNA helicase,3'-5' DNA helicase activity,Saccharomyces cerevisiae S288C,NaN,SGD,GO,HRQ1,Saccharomyces cerevisiae
4,YDR291W,Gene_MolecularFunction,GO:0042162,Gene,enables,MolecularFunction,infores:sgd,ATP-dependent 3'-5' DNA helicase,telomeric DNA binding,Saccharomyces cerevisiae S288C,NaN,SGD,GO,HRQ1,Saccharomyces cerevisiae
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37184,YOR008C,Gene_MolecularFunction,GO:0004888,Gene,enables,MolecularFunction,infores:go-central,HCS77|WSC1,transmembrane signaling receptor activity,Saccharomyces cerevisiae S288C,NaN,SGD,GO,SLG1,Saccharomyces cerevisiae
37185,YMR054W,Gene_MolecularFunction,GO:0051117,Gene,enables,MolecularFunction,infores:go-central,H(+)-transporting V0 sector ATPase subunit a,ATPase binding,Saccharomyces cerevisiae S288C,NaN,SGD,GO,STV1,Saccharomyces cerevisiae
37186,YOR210W,Gene_MolecularFunction,GO:0008270,Gene,enables,MolecularFunction,infores:go-central,ABC10-beta|DNA-directed RNA polymerase core su...,zinc ion binding,Saccharomyces cerevisiae S288C,NaN,SGD,GO,RPB10,Saccharomyces cerevisiae
37187,YDL083C,Gene_MolecularFunction,GO:0003723,Gene,enables,MolecularFunction,infores:go-central,40S ribosomal protein uS9 RPS16B|rp61R|S16B|S9...,RNA binding,Saccharomyces cerevisiae S288C,NaN,SGD,GO,RPS16B,Saccharomyces cerevisiae


In [211]:
save(Gene_Yeast_MolecularActivity, 'Yeast/Gene_Yeast_MolecularFunction.csv')

Saved 37,189 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Yeast/Gene_Yeast_MolecularFunction.csv


# celegans

In [230]:
celegans_ncbi = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Caenorhabditis_elegans.gene_info',sep = '\t')
celegans_ncbi['LocusTag'] = celegans_ncbi['LocusTag'].str.replace('CELE_', '', regex=False)
# Extract WormBase ID
celegans_ncbi['WormBase_ID'] = (
    celegans_ncbi['dbXrefs']
    .str.extract(r'WormBase:(WBGene\d+)')
)

# Keep only useful columns
celegans_ncbi_clean = celegans_ncbi[
    [
        'Symbol',
        'LocusTag',
        'WormBase_ID',
        'description'
    ]
]

# Remove duplicates
celegans_ncbi_clean = celegans_ncbi_clean.drop_duplicates()

# Remove rows with missing IDs
celegans_ncbi_clean = celegans_ncbi_clean[
    ~celegans_ncbi_clean['WormBase_ID'].isna()
]

celegans_ncbi_locustag_dict = dict(zip(celegans_ncbi_clean['WormBase_ID'],celegans_ncbi_clean['LocusTag']))
celegans_ncbi_locustag_desc_dict = dict(zip(celegans_ncbi_clean['LocusTag'],celegans_ncbi_clean['description']))
celegans_ncbi_locustag_dict

{'WBGene00022277': 'Y74C9A.3',
 'WBGene00022276': 'Y74C9A.2',
 'WBGene00022278': 'Y74C9A.4',
 'WBGene00022279': 'Y74C9A.5',
 'WBGene00021677': 'Y48G1C.4',
 'WBGene00021678': 'Y48G1C.5',
 'WBGene00021679': 'Y48G1C.6',
 'WBGene00021676': 'Y48G1C.1',
 'WBGene00004274': 'F53G12.1',
 'WBGene00004418': 'F53G12.10',
 'WBGene00018774': 'F53G12.9',
 'WBGene00018773': 'F53G12.8',
 'WBGene00000622': 'F53G12.7',
 'WBGene00004962': 'F53G12.6',
 'WBGene00003229': 'F53G12.5',
 'WBGene00000253': 'F56C11.1',
 'WBGene00004225': 'F56C11.2',
 'WBGene00018958': 'F56C11.6',
 'WBGene00018955': 'F56C11.3',
 'WBGene00021667': 'Y48G1BM.1',
 'WBGene00016903': 'C53D5.2',
 'WBGene00016905': 'C53D5.4',
 'WBGene00016902': 'C53D5.1',
 'WBGene00016906': 'C53D5.5',
 'WBGene00002079': 'Y48G1A.5',
 'WBGene00021660': 'Y48G1A.4',
 'WBGene00000917': 'Y48G1A.3',
 'WBGene00021661': 'Y48G1A.6',
 'WBGene00021658': 'Y48G1A.2',
 'WBGene00020091': 'R119.7',
 'WBGene00020089': 'R119.3',
 'WBGene00006385': 'R119.6',
 'WBGene00020090

In [274]:
Cele_AnatomicalEntity_AnatomicalEntity = pd.read_csv(
    f'{Semi_processed}Celegans/Cele_AnatomicalEntity_AnatomicalEntity_MONARCH.csv'
)

Cele_AnatomicalEntity_AnatomicalEntity = (
    Cele_AnatomicalEntity_AnatomicalEntity.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Cele_AnatomicalEntity_AnatomicalEntity = (
    Cele_AnatomicalEntity_AnatomicalEntity.loc[
        :,
        ~Cele_AnatomicalEntity_AnatomicalEntity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Cele_AnatomicalEntity_AnatomicalEntity = (
    select_cols(Cele_AnatomicalEntity_AnatomicalEntity)
)

Cele_AnatomicalEntity_AnatomicalEntity['Head_ID_IS'] = 'WormBase'
Cele_AnatomicalEntity_AnatomicalEntity['Tail_ID_IS'] = 'WormBase'

Cele_AnatomicalEntity_AnatomicalEntity['Relation'] = (
    'AnatomicalEntity_AnatomicalEntity'
)

Cele_AnatomicalEntity_AnatomicalEntity['Species'] = (
    'Caenorhabditis elegans'
)
save(Cele_AnatomicalEntity_AnatomicalEntity, 'Celegans/Cele_AnatomicalEntity_AnatomicalEntity.csv')
Cele_AnatomicalEntity_AnatomicalEntity

Saved 15,092 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Cele_AnatomicalEntity_AnatomicalEntity.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,WBbt:0000101,AnatomicalEntity_AnatomicalEntity,WBbt:0000100,AnatomicalEntity,subclass_of,AnatomicalEntity,infores:wbbt,Lineage,C. elegans anatomical entity,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,Caenorhabditis elegans
1,WBbt:0000102,AnatomicalEntity_AnatomicalEntity,WBbt:0000101,AnatomicalEntity,subclass_of,AnatomicalEntity,infores:wbbt,P0 nucleus,Lineage,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,Caenorhabditis elegans
2,WBbt:0000102,AnatomicalEntity_AnatomicalEntity,WBbt:0006803,AnatomicalEntity,subclass_of,AnatomicalEntity,infores:wbbt,P0 nucleus,Nucleus,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,Caenorhabditis elegans
3,WBbt:0000102,AnatomicalEntity_AnatomicalEntity,WBbt:0004422,AnatomicalEntity,related_to,AnatomicalEntity,infores:wbbt,P0 nucleus,P0,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,Caenorhabditis elegans
4,WBbt:0001001,AnatomicalEntity_AnatomicalEntity,WBbt:0006803,AnatomicalEntity,subclass_of,AnatomicalEntity,infores:wbbt,AB nucleus,Nucleus,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15087,WBbt:0008638,AnatomicalEntity_AnatomicalEntity,WBbt:0004148,AnatomicalEntity,related_to,AnatomicalEntity,infores:wbbt,Z4.appaaap nucleus hermaphrodite,Z4.appaaap hermaphrodite,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,Caenorhabditis elegans
15088,WBbt:0008639,AnatomicalEntity_AnatomicalEntity,WBbt:0003565,AnatomicalEntity,subclass_of,AnatomicalEntity,infores:wbbt,Z4.appaaaa nucleus hermaphrodite,Z4.appaaaa nucleus,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,Caenorhabditis elegans
15089,WBbt:0008639,AnatomicalEntity_AnatomicalEntity,WBbt:0004150,AnatomicalEntity,related_to,AnatomicalEntity,infores:wbbt,Z4.appaaaa nucleus hermaphrodite,Z4.appaaaa hermaphrodite,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,Caenorhabditis elegans
15090,WBbt:0008640,AnatomicalEntity_AnatomicalEntity,WBbt:0003565,AnatomicalEntity,subclass_of,AnatomicalEntity,infores:wbbt,Z4.appaaaa nucleus male,Z4.appaaaa nucleus,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,Caenorhabditis elegans


In [275]:
Cele_Gene_Pathway = pd.read_csv(
    f'{Semi_processed}Celegans/Cele_Gene_Pathway_MONARCH.csv'
)
Cele_Gene_Pathway['Head'] = Cele_Gene_Pathway['Head'].str.replace('WB:', '', regex=False)
Cele_Gene_Pathway['Head'] = Cele_Gene_Pathway['Head'].map(celegans_ncbi_locustag_dict)
Cele_Gene_Pathway['Head_Detail_name'] = Cele_Gene_Pathway['Head'].map(celegans_ncbi_locustag_desc_dict)


Cele_Gene_Pathway = (
    Cele_Gene_Pathway.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Cele_Gene_Pathway = (
    Cele_Gene_Pathway.loc[
        :,
        ~Cele_Gene_Pathway.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Cele_Gene_Pathway = (
    select_cols(Cele_Gene_Pathway)
)

Cele_Gene_Pathway['Head_ID_IS'] = 'WormBase'

Cele_Gene_Pathway['Relation'] = (
    'Gene_Pathway'
)

Cele_Gene_Pathway['Species'] = (
    'Caenorhabditis elegans'
)
Cele_Gene_Pathway

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,K03H1.13,Gene_Pathway,R-HSA-6787639,Gene,participates_in,Pathway,infores:reactome,L-fucokinase domain-containing protein,GDP-fucose biosynthesis,Caenorhabditis elegans,Homo sapiens,WormBase,Reactome,K03H1.13,Caenorhabditis elegans
1,W01A8.8,Gene_Pathway,R-HSA-381426,Gene,participates_in,Pathway,infores:reactome,DUF19 domain-containing protein,Regulation of Insulin-like Growth Factor (IGF)...,Caenorhabditis elegans,Homo sapiens,WormBase,Reactome,W01A8.8,Caenorhabditis elegans
2,W01A8.8,Gene_Pathway,R-HSA-8957275,Gene,participates_in,Pathway,infores:reactome,DUF19 domain-containing protein,Post-translational protein phosphorylation,Caenorhabditis elegans,Homo sapiens,WormBase,Reactome,W01A8.8,Caenorhabditis elegans
3,Y37A1B.17,Gene_Pathway,R-HSA-193648,Gene,participates_in,Pathway,infores:reactome,SH3 domain-containing GRB2-like protein;SH3 do...,NRAGE signals death through JNK,Caenorhabditis elegans,Homo sapiens,WormBase,Reactome,dnbp-1,Caenorhabditis elegans
4,Y37A1B.17,Gene_Pathway,R-HSA-416482,Gene,participates_in,Pathway,infores:reactome,SH3 domain-containing GRB2-like protein;SH3 do...,G alpha (12/13) signalling events,Caenorhabditis elegans,Homo sapiens,WormBase,Reactome,dnbp-1,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12693,Y59A8B.25,Gene_Pathway,R-HSA-6811440,Gene,participates_in,Pathway,infores:reactome,DUF727 domain-containing protein,Retrograde transport at the Trans-Golgi-Network,Caenorhabditis elegans,Homo sapiens,WormBase,Reactome,Y59A8B.25,Caenorhabditis elegans
12694,ZK863.9,Gene_Pathway,R-HSA-1236978,Gene,participates_in,Pathway,infores:reactome,C-type lectin domain-containing protein,Cross-presentation of soluble exogenous antige...,Caenorhabditis elegans,Homo sapiens,WormBase,Reactome,clec-224,Caenorhabditis elegans
12695,ZK863.9,Gene_Pathway,R-HSA-446203,Gene,participates_in,Pathway,infores:reactome,C-type lectin domain-containing protein,Asparagine N-linked glycosylation,Caenorhabditis elegans,Homo sapiens,WormBase,Reactome,clec-224,Caenorhabditis elegans
12696,ZK863.9,Gene_Pathway,R-HSA-5621480,Gene,participates_in,Pathway,infores:reactome,C-type lectin domain-containing protein,Dectin-2 family,Caenorhabditis elegans,Homo sapiens,WormBase,Reactome,clec-224,Caenorhabditis elegans


In [276]:
save(Cele_Gene_Pathway, 'Celegans/Cele_Gene_Pathway.csv')

Saved 12,698 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Cele_Gene_Pathway.csv


In [277]:
##

In [278]:
Cele_Phenotype_CellularComponent = pd.read_csv(
    f'{Semi_processed}Celegans/Cele_Phenotype_CellularComponent_Monarch.csv'
)

Cele_Phenotype_CellularComponent = (
    Cele_Phenotype_CellularComponent.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Cele_Phenotype_CellularComponent = (
    Cele_Phenotype_CellularComponent.loc[
        :,
        ~Cele_Phenotype_CellularComponent.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Cele_Phenotype_CellularComponent = (
    select_cols(Cele_Phenotype_CellularComponent)
)

Cele_Phenotype_CellularComponent['Head_ID_IS'] = 'WormBasePhenotype'
Cele_Phenotype_CellularComponent['Tail_ID_IS'] = 'Quick_GO'

Cele_Phenotype_CellularComponent['Relation'] = (
    'Phenotype_CellularComponent'
)
Cele_Phenotype_CellularComponent['Head_type'] = (
    'Phenotype'
)
Cele_Phenotype_CellularComponent['Tail_type'] = (
    'CellularComponent'
)

Cele_Phenotype_CellularComponent['Species'] = (
    'Caenorhabditis elegans'
)

save(
    Cele_Phenotype_CellularComponent,
    'Celegans/Cele_Phenotype_CellularComponent.csv'
)

Cele_Phenotype_CellularComponent

Saved 76 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Cele_Phenotype_CellularComponent.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,WBPhenotype:0000103,Phenotype_CellularComponent,GO:0044840,Phenotype,related_to,CellularComponent,infores:upheno,gut granule development variant,gut granule,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
1,WBPhenotype:0000161,Phenotype_CellularComponent,GO:0005634,Phenotype,related_to,CellularComponent,infores:upheno,nuclear rotation variant,nucleus,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
2,WBPhenotype:0000180,Phenotype_CellularComponent,GO:0030424,Phenotype,related_to,CellularComponent,infores:upheno,axon morphology variant,axon,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
3,WBPhenotype:0000181,Phenotype_CellularComponent,GO:0030424,Phenotype,related_to,CellularComponent,infores:upheno,axon trajectory variant,axon,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
4,WBPhenotype:0000200,Phenotype_CellularComponent,GO:0031012,Phenotype,related_to,CellularComponent,infores:upheno,pericellular component development variant,extracellular matrix,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71,WBPhenotype:0002519,Phenotype_CellularComponent,GO:0030424,Phenotype,related_to,CellularComponent,infores:upheno,axon length reduced,axon,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
72,WBPhenotype:0002591,Phenotype_CellularComponent,GO:0030425,Phenotype,related_to,CellularComponent,infores:upheno,dendrite length reduced,dendrite,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
73,WBPhenotype:0002617,Phenotype_CellularComponent,GO:0005829,Phenotype,related_to,CellularComponent,infores:upheno,cytosolic calcium levels increased,cytosol,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
74,WBPhenotype:0002618,Phenotype_CellularComponent,GO:0005829,Phenotype,related_to,CellularComponent,infores:upheno,cytosolic calcium levels decreased,cytosol,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans


In [279]:
##

In [280]:
Cele_Phenotype_ChemicalEntity = pd.read_csv(
    f'{Semi_processed}Celegans/Cele_Phenotype_ChemicalEntity_Monarch.csv'
)
Cele_Phenotype_ChemicalEntity = (
    Cele_Phenotype_ChemicalEntity.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Cele_Phenotype_ChemicalEntity = (
    Cele_Phenotype_ChemicalEntity.loc[
        :,
        ~Cele_Phenotype_ChemicalEntity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Cele_Phenotype_ChemicalEntity = (
    select_cols(Cele_Phenotype_ChemicalEntity)
)

Cele_Phenotype_ChemicalEntity['Head_ID_IS'] = 'WormBasePhenotype'
Cele_Phenotype_ChemicalEntity['Tail_ID_IS'] = 'PubChem'

Cele_Phenotype_ChemicalEntity['Relation'] = (
    'Phenotype_ChemicalEntity'
)
Cele_Phenotype_ChemicalEntity['Head_type'] = (
    'Phenotype'
)
Cele_Phenotype_ChemicalEntity['Tail_type'] = (
    'ChemicalEntity'
)

Cele_Phenotype_ChemicalEntity['Tail_IUPAC_name'] = (
    Cele_Phenotype_ChemicalEntity['Tail']
    .astype(str)
    .map(Pubchem_IUPAC_CID_Dict)
)

Cele_Phenotype_ChemicalEntity['Species'] = (
    'Caenorhabditis elegans'
)

save(
    Cele_Phenotype_ChemicalEntity,
    'Celegans/Cele_Phenotype_ChemicalEntity.csv'
)
Cele_Phenotype_ChemicalEntity

Saved 11 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Cele_Phenotype_ChemicalEntity.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Tail_IUPAC_name,Species
0,WBPhenotype:0000507,Phenotype_ChemicalEntity,187,Phenotype,related_to,ChemicalEntity,infores:upheno,acetylcholine levels variant,acetylcholine,Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,2-acetyloxyethyl(trimethyl)azanium,Caenorhabditis elegans
1,WBPhenotype:0000552,Phenotype_ChemicalEntity,119,Phenotype,related_to,ChemicalEntity,infores:upheno,GABA levels variant,gamma-aminobutyric acid,Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,4-aminobutanoic acid,Caenorhabditis elegans
2,WBPhenotype:0001574,Phenotype_ChemicalEntity,5957,Phenotype,related_to,ChemicalEntity,infores:upheno,ATP levels variant,ATP,Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,"[[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihy...",Caenorhabditis elegans
3,WBPhenotype:0001575,Phenotype_ChemicalEntity,5957,Phenotype,related_to,ChemicalEntity,infores:upheno,ATP levels reduced,ATP,Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,"[[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihy...",Caenorhabditis elegans
4,WBPhenotype:0001899,Phenotype_ChemicalEntity,124886,Phenotype,related_to,ChemicalEntity,infores:upheno,glutathione levels variant,glutathione,Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,(2S)-2-amino-5-[[(2R)-1-(carboxymethylamino)-1...,Caenorhabditis elegans
5,WBPhenotype:0002264,Phenotype_ChemicalEntity,5957,Phenotype,related_to,ChemicalEntity,infores:upheno,ATP levels increased,ATP,Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,"[[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihy...",Caenorhabditis elegans
6,WBPhenotype:0002350,Phenotype_ChemicalEntity,784,Phenotype,related_to,ChemicalEntity,infores:upheno,hydrogen peroxide levels increased,hydrogen peroxide,Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,hydrogen peroxide,Caenorhabditis elegans
7,WBPhenotype:0002351,Phenotype_ChemicalEntity,784,Phenotype,related_to,ChemicalEntity,infores:upheno,hydrogen peroxide levels reduced,hydrogen peroxide,Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,hydrogen peroxide,Caenorhabditis elegans
8,WBPhenotype:0002617,Phenotype_ChemicalEntity,271,Phenotype,related_to,ChemicalEntity,infores:upheno,cytosolic calcium levels increased,calcium(2+),Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,calcium(2+),Caenorhabditis elegans
9,WBPhenotype:0002618,Phenotype_ChemicalEntity,271,Phenotype,related_to,ChemicalEntity,infores:upheno,cytosolic calcium levels decreased,calcium(2+),Caenorhabditis elegans,NaN,WormBasePhenotype,PubChem,calcium(2+),Caenorhabditis elegans


In [281]:
##

In [283]:
Cele_PhenotypicFeature_BiologicalProcess = pd.read_csv(
    f'{Semi_processed}Celegans/Cele_PhenotypicFeature_BiologicalProcess_Monarch.csv'
)
Cele_PhenotypicFeature_BiologicalProcess = (
    Cele_PhenotypicFeature_BiologicalProcess.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Cele_PhenotypicFeature_BiologicalProcess = (
    Cele_PhenotypicFeature_BiologicalProcess.loc[
        :,
        ~Cele_PhenotypicFeature_BiologicalProcess.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Cele_PhenotypicFeature_BiologicalProcess = (
    select_cols(Cele_PhenotypicFeature_BiologicalProcess)
)

Cele_PhenotypicFeature_BiologicalProcess['Head_ID_IS'] = (
    'WormBasePhenotype'
)

Cele_PhenotypicFeature_BiologicalProcess['Tail_ID_IS'] = (
    'Quick_GO'
)

Cele_PhenotypicFeature_BiologicalProcess['Relation'] = (
    'Phenotype_BiologicalProcess'
)
Cele_PhenotypicFeature_BiologicalProcess['Head_type'] = (
    'Phenotype'
)
Cele_PhenotypicFeature_BiologicalProcess['Tail_type'] = (
    'BiologicalProcess'
)

Cele_PhenotypicFeature_BiologicalProcess['Species'] = (
    'Caenorhabditis elegans'
)

save(
    Cele_PhenotypicFeature_BiologicalProcess,
    'Celegans/Cele_PhenotypicFeature_BiologicalProcess.csv'
)

Cele_PhenotypicFeature_BiologicalProcess

Saved 488 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Cele_PhenotypicFeature_BiologicalProcess.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,WBPhenotype:0000005,Phenotype_BiologicalProcess,GO:0018991,Phenotype,related_to,BiologicalProcess,infores:upheno,hyperactive egg laying,egg-laying behavior,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
1,WBPhenotype:0000013,Phenotype_BiologicalProcess,GO:0043053,Phenotype,related_to,BiologicalProcess,infores:upheno,dauer defective,dauer entry,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
2,WBPhenotype:0000015,Phenotype_BiologicalProcess,GO:0050918,Phenotype,related_to,BiologicalProcess,infores:upheno,positive chemotaxis defective,positive chemotaxis,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
3,WBPhenotype:0000018,Phenotype_BiologicalProcess,GO:0043050,Phenotype,related_to,BiologicalProcess,infores:upheno,pharyngeal pumping increased,nematode pharyngeal pumping,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
4,WBPhenotype:0000019,Phenotype_BiologicalProcess,GO:0043050,Phenotype,related_to,BiologicalProcess,infores:upheno,pharyngeal pumping reduced,nematode pharyngeal pumping,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,WBPhenotype:0004022,Phenotype_BiologicalProcess,GO:0040011,Phenotype,related_to,BiologicalProcess,infores:upheno,amplitude of sinusoidal movement variant,locomotion,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
484,WBPhenotype:0004024,Phenotype_BiologicalProcess,GO:0040011,Phenotype,related_to,BiologicalProcess,infores:upheno,wavelength of movement variant,locomotion,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
485,WBPhenotype:0004025,Phenotype_BiologicalProcess,GO:0040011,Phenotype,related_to,BiologicalProcess,infores:upheno,velocity of movement variant,locomotion,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
486,WBPhenotype:0004030,Phenotype_BiologicalProcess,GO:0034606,Phenotype,related_to,BiologicalProcess,infores:upheno,male response to hermaphrodite variant,response to hermaphrodite contact,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans


In [284]:
Cele_PhenotypicFeature_MolecularActivity = pd.read_csv(
    f'{Semi_processed}Celegans/Cele_PhenotypicFeature_MolecularActivity_Monarch.csv'
)

Cele_PhenotypicFeature_MolecularActivity = (
    Cele_PhenotypicFeature_MolecularActivity.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Cele_PhenotypicFeature_MolecularActivity = (
    Cele_PhenotypicFeature_MolecularActivity.loc[
        :,
        ~Cele_PhenotypicFeature_MolecularActivity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Cele_PhenotypicFeature_MolecularActivity = (
    select_cols(Cele_PhenotypicFeature_MolecularActivity)
)

Cele_PhenotypicFeature_MolecularActivity['Head_ID_IS'] = (
    'WormBasePhenotype'
)

Cele_PhenotypicFeature_MolecularActivity['Tail_ID_IS'] = (
    'Quick_GO'
)

Cele_PhenotypicFeature_MolecularActivity['Relation'] = (
    'Phenotype_MolecularFunction'
)

Cele_PhenotypicFeature_MolecularActivity['Head_type'] = (
    'Phenotype'
)

Cele_PhenotypicFeature_MolecularActivity['Tail_type'] = (
    'MolecularFunction'
)

Cele_PhenotypicFeature_MolecularActivity['Species'] = (
    'Caenorhabditis elegans'
)

save(
    Cele_PhenotypicFeature_MolecularActivity,
    'Celegans/Cele_PhenotypicFeature_MolecularFunction.csv'
)

Cele_PhenotypicFeature_MolecularActivity

Saved 4 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Cele_PhenotypicFeature_MolecularFunction.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,WBPhenotype:0000124,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,infores:upheno,enzyme activity reduced,catalytic activity,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
1,WBPhenotype:0000125,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,infores:upheno,enzyme activity increased,catalytic activity,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
2,WBPhenotype:0000727,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,infores:upheno,enzyme activity variant,catalytic activity,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans
3,WBPhenotype:0001695,Phenotype_MolecularFunction,GO:0016757,Phenotype,related_to,MolecularFunction,infores:upheno,glycosyltransferase activity variant,glycosyltransferase activity,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,Caenorhabditis elegans


In [285]:
Gene_Cele_Anatomy = pd.read_csv(
    f'{Semi_processed}Celegans/Gene_Cele_Anatomy_MONARCH.csv'
)

Gene_Cele_Anatomy['Head'] = (
    Gene_Cele_Anatomy['Head']
    .str.replace('WB:', '', regex=False)
)

Gene_Cele_Anatomy['Head'] = (
    Gene_Cele_Anatomy['Head']
    .map(celegans_ncbi_locustag_dict)
)

Gene_Cele_Anatomy['Head_Detail_name'] = (
    Gene_Cele_Anatomy['Head']
    .map(celegans_ncbi_locustag_desc_dict)
)

Gene_Cele_Anatomy = (
    Gene_Cele_Anatomy.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Cele_Anatomy = (
    Gene_Cele_Anatomy.loc[
        :,
        ~Gene_Cele_Anatomy.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Cele_Anatomy = (
    select_cols(Gene_Cele_Anatomy)
)

Gene_Cele_Anatomy['Head_ID_IS'] = 'WormBase'
Gene_Cele_Anatomy['Tail_ID_IS'] = 'WormBaseAnatomy'
Gene_Cele_Anatomy['Tail_type'] = 'Anatomy'



Gene_Cele_Anatomy['Relation'] = (
    'Gene_Anatomy'
)

Gene_Cele_Anatomy['Species'] = (
    'Caenorhabditis elegans'
)


save(
    Gene_Cele_Anatomy,
    'Celegans/Gene_Cele_Anatomy.csv'
)

Gene_Cele_Anatomy

Saved 105,006 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Gene_Cele_Anatomy.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,ZK287.5,Gene_Anatomy,WBbt:0000100,Gene,expressed_in,Anatomy,infores:wormbase,RING-box protein 1,C. elegans anatomical entity,Caenorhabditis elegans,NaN,WormBase,WormBaseAnatomy,rbx-1,Caenorhabditis elegans
1,ZK287.5,Gene_Anatomy,WBbt:0005784,Gene,expressed_in,Anatomy,infores:wormbase,RING-box protein 1,germ line,Caenorhabditis elegans,NaN,WormBase,WormBaseAnatomy,rbx-1,Caenorhabditis elegans
2,ZK287.5,Gene_Anatomy,WBbt:0000100,Gene,expressed_in,Anatomy,infores:wormbase,RING-box protein 1,C. elegans anatomical entity,Caenorhabditis elegans,NaN,WormBase,WormBaseAnatomy,rbx-1,Caenorhabditis elegans
3,ZK287.5,Gene_Anatomy,WBbt:0000100,Gene,expressed_in,Anatomy,infores:wormbase,RING-box protein 1,C. elegans anatomical entity,Caenorhabditis elegans,NaN,WormBase,WormBaseAnatomy,rbx-1,Caenorhabditis elegans
4,ZK287.5,Gene_Anatomy,WBbt:0000100,Gene,expressed_in,Anatomy,infores:wormbase,RING-box protein 1,C. elegans anatomical entity,Caenorhabditis elegans,NaN,WormBase,WormBaseAnatomy,rbx-1,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105001,ZK287.5,Gene_Anatomy,WBbt:0000100,Gene,expressed_in,Anatomy,infores:wormbase,RING-box protein 1,C. elegans anatomical entity,Caenorhabditis elegans,NaN,WormBase,WormBaseAnatomy,rbx-1,Caenorhabditis elegans
105002,ZK287.5,Gene_Anatomy,WBbt:0000100,Gene,expressed_in,Anatomy,infores:wormbase,RING-box protein 1,C. elegans anatomical entity,Caenorhabditis elegans,NaN,WormBase,WormBaseAnatomy,rbx-1,Caenorhabditis elegans
105003,ZK287.5,Gene_Anatomy,WBbt:0000100,Gene,expressed_in,Anatomy,infores:wormbase,RING-box protein 1,C. elegans anatomical entity,Caenorhabditis elegans,NaN,WormBase,WormBaseAnatomy,rbx-1,Caenorhabditis elegans
105004,ZK287.5,Gene_Anatomy,WBbt:0000100,Gene,expressed_in,Anatomy,infores:wormbase,RING-box protein 1,C. elegans anatomical entity,Caenorhabditis elegans,NaN,WormBase,WormBaseAnatomy,rbx-1,Caenorhabditis elegans


In [286]:
Gene_Cele_BiologicalProcess = pd.read_csv(
    f'{Semi_processed}Celegans/Gene_Cele_BiologicalProcess_MONARCH.csv'
)
Gene_Cele_BiologicalProcess['Head'] = (
    Gene_Cele_BiologicalProcess['Head']
    .str.replace('WB:', '', regex=False)
)

Gene_Cele_BiologicalProcess['Head'] = (
    Gene_Cele_BiologicalProcess['Head']
    .map(celegans_ncbi_locustag_dict)
)

Gene_Cele_BiologicalProcess['Head_Detail_name'] = (
    Gene_Cele_BiologicalProcess['Head']
    .map(celegans_ncbi_locustag_desc_dict)
)

Gene_Cele_BiologicalProcess = (
    Gene_Cele_BiologicalProcess.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Cele_BiologicalProcess = (
    Gene_Cele_BiologicalProcess.loc[
        :,
        ~Gene_Cele_BiologicalProcess.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Cele_BiologicalProcess = (
    select_cols(Gene_Cele_BiologicalProcess)
)

Gene_Cele_BiologicalProcess['Head_ID_IS'] = 'WormBase'
Gene_Cele_BiologicalProcess['Tail_ID_IS'] = 'Quick_GO'

Gene_Cele_BiologicalProcess['Relation'] = (
    'Gene_BiologicalProcess'
)

Gene_Cele_BiologicalProcess['Tail_type'] = (
    'BiologicalProcess'
)

Gene_Cele_BiologicalProcess['Species'] = (
    'Caenorhabditis elegans'
)

save(
    Gene_Cele_BiologicalProcess,
    'Celegans/Gene_Cele_BiologicalProcess.csv'
)
Gene_Cele_BiologicalProcess

Saved 41,253 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Gene_Cele_BiologicalProcess.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,Y75B8A.11,Gene_BiologicalProcess,GO:1901046,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,LURY-1-2,positive regulation of egg-laying behavior,Caenorhabditis elegans,NaN,WormBase,Quick_GO,lury-1,Caenorhabditis elegans
1,Y75B8A.11,Gene_BiologicalProcess,GO:2000252,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,LURY-1-2,negative regulation of feeding behavior,Caenorhabditis elegans,NaN,WormBase,Quick_GO,lury-1,Caenorhabditis elegans
2,Y75B8A.16,Gene_BiologicalProcess,GO:0098656,Gene,actively_involved_in,BiologicalProcess,infores:goc,Golgi pH regulator,monoatomic anion transmembrane transport,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gphr-1,Caenorhabditis elegans
3,Y75B8A.17,Gene_BiologicalProcess,GO:0008156,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,Geminin homolog,negative regulation of DNA replication,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gmn-1,Caenorhabditis elegans
4,Y75B8A.17,Gene_BiologicalProcess,GO:0008156,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,Geminin homolog,negative regulation of DNA replication,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gmn-1,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41248,Y75B8A.7,Gene_BiologicalProcess,GO:0006364,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,U3 small nucleolar ribonucleoprotein MPP10,rRNA processing,Caenorhabditis elegans,NaN,WormBase,Quick_GO,Y75B8A.7,Caenorhabditis elegans
41249,Y75B8A.7,Gene_BiologicalProcess,GO:0042254,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,U3 small nucleolar ribonucleoprotein MPP10,ribosome biogenesis,Caenorhabditis elegans,NaN,WormBase,Quick_GO,Y75B8A.7,Caenorhabditis elegans
41250,Y75B8A.11,Gene_BiologicalProcess,GO:0007218,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,LURY-1-2,neuropeptide signaling pathway,Caenorhabditis elegans,NaN,WormBase,Quick_GO,lury-1,Caenorhabditis elegans
41251,Y75B8A.11,Gene_BiologicalProcess,GO:0008340,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,LURY-1-2,determination of adult lifespan,Caenorhabditis elegans,NaN,WormBase,Quick_GO,lury-1,Caenorhabditis elegans


In [287]:

Gene_Cele_CellularComponent = pd.read_csv(
    f'{Semi_processed}Celegans/Gene_Cele_CellularComponent_MONARCH.csv'
)

Gene_Cele_CellularComponent['Head'] = (
    Gene_Cele_CellularComponent['Head']
    .str.replace('WB:', '', regex=False)
)

Gene_Cele_CellularComponent['Head'] = (
    Gene_Cele_CellularComponent['Head']
    .map(celegans_ncbi_locustag_dict)
)

Gene_Cele_CellularComponent['Head_Detail_name'] = (
    Gene_Cele_CellularComponent['Head']
    .map(celegans_ncbi_locustag_desc_dict)
)

Gene_Cele_CellularComponent = (
    Gene_Cele_CellularComponent.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Cele_CellularComponent = (
    Gene_Cele_CellularComponent.loc[
        :,
        ~Gene_Cele_CellularComponent.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Cele_CellularComponent = (
    select_cols(Gene_Cele_CellularComponent)
)

Gene_Cele_CellularComponent['Head_ID_IS'] = 'WormBase'
Gene_Cele_CellularComponent['Tail_ID_IS'] = 'Quick_GO'

Gene_Cele_CellularComponent['Relation'] = (
    'Gene_CellularComponent'
)

Gene_Cele_CellularComponent['Tail_type'] = (
    'CellularComponent'
)

Gene_Cele_CellularComponent['Species'] = (
    'Caenorhabditis elegans'
)

save(
    Gene_Cele_CellularComponent,
    'Celegans/Gene_Cele_CellularComponent.csv'
)
Gene_Cele_CellularComponent

Saved 40,938 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Gene_Cele_CellularComponent.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,Y75B8A.16,Gene_CellularComponent,GO:0016020,Gene,located_in,CellularComponent,infores:interpro,Golgi pH regulator,membrane,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gphr-1,Caenorhabditis elegans
1,Y75B8A.16,Gene_CellularComponent,GO:0016020,Gene,located_in,CellularComponent,infores:uniprot,Golgi pH regulator,membrane,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gphr-1,Caenorhabditis elegans
2,Y75B8A.16,Gene_CellularComponent,GO:0016020,Gene,located_in,CellularComponent,infores:uniprot,Golgi pH regulator,membrane,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gphr-1,Caenorhabditis elegans
3,Y75B8A.17,Gene_CellularComponent,GO:0005634,Gene,located_in,CellularComponent,infores:uniprot,Geminin homolog,nucleus,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gmn-1,Caenorhabditis elegans
4,Y75B8A.17,Gene_CellularComponent,GO:0005634,Gene,located_in,CellularComponent,infores:uniprot,Geminin homolog,nucleus,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gmn-1,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40933,Y75B8A.7,Gene_CellularComponent,GO:0034457,Gene,part_of,CellularComponent,infores:interpro,U3 small nucleolar ribonucleoprotein MPP10,Mpp10 complex,Caenorhabditis elegans,NaN,WormBase,Quick_GO,Y75B8A.7,Caenorhabditis elegans
40934,Y75B8A.7,Gene_CellularComponent,GO:0034457,Gene,part_of,CellularComponent,infores:uniprot,U3 small nucleolar ribonucleoprotein MPP10,Mpp10 complex,Caenorhabditis elegans,NaN,WormBase,Quick_GO,Y75B8A.7,Caenorhabditis elegans
40935,Y75B8A.7,Gene_CellularComponent,GO:1990904,Gene,part_of,CellularComponent,infores:uniprot,U3 small nucleolar ribonucleoprotein MPP10,ribonucleoprotein complex,Caenorhabditis elegans,NaN,WormBase,Quick_GO,Y75B8A.7,Caenorhabditis elegans
40936,Y75B8A.11,Gene_CellularComponent,GO:0005576,Gene,located_in,CellularComponent,infores:uniprot,LURY-1-2,extracellular region,Caenorhabditis elegans,NaN,WormBase,Quick_GO,lury-1,Caenorhabditis elegans


In [288]:
Gene_Cele_MolecularActivity = pd.read_csv(
    f'{Semi_processed}Celegans/Gene_Cele_MolecularActivity_MONARCH.csv'
)

Gene_Cele_MolecularActivity['Head'] = (
    Gene_Cele_MolecularActivity['Head']
    .str.replace('WB:', '', regex=False)
)

Gene_Cele_MolecularActivity['Head'] = (
    Gene_Cele_MolecularActivity['Head']
    .map(celegans_ncbi_locustag_dict)
)

Gene_Cele_MolecularActivity['Head_Detail_name'] = (
    Gene_Cele_MolecularActivity['Head']
    .map(celegans_ncbi_locustag_desc_dict)
)

Gene_Cele_MolecularActivity = (
    Gene_Cele_MolecularActivity.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Cele_MolecularActivity = (
    Gene_Cele_MolecularActivity.loc[
        :,
        ~Gene_Cele_MolecularActivity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Cele_MolecularActivity = (
    select_cols(Gene_Cele_MolecularActivity)
)

Gene_Cele_MolecularActivity['Head_ID_IS'] = 'WormBase'
Gene_Cele_MolecularActivity['Tail_ID_IS'] = 'Quick_GO'

Gene_Cele_MolecularActivity['Relation'] = (
    'Gene_MolecularFunction'
)

Gene_Cele_MolecularActivity['Tail_type'] = (
    'MolecularFunction'
)

Gene_Cele_MolecularActivity['Species'] = (
    'Caenorhabditis elegans'
)

save(
    Gene_Cele_MolecularActivity,
    'Celegans/Gene_Cele_MolecularFunction.csv'
)

Gene_Cele_MolecularActivity 

Saved 39,781 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Gene_Cele_MolecularFunction.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,Y75B8A.14,Gene_MolecularFunction,GO:0005515,Gene,enables,MolecularFunction,infores:intact,GPN-loop GTPase 3,protein binding,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gex-4,Caenorhabditis elegans
1,Y75B8A.14,Gene_MolecularFunction,GO:0005515,Gene,enables,MolecularFunction,infores:intact,GPN-loop GTPase 3,protein binding,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gex-4,Caenorhabditis elegans
2,Y75B8A.14,Gene_MolecularFunction,GO:0005525,Gene,enables,MolecularFunction,infores:uniprot,GPN-loop GTPase 3,GTP binding,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gex-4,Caenorhabditis elegans
3,Y75B8A.14,Gene_MolecularFunction,GO:0016787,Gene,enables,MolecularFunction,infores:uniprot,GPN-loop GTPase 3,hydrolase activity,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gex-4,Caenorhabditis elegans
4,Y75B8A.17,Gene_MolecularFunction,GO:0005515,Gene,enables,MolecularFunction,infores:uniprot,Geminin homolog,protein binding,Caenorhabditis elegans,NaN,WormBase,Quick_GO,gmn-1,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39776,Y75B8A.6,Gene_MolecularFunction,GO:0008270,Gene,enables,MolecularFunction,infores:interpro,CXXC-type domain-containing protein,zinc ion binding,Caenorhabditis elegans,NaN,WormBase,Quick_GO,Y75B8A.6,Caenorhabditis elegans
39777,Y75B8A.6,Gene_MolecularFunction,GO:0017056,Gene,enables,MolecularFunction,infores:interpro,CXXC-type domain-containing protein,structural constituent of nuclear pore,Caenorhabditis elegans,NaN,WormBase,Quick_GO,Y75B8A.6,Caenorhabditis elegans
39778,Y75B8A.6,Gene_MolecularFunction,GO:0046872,Gene,enables,MolecularFunction,infores:uniprot,CXXC-type domain-containing protein,metal ion binding,Caenorhabditis elegans,NaN,WormBase,Quick_GO,Y75B8A.6,Caenorhabditis elegans
39779,Y75B8A.10,Gene_MolecularFunction,GO:0046872,Gene,enables,MolecularFunction,infores:uniprot,RING-type domain-containing protein,metal ion binding,Caenorhabditis elegans,NaN,WormBase,Quick_GO,Y75B8A.10,Caenorhabditis elegans


In [289]:
Gene_Cele_Phenotype = pd.read_csv(
    f'{Semi_processed}Celegans/Gene_Cele_Phenotype_MONARCH.csv'
)
Gene_Cele_Phenotype['Head'] = (
    Gene_Cele_Phenotype['Head']
    .str.replace('WB:', '', regex=False)
)

Gene_Cele_Phenotype['Head'] = (
    Gene_Cele_Phenotype['Head']
    .map(celegans_ncbi_locustag_dict)
)

Gene_Cele_Phenotype['Head_Detail_name'] = (
    Gene_Cele_Phenotype['Head']
    .map(celegans_ncbi_locustag_desc_dict)
)

Gene_Cele_Phenotype = (
    Gene_Cele_Phenotype.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Cele_Phenotype = (
    Gene_Cele_Phenotype.loc[
        :,
        ~Gene_Cele_Phenotype.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Cele_Phenotype = (
    select_cols(Gene_Cele_Phenotype)
)

Gene_Cele_Phenotype['Head_ID_IS'] = 'WormBase'
Gene_Cele_Phenotype['Tail_ID_IS'] = 'WormBasePhenotype'

Gene_Cele_Phenotype['Relation'] = (
    'Gene_Phenotype'
)

Gene_Cele_Phenotype['Tail_type'] = (
    'Phenotype'
)

Gene_Cele_Phenotype['Species'] = (
    'Caenorhabditis elegans'
)
save(
    Gene_Cele_Phenotype,
    'Celegans/Gene_Cele_Phenotype.csv'
)
Gene_Cele_Phenotype

Saved 26,686 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/Gene_Cele_Phenotype.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,F43C1.2,Gene_Phenotype,WBPhenotype:0001191,Gene,has_phenotype,Phenotype,infores:wormbase,Mitogen-activated protein kinase mpk-1,pharyngeal muscle physiology variant,Caenorhabditis elegans,NaN,WormBase,WormBasePhenotype,mpk-1,Caenorhabditis elegans
1,C35B8.2,Gene_Phenotype,WBPhenotype:0000154,Gene,has_phenotype,Phenotype,infores:wormbase,Protein vav;Protein vav-1,reduced brood size,Caenorhabditis elegans,NaN,WormBase,WormBasePhenotype,vav-1,Caenorhabditis elegans
2,C35B8.2,Gene_Phenotype,WBPhenotype:0000246,Gene,has_phenotype,Phenotype,infores:wormbase,Protein vav;Protein vav-1,defecation cycle variable length,Caenorhabditis elegans,NaN,WormBase,WormBasePhenotype,vav-1,Caenorhabditis elegans
3,C35B8.2,Gene_Phenotype,WBPhenotype:0000666,Gene,has_phenotype,Phenotype,infores:wormbase,Protein vav;Protein vav-1,ovulation variant,Caenorhabditis elegans,NaN,WormBase,WormBasePhenotype,vav-1,Caenorhabditis elegans
4,C35B8.2,Gene_Phenotype,WBPhenotype:0000668,Gene,has_phenotype,Phenotype,infores:wormbase,Protein vav;Protein vav-1,endomitotic oocytes,Caenorhabditis elegans,NaN,WormBase,WormBasePhenotype,vav-1,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26681,F49E12.6,Gene_Phenotype,WBPhenotype:0000054,Gene,has_phenotype,Phenotype,infores:wormbase,Transcription factor efl-3,larval lethal,Caenorhabditis elegans,NaN,WormBase,WormBasePhenotype,efl-3,Caenorhabditis elegans
26682,F49E12.6,Gene_Phenotype,WBPhenotype:0000054,Gene,has_phenotype,Phenotype,infores:wormbase,Transcription factor efl-3,larval lethal,Caenorhabditis elegans,NaN,WormBase,WormBasePhenotype,efl-3,Caenorhabditis elegans
26683,NaN,Gene_Phenotype,WBPhenotype:0000062,Gene,has_phenotype,Phenotype,infores:wormbase,NaN,lethal,Caenorhabditis elegans,NaN,WormBase,WormBasePhenotype,let-72,Caenorhabditis elegans
26684,NaN,Gene_Phenotype,WBPhenotype:0000054,Gene,has_phenotype,Phenotype,infores:wormbase,NaN,larval lethal,Caenorhabditis elegans,NaN,WormBase,WormBasePhenotype,let-72,Caenorhabditis elegans


In [290]:

GENE_GENE_Cele_Cele = pd.read_csv(
    f'{Semi_processed}Celegans/GENE_GENE_Cele_Cele_MONARCH.csv'
)
GENE_GENE_Cele_Cele[['Head_name', 'Head']] = GENE_GENE_Cele_Cele[['Head', 'Head_name']]
GENE_GENE_Cele_Cele[['Tail_name', 'Tail']] = GENE_GENE_Cele_Cele[['Tail', 'Tail_name']]
GENE_GENE_Cele_Cele['Head'] = (
    GENE_GENE_Cele_Cele['Head']
    .str.replace('WB:', '', regex=False)
)

GENE_GENE_Cele_Cele['Tail'] = (
    GENE_GENE_Cele_Cele['Tail']
    .str.replace('WB:', '', regex=False)
)

GENE_GENE_Cele_Cele['Head'] = (
    GENE_GENE_Cele_Cele['Head']
    .map(celegans_ncbi_locustag_dict)
)

GENE_GENE_Cele_Cele['Tail'] = (
    GENE_GENE_Cele_Cele['Tail']
    .map(celegans_ncbi_locustag_dict)
)

GENE_GENE_Cele_Cele['Head_Detail_name'] = (
    GENE_GENE_Cele_Cele['Head']
    .map(celegans_ncbi_locustag_desc_dict)
)

GENE_GENE_Cele_Cele['Tail_Detail_name'] = (
    GENE_GENE_Cele_Cele['Tail']
    .map(celegans_ncbi_locustag_desc_dict)
)

GENE_GENE_Cele_Cele = (
    GENE_GENE_Cele_Cele.rename(columns={
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
GENE_GENE_Cele_Cele = (
    GENE_GENE_Cele_Cele.loc[
        :,
        ~GENE_GENE_Cele_Cele.columns.duplicated()
    ]
)

# Keep only desired columns that exist
GENE_GENE_Cele_Cele = (
    select_cols(GENE_GENE_Cele_Cele)
)

GENE_GENE_Cele_Cele['Head_ID_IS'] = 'WormBase'
GENE_GENE_Cele_Cele['Tail_ID_IS'] = 'WormBase'

GENE_GENE_Cele_Cele['Relation'] = (
    'Gene_Gene'
)

GENE_GENE_Cele_Cele['Species'] = (
    'Caenorhabditis elegans'
)

save(
    GENE_GENE_Cele_Cele,
    'Celegans/GENE_GENE_Cele_Cele.csv'
)
GENE_GENE_Cele_Cele

Saved 146,558 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Celegans/GENE_GENE_Cele_Cele.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,KG_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Tail_name,Species
0,2RSSE.1,Gene_Gene,ZK669.1,Gene,interacts_with,Gene,infores:string,Monarch_KG,Rho-GAP domain-containing protein,F-BAR domain-containing protein,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,rga-9,spv-1,Caenorhabditis elegans
1,2RSSE.1,Gene_Gene,Y34B4A.8,Gene,interacts_with,Gene,infores:string,Monarch_KG,Rho-GAP domain-containing protein,Rho-GAP domain-containing protein,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,rga-9,rga-8,Caenorhabditis elegans
2,2RSSE.1,Gene_Gene,BE0003N10.2,Gene,interacts_with,Gene,infores:string,Monarch_KG,Rho-GAP domain-containing protein,Chimerin 2,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,rga-9,chin-1,Caenorhabditis elegans
3,3R5.1,Gene_Gene,F12F6.7,Gene,interacts_with,Gene,infores:string,Monarch_KG,Protection of telomeres protein 1 ssDNA-bindin...,putative DNA polymerase delta small subunit,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,pot-3,F12F6.7,Caenorhabditis elegans
4,3R5.1,Gene_Gene,W08A12.2,Gene,interacts_with,Gene,infores:string,Monarch_KG,Protection of telomeres protein 1 ssDNA-bindin...,Neuropeptide-Like Protein,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,pot-3,W08A12.2,Caenorhabditis elegans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146553,T09B4.10,Gene_Gene,F26D10.3,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,E3 ubiquitin-protein ligase CHIP,Heat shock protein hsp-1,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,chn-1,hsp-1,Caenorhabditis elegans
146554,T09B4.10,Gene_Gene,Y55D5A.5,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,E3 ubiquitin-protein ligase CHIP,Insulin-like receptor subunit beta;Protein kin...,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,chn-1,daf-2,Caenorhabditis elegans
146555,T09B4.10,Gene_Gene,Y55D5A.5,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,E3 ubiquitin-protein ligase CHIP,Insulin-like receptor subunit beta;Protein kin...,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,chn-1,daf-2,Caenorhabditis elegans
146556,T09B4.10,Gene_Gene,F26D10.3,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,E3 ubiquitin-protein ligase CHIP,Heat shock protein hsp-1,Caenorhabditis elegans,Caenorhabditis elegans,WormBase,WormBase,chn-1,hsp-1,Caenorhabditis elegans


# Drosophila

In [310]:
droso_ncbi = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Drosophila_melanogaster.gene_info',sep = '\t')
droso_ncbi['LocusTag'] = droso_ncbi['LocusTag'].str.replace('Dmel_', '', regex=False)
# Extract WormBase ID
droso_ncbi['FlybaseBase_ID'] = (
    droso_ncbi['dbXrefs']
    .str.extract(r'FLYBASE:(FBgn\d+)')
)

# Keep only useful columns
droso_ncbi_clean = droso_ncbi[
    [
        'Symbol',
        'LocusTag',
        'FlybaseBase_ID',
        'description'
    ]
]

# Remove duplicates
droso_ncbi_clean = droso_ncbi_clean.drop_duplicates()

# Remove rows with missing IDs
droso_ncbi_clean = droso_ncbi_clean[
    ~droso_ncbi_clean['FlybaseBase_ID'].isna()
]

droso_ncbi_locustag_dict = dict(zip(droso_ncbi_clean['FlybaseBase_ID'],droso_ncbi_clean['LocusTag']))
droso_ncbi_locustag_desc_dict = dict(zip(droso_ncbi_clean['LocusTag'],droso_ncbi_clean['description']))
droso_ncbi_locustag_desc_dict
droso_ncbi_locustag_dict

{'FBgn0040373': 'CG3038',
 'FBgn0040372': 'CG2995',
 'FBgn0261446': 'CG13377',
 'FBgn0000316': 'CG2945',
 'FBgn0005427': 'CG3114',
 'FBgn0040370': 'CG13375',
 'FBgn0040371': 'CG12470',
 'FBgn0029521': 'CG17885',
 'FBgn0024989': 'CG3777',
 'FBgn0004034': 'CG3757',
 'FBgn0000022': 'CG3796',
 'FBgn0004170': 'CG3827',
 'FBgn0002561': 'CG3839',
 'FBgn0011822': 'CG13374',
 'FBgn0000137': 'CG3258',
 'FBgn0010019': 'CG3972',
 'FBgn0029522': 'CG13373',
 'FBgn0029524': 'CG3176',
 'FBgn0029525': 'CG18273',
 'FBgn0023536': 'CG3156',
 'FBgn0023537': 'CG17896',
 'FBgn0023534': 'CG17778',
 'FBgn0004648': 'CG4122',
 'FBgn0260400': 'CG4262',
 'FBgn0024983': 'CG4293',
 'FBgn0000108': 'CG7727',
 'FBgn0261930': 'CG6172',
 'FBgn0025633': 'CG13366',
 'FBgn0029529': 'CG13365',
 'FBgn0016038': 'CG17828',
 'FBgn0025640': 'CG13369',
 'FBgn0002579': 'CG7622',
 'FBgn0001341': 'CG6189',
 'FBgn0020381': 'CG7486',
 'FBgn0003575': 'CG6222',
 'FBgn0025634': 'CG13367',
 'FBgn0025638': 'CG16982',
 'FBgn0025639': 'CG1336

In [298]:
Droso_AnatomicalEntity_AnatomicalEntity = pd.read_csv(
    f'{Semi_processed}Drosophila/Droso_AnatomicalEntity_AnatomicalEntity_MONARCH.csv'
)

Droso_AnatomicalEntity_AnatomicalEntity = (
    Droso_AnatomicalEntity_AnatomicalEntity.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Droso_AnatomicalEntity_AnatomicalEntity = (
    Droso_AnatomicalEntity_AnatomicalEntity.loc[
        :,
        ~Droso_AnatomicalEntity_AnatomicalEntity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Droso_AnatomicalEntity_AnatomicalEntity = (
    select_cols(Droso_AnatomicalEntity_AnatomicalEntity)
)

Droso_AnatomicalEntity_AnatomicalEntity['Head_ID_IS'] = 'flybase'
Droso_AnatomicalEntity_AnatomicalEntity['Tail_ID_IS'] = 'flybase'

Droso_AnatomicalEntity_AnatomicalEntity['Relation'] = (
    'Anatomy_Anatomy'
)
Droso_AnatomicalEntity_AnatomicalEntity['Head_type'] = ('Anatomy')
Droso_AnatomicalEntity_AnatomicalEntity['Tail_type'] = ('Anatomy')
Droso_AnatomicalEntity_AnatomicalEntity['Species'] = (
    'Drosophila melanogaster'
)
save(Droso_AnatomicalEntity_AnatomicalEntity, 'Drosophila/Droso_AnatomicalEntity_AnatomicalEntity.csv')
Droso_AnatomicalEntity_AnatomicalEntity

Saved 131,953 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Drosophila/Droso_AnatomicalEntity_AnatomicalEntity.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,FBbt:00000000,Anatomy_Anatomy,FBbt:10000000,Anatomy,subclass_of,Anatomy,infores:fbbt,germ layer derivative,anatomical entity,Drosophila melanogaster,Drosophila melanogaster,flybase,flybase,Drosophila melanogaster
1,FBbt:00000000,Anatomy_Anatomy,FBbt:00000110,Anatomy,related_to,Anatomy,infores:fbbt,germ layer derivative,germ layer,Drosophila melanogaster,Drosophila melanogaster,flybase,flybase,Drosophila melanogaster
2,FBbt:00000001,Anatomy_Anatomy,FBbt:00100313,Anatomy,subclass_of,Anatomy,infores:fbbt,organism,multicellular structure,Drosophila melanogaster,Drosophila melanogaster,flybase,flybase,Drosophila melanogaster
3,FBbt:00000002,Anatomy_Anatomy,FBbt:00057001,Anatomy,subclass_of,Anatomy,infores:fbbt,tagma,anterior-posterior subdivision of organism,Drosophila melanogaster,Drosophila melanogaster,flybase,flybase,Drosophila melanogaster
4,FBbt:00000002,Anatomy_Anatomy,FBbt:00000001,Anatomy,related_to,Anatomy,infores:fbbt,tagma,organism,Drosophila melanogaster,Drosophila melanogaster,flybase,flybase,Drosophila melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131948,FBbt:20011361,Anatomy_Anatomy,FBbt:00003982,Anatomy,related_to,Anatomy,infores:fbbt,descending neuron of the posterior brain DNp70,antennal mechanosensory and motor center,Drosophila melanogaster,Drosophila melanogaster,flybase,flybase,Drosophila melanogaster
131949,FBbt:20011361,Anatomy_Anatomy,FBbt:00014013,Anatomy,related_to,Anatomy,infores:fbbt,descending neuron of the posterior brain DNp70,adult gnathal ganglion,Drosophila melanogaster,Drosophila melanogaster,flybase,flybase,Drosophila melanogaster
131950,FBbt:20011361,Anatomy_Anatomy,FBbt:00040042,Anatomy,related_to,Anatomy,infores:fbbt,descending neuron of the posterior brain DNp70,posterior ventrolateral protocerebrum,Drosophila melanogaster,Drosophila melanogaster,flybase,flybase,Drosophila melanogaster
131951,FBbt:20011361,Anatomy_Anatomy,FBbt:00047139,Anatomy,related_to,Anatomy,infores:fbbt,descending neuron of the posterior brain DNp70,adult leg neuropil,Drosophila melanogaster,Drosophila melanogaster,flybase,flybase,Drosophila melanogaster


In [301]:
# Droso_AnatomicalEntity_BiologicalProcess = pd.read_csv(
#     f'{Semi_processed}Drosophila/Droso_AnatomicalEntity_BiologicalProcess_Monarch.csv')
# Droso_AnatomicalEntity_BiologicalProcess

In [305]:
Droso_AnatomicalEntity_CellularComponent = pd.read_csv(
    f'{Semi_processed}Drosophila/Droso_AnatomicalEntity_CellularComponent_Monarch.csv'
)

Droso_AnatomicalEntity_CellularComponent = (
    Droso_AnatomicalEntity_CellularComponent.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Droso_AnatomicalEntity_CellularComponent = (
    Droso_AnatomicalEntity_CellularComponent.loc[
        :,
        ~Droso_AnatomicalEntity_CellularComponent.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Droso_AnatomicalEntity_CellularComponent = (
    select_cols(Droso_AnatomicalEntity_CellularComponent)
)

Droso_AnatomicalEntity_CellularComponent['Head_ID_IS'] = 'flybase'
Droso_AnatomicalEntity_CellularComponent['Tail_ID_IS'] = 'Quick_GO'

Droso_AnatomicalEntity_CellularComponent['Relation'] = (
    'Anatomy_Anatomy'
)
Droso_AnatomicalEntity_CellularComponent['Head_type'] = ('Anatomy')
Droso_AnatomicalEntity_CellularComponent['Tail_type'] = ('CellularComponent')
Droso_AnatomicalEntity_CellularComponent['Species'] = (
    'Drosophila melanogaster'
)
save(Droso_AnatomicalEntity_CellularComponent, 'Drosophila/Droso_AnatomicalEntity_CellularComponent.csv')

Droso_AnatomicalEntity_CellularComponent

Saved 2,766 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Drosophila/Droso_AnatomicalEntity_CellularComponent.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,WBbt:0001001,Anatomy_Anatomy,GO:0005634,Anatomy,subclass_of,CellularComponent,infores:wbbt,AB nucleus,nucleus,Drosophila melanogaster,NaN,flybase,Quick_GO,Drosophila melanogaster
1,WBbt:0001002,Anatomy_Anatomy,GO:0005634,Anatomy,subclass_of,CellularComponent,infores:wbbt,ABa nucleus,nucleus,Drosophila melanogaster,NaN,flybase,Quick_GO,Drosophila melanogaster
2,WBbt:0001003,Anatomy_Anatomy,GO:0005634,Anatomy,subclass_of,CellularComponent,infores:wbbt,ABal nucleus,nucleus,Drosophila melanogaster,NaN,flybase,Quick_GO,Drosophila melanogaster
3,WBbt:0001004,Anatomy_Anatomy,GO:0005634,Anatomy,subclass_of,CellularComponent,infores:wbbt,ABala nucleus,nucleus,Drosophila melanogaster,NaN,flybase,Quick_GO,Drosophila melanogaster
4,WBbt:0001005,Anatomy_Anatomy,GO:0005634,Anatomy,subclass_of,CellularComponent,infores:wbbt,ABalaa nucleus,nucleus,Drosophila melanogaster,NaN,flybase,Quick_GO,Drosophila melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2761,WBbt:0008067,Anatomy_Anatomy,GO:0005634,Anatomy,subclass_of,CellularComponent,infores:wbbt,Z1.a nucleus,nucleus,Drosophila melanogaster,NaN,flybase,Quick_GO,Drosophila melanogaster
2762,WBbt:0008615,Anatomy_Anatomy,GO:0005634,Anatomy,subclass_of,CellularComponent,infores:wbbt,ABplpaapapaa nucleus,nucleus,Drosophila melanogaster,NaN,flybase,Quick_GO,Drosophila melanogaster
2763,WBbt:0008616,Anatomy_Anatomy,GO:0005634,Anatomy,subclass_of,CellularComponent,infores:wbbt,ABplpaapapap nucleus,nucleus,Drosophila melanogaster,NaN,flybase,Quick_GO,Drosophila melanogaster
2764,WBbt:0008617,Anatomy_Anatomy,GO:0005634,Anatomy,subclass_of,CellularComponent,infores:wbbt,ABprpaapapap nucleus,nucleus,Drosophila melanogaster,NaN,flybase,Quick_GO,Drosophila melanogaster


In [315]:
Droso_Gene_Pathway = pd.read_csv(
    f'{Semi_processed}Drosophila/Droso_Gene_Pathway_MONARCH.csv'
)

Droso_Gene_Pathway['Head'] = Droso_Gene_Pathway['Head'].str.replace('FB:', '', regex=False)
Droso_Gene_Pathway['Head'] = Droso_Gene_Pathway['Head'].map(droso_ncbi_locustag_dict)
Droso_Gene_Pathway['Head_Detail_name'] = Droso_Gene_Pathway['Head'].map(droso_ncbi_locustag_desc_dict)


Droso_Gene_Pathway = (
    Droso_Gene_Pathway.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'}))

# Remove duplicate columns
Droso_Gene_Pathway = (
    Droso_Gene_Pathway.loc[
        :,
        ~Droso_Gene_Pathway.columns.duplicated()])

# Keep only desired columns that exist
Droso_Gene_Pathway = (
    select_cols(Droso_Gene_Pathway))

Droso_Gene_Pathway['Head_ID_IS'] = 'FlyBase'

Droso_Gene_Pathway['Relation'] = ('Gene_Pathway')

Droso_Gene_Pathway['Species'] = (
    'Drosophila melanogaster')
save(Droso_Gene_Pathway, 'Drosophila/Droso_Gene_Pathway.csv')
Droso_Gene_Pathway

Saved 18,469 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Drosophila/Droso_Gene_Pathway.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,CG2316,Gene_Pathway,R-HSA-390247,Gene,participates_in,Pathway,infores:reactome,ATP binding cassette subfamily D member 1,Beta-oxidation of very long chain fatty acids,Drosophila melanogaster,Homo sapiens,FlyBase,Reactome,Abcd1,Drosophila melanogaster
1,CG2316,Gene_Pathway,R-HSA-9603798,Gene,participates_in,Pathway,infores:reactome,ATP binding cassette subfamily D member 1,Class I peroxisomal membrane protein import,Drosophila melanogaster,Homo sapiens,FlyBase,Reactome,Abcd1,Drosophila melanogaster
2,CG1587,Gene_Pathway,R-HSA-186763,Gene,participates_in,Pathway,infores:reactome,Crk oncogene,Downstream signal transduction,Drosophila melanogaster,Homo sapiens,FlyBase,Reactome,Crk,Drosophila melanogaster
3,CG1587,Gene_Pathway,R-HSA-4420097,Gene,participates_in,Pathway,infores:reactome,Crk oncogene,VEGFA-VEGFR2 Pathway,Drosophila melanogaster,Homo sapiens,FlyBase,Reactome,Crk,Drosophila melanogaster
4,CG1587,Gene_Pathway,R-HSA-8849471,Gene,participates_in,Pathway,infores:reactome,Crk oncogene,"PTK6 Regulates RHO GTPases, RAS GTPase and MAP...",Drosophila melanogaster,Homo sapiens,FlyBase,Reactome,Crk,Drosophila melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18464,CG34403,Gene_Pathway,R-HSA-9825892,Gene,participates_in,Pathway,infores:reactome,pangolin,Regulation of MITF-M-dependent genes involved ...,Drosophila melanogaster,Homo sapiens,FlyBase,Reactome,pan,Drosophila melanogaster
18465,CG1651,Gene_Pathway,R-HSA-6807878,Gene,participates_in,Pathway,infores:reactome,Ankyrin,COPI-mediated anterograde transport,Drosophila melanogaster,Homo sapiens,FlyBase,Reactome,Ank,Drosophila melanogaster
18466,CG2316,Gene_Pathway,R-HSA-1369062,Gene,participates_in,Pathway,infores:reactome,ATP binding cassette subfamily D member 1,ABC transporters in lipid homeostasis,Drosophila melanogaster,Homo sapiens,FlyBase,Reactome,Abcd1,Drosophila melanogaster
18467,CG2316,Gene_Pathway,R-HSA-2046105,Gene,participates_in,Pathway,infores:reactome,ATP binding cassette subfamily D member 1,Linoleic acid (LA) metabolism,Drosophila melanogaster,Homo sapiens,FlyBase,Reactome,Abcd1,Drosophila melanogaster


In [318]:
Gene_Droso_Anatomy = pd.read_csv(
    f'{Semi_processed}Drosophila/Gene_Droso_Anatomy_MONARCH.csv'
)

Gene_Droso_Anatomy['Head'] = Gene_Droso_Anatomy['Head'].str.replace('FB:', '', regex=False)
Gene_Droso_Anatomy['Head'] = Gene_Droso_Anatomy['Head'].map(droso_ncbi_locustag_dict)
Gene_Droso_Anatomy['Head_Detail_name'] = Gene_Droso_Anatomy['Head'].map(droso_ncbi_locustag_desc_dict)


Gene_Droso_Anatomy = (
    Gene_Droso_Anatomy.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'}))

# Remove duplicate columns
Gene_Droso_Anatomy = (
    Gene_Droso_Anatomy.loc[
        :,
        ~Gene_Droso_Anatomy.columns.duplicated()])

# Keep only desired columns that exist
Gene_Droso_Anatomy = (
    select_cols(Gene_Droso_Anatomy))

Gene_Droso_Anatomy['Head_ID_IS'] = 'FlyBase'
Gene_Droso_Anatomy['Tail_ID_IS'] = 'FlyBase'


Gene_Droso_Anatomy['Relation'] = ('Gene_Anatomy')

Gene_Droso_Anatomy['Species'] = (
    'Drosophila melanogaster')

save(Gene_Droso_Anatomy, 'Drosophila/Gene_Droso_Anatomy.csv')
Gene_Droso_Anatomy

Saved 397,421 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Drosophila/Gene_Droso_Anatomy.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,CG8340,Gene_Anatomy,FBbt:00003007,Gene,expressed_in,AnatomicalEntity,infores:flybase,upstream of RpIII128,adult head,Drosophila melanogaster,NaN,FlyBase,FlyBase,128up,Drosophila melanogaster
1,CG31196,Gene_Anatomy,FBbt:00001056,Gene,expressed_in,AnatomicalEntity,infores:flybase,14-3-3epsilon,presumptive embryonic/larval central nervous s...,Drosophila melanogaster,NaN,FlyBase,FlyBase,14-3-3ε,Drosophila melanogaster
2,CG31196,Gene_Anatomy,FBbt:00001134,Gene,expressed_in,AnatomicalEntity,infores:flybase,14-3-3epsilon,presumptive embryonic/larval peripheral nervou...,Drosophila melanogaster,NaN,FlyBase,FlyBase,14-3-3ε,Drosophila melanogaster
3,CG31196,Gene_Anatomy,FBbt:00000092,Gene,expressed_in,AnatomicalEntity,infores:flybase,14-3-3epsilon,primordial germ cell,Drosophila melanogaster,NaN,FlyBase,FlyBase,14-3-3ε,Drosophila melanogaster
4,CG31196,Gene_Anatomy,FBbt:00005409,Gene,expressed_in,AnatomicalEntity,infores:flybase,14-3-3epsilon,ring canal,Drosophila melanogaster,NaN,FlyBase,FlyBase,14-3-3ε,Drosophila melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
397416,CG15304,Gene_Anatomy,FBbt:00003154,Gene,expressed_in,AnatomicalEntity,infores:flybase,Neb-cGP,adult heart,Drosophila melanogaster,NaN,FlyBase,FlyBase,Neb-cGP,Drosophila melanogaster
397417,CG15304,Gene_Anatomy,FBbt:00003154,Gene,expressed_in,AnatomicalEntity,infores:flybase,Neb-cGP,adult heart,Drosophila melanogaster,NaN,FlyBase,FlyBase,Neb-cGP,Drosophila melanogaster
397418,CG15304,Gene_Anatomy,FBbt:00003154,Gene,expressed_in,AnatomicalEntity,infores:flybase,Neb-cGP,adult heart,Drosophila melanogaster,NaN,FlyBase,FlyBase,Neb-cGP,Drosophila melanogaster
397419,CG15304,Gene_Anatomy,FBbt:00003154,Gene,expressed_in,AnatomicalEntity,infores:flybase,Neb-cGP,adult heart,Drosophila melanogaster,NaN,FlyBase,FlyBase,Neb-cGP,Drosophila melanogaster


In [322]:
Gene_Droso_BiologicalProcess = pd.read_csv(f'{Semi_processed}Drosophila/Gene_Droso_BiologicalProcess_MONARCH.csv')
Gene_Droso_BiologicalProcess['Head'] = (
    Gene_Droso_BiologicalProcess['Head']
    .str.replace('FB:', '', regex=False))

Gene_Droso_BiologicalProcess['Head'] = (
    Gene_Droso_BiologicalProcess['Head']
    .map(droso_ncbi_locustag_dict))

Gene_Droso_BiologicalProcess['Head_Detail_name'] = (
    Gene_Droso_BiologicalProcess['Head']
    .map(droso_ncbi_locustag_desc_dict))

Gene_Droso_BiologicalProcess = (
    Gene_Droso_BiologicalProcess.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'}))

# Remove duplicate columns
Gene_Droso_BiologicalProcess = (
    Gene_Droso_BiologicalProcess.loc[
        :,
        ~Gene_Droso_BiologicalProcess.columns.duplicated()])

# Keep only desired columns that exist
Gene_Droso_BiologicalProcess = (
    select_cols(Gene_Droso_BiologicalProcess)
)

Gene_Droso_BiologicalProcess['Head_ID_IS'] = 'FlyBase'
Gene_Droso_BiologicalProcess['Tail_ID_IS'] = 'Quick_GO'

Gene_Droso_BiologicalProcess['Relation'] = (
    'Gene_BiologicalProcess'
)

Gene_Droso_BiologicalProcess['Tail_type'] = (
    'BiologicalProcess'
)

Gene_Droso_BiologicalProcess['Species'] = (
    'Drosophila melanogaster'
)
save(Gene_Droso_BiologicalProcess,'Drosophila/Gene_Droso_BiologicalProcess.csv')
Gene_Droso_BiologicalProcess

Saved 59,900 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Drosophila/Gene_Droso_BiologicalProcess.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,CG5392,Gene_BiologicalProcess,GO:0030198,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Reversion-inducing-cysteine-rich protein with ...,extracellular matrix organization,Drosophila melanogaster,NaN,FlyBase,Quick_GO,Reck,Drosophila melanogaster
1,CG12754,Gene_BiologicalProcess,GO:0050911,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Odorant receptor 42b,detection of chemical stimulus involved in sen...,Drosophila melanogaster,NaN,FlyBase,Quick_GO,Or42b,Drosophila melanogaster
2,CG17063,Gene_BiologicalProcess,GO:0007602,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Innexin 6,phototransduction,Drosophila melanogaster,NaN,FlyBase,Quick_GO,Inx6,Drosophila melanogaster
3,CG7980,Gene_BiologicalProcess,GO:0016192,Gene,actively_involved_in,BiologicalProcess,infores:go-central,RabX5,vesicle-mediated transport,Drosophila melanogaster,NaN,FlyBase,Quick_GO,RabX5,Drosophila melanogaster
4,CG7524,Gene_BiologicalProcess,GO:0030154,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Src oncogene at 64B,cell differentiation,Drosophila melanogaster,NaN,FlyBase,Quick_GO,Src64B,Drosophila melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59895,CG4525,Gene_BiologicalProcess,GO:0035735,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Tetratricopeptide repeat domain 26,intraciliary transport involved in cilium asse...,Drosophila melanogaster,NaN,FlyBase,Quick_GO,Ttc26,Drosophila melanogaster
59896,CG12384,Gene_BiologicalProcess,GO:0034198,Gene,actively_involved_in,BiologicalProcess,infores:go-central,uncharacterized protein,cellular response to amino acid starvation,Drosophila melanogaster,NaN,FlyBase,Quick_GO,CG12384,Drosophila melanogaster
59897,CG2229,Gene_BiologicalProcess,GO:0045087,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Jonah 99Fii,innate immune response,Drosophila melanogaster,NaN,FlyBase,Quick_GO,Jon99Fii,Drosophila melanogaster
59898,CG5434,Gene_BiologicalProcess,GO:0006614,Gene,actively_involved_in,BiologicalProcess,infores:go-central,Signal recognition particle 72,SRP-dependent cotranslational protein targetin...,Drosophila melanogaster,NaN,FlyBase,Quick_GO,Srp72,Drosophila melanogaster


In [326]:
Gene_Droso_CellularComponent = pd.read_csv(
    f'{Semi_processed}Drosophila/Gene_Droso_CellularComponent_MONARCH.csv'
)

Gene_Droso_CellularComponent['Head'] = (
    Gene_Droso_CellularComponent['Head']
    .str.replace('FB:', '', regex=False))

Gene_Droso_CellularComponent['Head'] = (
    Gene_Droso_CellularComponent['Head']
    .map(droso_ncbi_locustag_dict))

Gene_Droso_CellularComponent['Head_Detail_name'] = (
    Gene_Droso_CellularComponent['Head']
    .map(droso_ncbi_locustag_desc_dict))

Gene_Droso_CellularComponent = (
    Gene_Droso_CellularComponent.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    }))

# Remove duplicate columns
Gene_Droso_CellularComponent = (
    Gene_Droso_CellularComponent.loc[
        :,
        ~Gene_Droso_CellularComponent.columns.duplicated()])

# Keep only desired columns that exist
Gene_Droso_CellularComponent = (
    select_cols(Gene_Droso_CellularComponent))

Gene_Droso_CellularComponent['Head_ID_IS'] = 'FlyBase'
Gene_Droso_CellularComponent['Tail_ID_IS'] = 'Quick_GO'

Gene_Droso_CellularComponent['Relation'] = ('Gene_CellularComponent')

Gene_Droso_CellularComponent['Tail_type'] = ('CellularComponent')

Gene_Droso_CellularComponent['Species'] = ('Drosophila melanogaster')

save(Gene_Droso_CellularComponent,'Drosophila/Gene_Droso_CellularComponent.csv')

Gene_Droso_CellularComponent

Saved 50,087 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Drosophila/Gene_Droso_CellularComponent.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,CG14971,Gene_CellularComponent,GO:0000138,Gene,expressed_in,CellularComponent,infores:flybase,uncharacterized protein,Golgi trans cisterna,Drosophila melanogaster,NaN,FlyBase,Quick_GO,CG14971,Drosophila melanogaster
1,CG6875,Gene_CellularComponent,GO:0005813,Gene,expressed_in,CellularComponent,infores:flybase,abnormal spindle,centrosome,Drosophila melanogaster,NaN,FlyBase,Quick_GO,asp,Drosophila melanogaster
2,CG2746,Gene_CellularComponent,GO:0022626,Gene,expressed_in,CellularComponent,infores:flybase,Ribosomal protein L19,cytosolic ribosome,Drosophila melanogaster,NaN,FlyBase,Quick_GO,RpL19,Drosophila melanogaster
3,CG10837,Gene_CellularComponent,GO:0005829,Gene,expressed_in,CellularComponent,infores:flybase,eukaryotic translation initiation factor 4B,cytosol,Drosophila melanogaster,NaN,FlyBase,Quick_GO,eIF4B,Drosophila melanogaster
4,CG7622,Gene_CellularComponent,GO:0022626,Gene,expressed_in,CellularComponent,infores:flybase,Ribosomal protein L36,cytosolic ribosome,Drosophila melanogaster,NaN,FlyBase,Quick_GO,RpL36,Drosophila melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
50082,CG7345,Gene_CellularComponent,GO:0005634,Gene,is_active_in,CellularComponent,infores:go-central,Sox21a,nucleus,Drosophila melanogaster,NaN,FlyBase,Quick_GO,Sox21a,Drosophila melanogaster
50083,CG12443,Gene_CellularComponent,GO:0005615,Gene,is_active_in,CellularComponent,infores:go-central,thisbe,extracellular space,Drosophila melanogaster,NaN,FlyBase,Quick_GO,ths,Drosophila melanogaster
50084,CG30021,Gene_CellularComponent,GO:0005886,Gene,is_active_in,CellularComponent,infores:go-central,menage a trois,plasma membrane,Drosophila melanogaster,NaN,FlyBase,Quick_GO,metro,Drosophila melanogaster
50085,CG31913,Gene_CellularComponent,GO:0005743,Gene,is_active_in,CellularComponent,infores:go-central,uncharacterized protein,mitochondrial inner membrane,Drosophila melanogaster,NaN,FlyBase,Quick_GO,CG31913,Drosophila melanogaster


In [328]:
Gene_Droso_MolecularActivity = pd.read_csv(
    f'{Semi_processed}Drosophila/Gene_Droso_MolecularActivity_MONARCH.csv'
)

Gene_Droso_MolecularActivity['Head'] = (
    Gene_Droso_MolecularActivity['Head']
    .str.replace('FB:', '', regex=False)
)

Gene_Droso_MolecularActivity['Head'] = (
    Gene_Droso_MolecularActivity['Head']
    .map(droso_ncbi_locustag_dict)
)

Gene_Droso_MolecularActivity['Head_Detail_name'] = (
    Gene_Droso_MolecularActivity['Head']
    .map(droso_ncbi_locustag_desc_dict)
)

Gene_Droso_MolecularActivity = (
    Gene_Droso_MolecularActivity.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Droso_MolecularActivity = (
    Gene_Droso_MolecularActivity.loc[
        :,
        ~Gene_Droso_MolecularActivity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Droso_MolecularActivity = (
    select_cols(Gene_Droso_MolecularActivity)
)

Gene_Droso_MolecularActivity['Head_ID_IS'] = 'FlyBase'
Gene_Droso_MolecularActivity['Tail_ID_IS'] = 'Quick_GO'

Gene_Droso_MolecularActivity['Relation'] = ('Gene_MolecularFunction')

Gene_Droso_MolecularActivity['Tail_type'] = ('MolecularFunction')

Gene_Droso_MolecularActivity['Species'] = ('Drosophila melanogaster')

save(Gene_Droso_MolecularActivity,'Drosophila/Gene_Droso_MolecularFunction.csv')

Gene_Droso_MolecularActivity

Saved 38,735 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Drosophila/Gene_Droso_MolecularFunction.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,CG12725,Gene_MolecularFunction,GO:0003735,Gene,enables,MolecularFunction,infores:go-central,uncharacterized protein,structural constituent of ribosome,Drosophila melanogaster,NaN,FlyBase,Quick_GO,CG12725,Drosophila melanogaster
1,CG13623,Gene_MolecularFunction,GO:0051539,Gene,enables,MolecularFunction,infores:go-central,uncharacterized protein,"4 iron, 4 sulfur cluster binding",Drosophila melanogaster,NaN,FlyBase,Quick_GO,CG13623,Drosophila melanogaster
2,CG14250,Gene_MolecularFunction,GO:0008010,Gene,enables,MolecularFunction,infores:go-central,TweedleQ,structural constituent of chitin-based larval ...,Drosophila melanogaster,NaN,FlyBase,Quick_GO,TwdlQ,Drosophila melanogaster
3,CG15864,Gene_MolecularFunction,GO:0004656,Gene,enables,MolecularFunction,infores:go-central,uncharacterized protein,procollagen-proline 4-dioxygenase activity,Drosophila melanogaster,NaN,FlyBase,Quick_GO,CG15864,Drosophila melanogaster
4,CG4661,Gene_MolecularFunction,GO:0097602,Gene,enables,MolecularFunction,infores:go-central,uncharacterized protein,cullin family protein binding,Drosophila melanogaster,NaN,FlyBase,Quick_GO,CG4661,Drosophila melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38730,CG13225,Gene_MolecularFunction,GO:0004984,Gene,enables,MolecularFunction,infores:go-central,Odorant receptor 47a,olfactory receptor activity,Drosophila melanogaster,NaN,FlyBase,Quick_GO,Or47a,Drosophila melanogaster
38731,CG6057,Gene_MolecularFunction,GO:0003677,Gene,enables,MolecularFunction,infores:go-central,Structural maintenance of chromosomes 1,DNA binding,Drosophila melanogaster,NaN,FlyBase,Quick_GO,SMC1,Drosophila melanogaster
38732,CG42258,Gene_MolecularFunction,GO:0004402,Gene,enables,MolecularFunction,infores:go-central,uncharacterized protein,histone acetyltransferase activity,Drosophila melanogaster,NaN,FlyBase,Quick_GO,CG42258,Drosophila melanogaster
38733,CG5682,Gene_MolecularFunction,GO:0004571,Gene,enables,MolecularFunction,infores:go-central,"ER degradation enhancer, mannosidase alpha-like 2","mannosyl-oligosaccharide 1,2-alpha-mannosidase...",Drosophila melanogaster,NaN,FlyBase,Quick_GO,Edem2,Drosophila melanogaster


In [330]:
# Gene_Droso_Phenotype = pd.read_csv(
#     f'{Semi_processed}Drosophila/Gene_Droso_Phenotype_MONARCH.csv'
# )
# Gene_Droso_Phenotype

In [334]:
GENE_GENE_Droso_Droso = pd.read_csv(
    f'{Semi_processed}Drosophila/GENE_GENE_Droso_Droso_MONARCH.csv'
)

GENE_GENE_Droso_Droso[['Head_name', 'Head']] = GENE_GENE_Droso_Droso[['Head', 'Head_name']]
GENE_GENE_Droso_Droso[['Tail_name', 'Tail']] = GENE_GENE_Droso_Droso[['Tail', 'Tail_name']]

GENE_GENE_Droso_Droso['Head'] = (
    GENE_GENE_Droso_Droso['Head']
    .str.replace('FB:', '', regex=False)
)

GENE_GENE_Droso_Droso['Tail'] = (
    GENE_GENE_Droso_Droso['Tail']
    .str.replace('FB:', '', regex=False)
)

GENE_GENE_Droso_Droso['Head'] = (
    GENE_GENE_Droso_Droso['Head']
    .map(droso_ncbi_locustag_dict)
)

GENE_GENE_Droso_Droso['Tail'] = (
    GENE_GENE_Droso_Droso['Tail']
    .map(droso_ncbi_locustag_dict)
)

GENE_GENE_Droso_Droso['Head_Detail_name'] = (
    GENE_GENE_Droso_Droso['Head']
    .map(droso_ncbi_locustag_desc_dict)
)

GENE_GENE_Droso_Droso['Tail_Detail_name'] = (
    GENE_GENE_Droso_Droso['Tail']
    .map(droso_ncbi_locustag_desc_dict)
)

GENE_GENE_Droso_Droso = (
    GENE_GENE_Droso_Droso.rename(columns={
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
GENE_GENE_Droso_Droso = (
    GENE_GENE_Droso_Droso.loc[
        :,
        ~GENE_GENE_Droso_Droso.columns.duplicated()
    ]
)

# Keep only desired columns that exist
GENE_GENE_Droso_Droso = (
    select_cols(GENE_GENE_Droso_Droso)
)

GENE_GENE_Droso_Droso['Head_ID_IS'] = 'FlyBase'
GENE_GENE_Droso_Droso['Tail_ID_IS'] = 'FlyBase'

GENE_GENE_Droso_Droso['Relation'] = (
    'Gene_Gene'
)

GENE_GENE_Droso_Droso['Species'] = (
    'Drosophila melanogaster'
)

save(
    GENE_GENE_Droso_Droso,
    'Drosophila/GENE_GENE_Droso_Droso.csv'
)

GENE_GENE_Droso_Droso

Saved 196,222 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Drosophila/GENE_GENE_Droso_Droso.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,KG_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Tail_name,Species
0,CG43897,Gene_Gene,CG30084,Gene,interacts_with,Gene,infores:string,Monarch_KG,uncharacterized protein,Z band alternatively spliced PDZ-motif protein 52,Drosophila melanogaster,Drosophila melanogaster,FlyBase,FlyBase,CG43897,Zasp52,Drosophila melanogaster
1,CG43897,Gene_Gene,CG6416,Gene,interacts_with,Gene,infores:string,Monarch_KG,uncharacterized protein,Z band alternatively spliced PDZ-motif protein 66,Drosophila melanogaster,Drosophila melanogaster,FlyBase,FlyBase,CG43897,Zasp66,Drosophila melanogaster
2,CG18321,Gene_Gene,CG32845,Gene,interacts_with,Gene,infores:string,Monarch_KG,midkine and pleiotrophin 2,uncharacterized protein,Drosophila melanogaster,Drosophila melanogaster,FlyBase,FlyBase,miple2,CG32845,Drosophila melanogaster
3,CG10916,Gene_Gene,CG32633,Gene,interacts_with,Gene,infores:string,Monarch_KG,uncharacterized protein,uncharacterized protein,Drosophila melanogaster,Drosophila melanogaster,FlyBase,FlyBase,CG10916,CG32633,Drosophila melanogaster
4,CG11144,Gene_Gene,CG33513,Gene,interacts_with,Gene,infores:string,Monarch_KG,metabotropic Glutamate Receptor,NMDA receptor 2,Drosophila melanogaster,Drosophila melanogaster,FlyBase,FlyBase,mGluR,Nmdar2,Drosophila melanogaster
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196217,CG3125,Gene_Gene,CG5859,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,Integrator 6,Integrator 8,Drosophila melanogaster,Drosophila melanogaster,FlyBase,FlyBase,IntS6,IntS8,Drosophila melanogaster
196218,CG3125,Gene_Gene,CG3173,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,Integrator 6,Integrator 1,Drosophila melanogaster,Drosophila melanogaster,FlyBase,FlyBase,IntS6,IntS1,Drosophila melanogaster
196219,CG3125,Gene_Gene,CG8211,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,Integrator 6,Integrator 2,Drosophila melanogaster,Drosophila melanogaster,FlyBase,FlyBase,IntS6,IntS2,Drosophila melanogaster
196220,CG3125,Gene_Gene,CG1554,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,Integrator 6,RNA polymerase II subunit A,Drosophila melanogaster,Drosophila melanogaster,FlyBase,FlyBase,IntS6,Polr2A,Drosophila melanogaster


# Zebrafish

In [ ]:
##

In [363]:
zebra_ncbi = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Danio_rerio.gene_info',sep = '\t')
zebra_ncbi['LocusTag'] = zebra_ncbi['LocusTag'].str.replace('Dmel_', '', regex=False)
# Extract Zfin ID
zebra_ncbi['ZFIN_ID'] = (
    zebra_ncbi['dbXrefs']
    .str.extract(r'ZFIN:(ZDB-GENE-\d+-\d+)')
)

# Keep only useful columns
zebra_ncbi_clean = zebra_ncbi[
    [
        'Symbol',
        'LocusTag',
        'ZFIN_ID',
        'description'
    ]
]

# # Remove duplicates
zebra_ncbi_clean = zebra_ncbi_clean.drop_duplicates()

# # Remove rows with missing IDs
zebra_ncbi_clean = zebra_ncbi_clean[
    ~zebra_ncbi_clean['description'].isna()
]

zebra_ncbi_locustag_dict = dict(zip(zebra_ncbi_clean['ZFIN_ID'],zebra_ncbi_clean['Symbol']))
zebra_ncbi_locustag_desc_dict = dict(zip(zebra_ncbi_clean['Symbol'],zebra_ncbi_clean['description']))
zebra_ncbi_zfin_desc_dict = dict(zip(zebra_ncbi_clean['ZFIN_ID'],zebra_ncbi_clean['description']))
zebra_ncbi_locustag_desc_dict
zebra_ncbi_locustag_dict

zebra_ncbi_clean

,Symbol,LocusTag,ZFIN_ID,description
0,tncb,-,ZDB-GENE-980526-104,tenascin Cb
1,sox19a,-,ZDB-GENE-980526-102,SRY-box transcription factor 19a
2,elavl1b,-,ZDB-GENE-000210-17,ELAV like RNA binding protein 1b
3,meis2b,-,ZDB-GENE-000210-23,Meis homeobox 2b
4,chico,-,ZDB-GENE-000210-28,chico
...,...,...,...,...
51235,LOC137496866,-,NaN,uncharacterized LOC137496866
51236,LOC137496867,-,NaN,golgin subfamily A member 6-like protein 22
51237,LOC137496868,-,NaN,small nucleolar RNA SNORA30/SNORA37 family
51238,LOC137496869,-,NaN,small nucleolar RNA SNORA30/SNORA37 family


In [365]:
zebra_ncbi_locustag_dict

{'ZDB-GENE-980526-104': 'tncb',
 'ZDB-GENE-980526-102': 'sox19a',
 'ZDB-GENE-000210-17': 'elavl1b',
 'ZDB-GENE-000210-23': 'meis2b',
 'ZDB-GENE-000210-28': 'chico',
 'ZDB-GENE-000210-20': 'cat',
 'ZDB-GENE-991124-5': 'tbx4',
 'ZDB-GENE-991124-7': 'tbx5a',
 'ZDB-GENE-000210-21': 'inhbaa',
 'ZDB-GENE-000210-31': 'vdra',
 'ZDB-GENE-000210-19': 'rbp4',
 'ZDB-GENE-000210-13': 'nccrp1',
 'ZDB-GENE-000210-15': 'btg2',
 'ZDB-GENE-990910-11': 'mitfa',
 'ZDB-GENE-980605-23': 'ptpn1',
 'ZDB-GENE-000210-25': 'khdrbs1a',
 'ZDB-GENE-000210-32': 'nme2b.1',
 'ZDB-GENE-000210-33': 'nme2b.2',
 'ZDB-GENE-000210-34': 'nme3',
 'ZDB-GENE-000210-35': 'nme7',
 'ZDB-GENE-980605-2': 'za1',
 'ZDB-GENE-980605-7': 'g3b',
 'ZDB-GENE-030716-1': 'elk4',
 'ZDB-GENE-991110-20': 'slc39a7',
 'ZDB-GENE-980526-109': 'ckma',
 'ZDB-GENE-000208-4': 'zic1',
 'ZDB-GENE-980526-272': 'gli',
 'ZDB-GENE-980526-299': 'hhex',
 'ZDB-GENE-980605-9': 'ube2kb',
 'ZDB-GENE-990715-9': 'skia',
 'ZDB-GENE-980526-14': 'klfb',
 'ZDB-GENE-98052

In [364]:
zebra_ncbi_zfin_desc_dict

{'ZDB-GENE-980526-104': 'tenascin Cb',
 'ZDB-GENE-980526-102': 'SRY-box transcription factor 19a',
 'ZDB-GENE-000210-17': 'ELAV like RNA binding protein 1b',
 'ZDB-GENE-000210-23': 'Meis homeobox 2b',
 'ZDB-GENE-000210-28': 'chico',
 'ZDB-GENE-000210-20': 'catalase',
 'ZDB-GENE-991124-5': 'T-box transcription factor 4',
 'ZDB-GENE-991124-7': 'T-box transcription factor 5a',
 'ZDB-GENE-000210-21': 'inhibin subunit beta Aa',
 'ZDB-GENE-000210-31': 'vitamin D receptor a',
 'ZDB-GENE-000210-19': 'retinol binding protein 4, plasma',
 'ZDB-GENE-000210-13': 'P1, F-box associated domain containing',
 'ZDB-GENE-000210-15': 'B-cell translocation gene 2',
 'ZDB-GENE-990910-11': 'melanocyte inducing transcription factor a',
 'ZDB-GENE-980605-23': 'protein tyrosine phosphatase non-receptor type 1',
 'ZDB-GENE-000210-25': 'KH domain containing, RNA binding, signal transduction associated 1a',
 'ZDB-GENE-000210-32': 'NME/NM23 nucleoside diphosphate kinase 2b, tandem duplicate 1',
 'ZDB-GENE-000210-33

In [362]:
zebra_ncbi_locustag_desc_dict

{'tncb': 'tenascin Cb',
 'sox19a': 'SRY-box transcription factor 19a',
 'elavl1b': 'ELAV like RNA binding protein 1b',
 'meis2b': 'Meis homeobox 2b',
 'chico': 'chico',
 'cat': 'catalase',
 'tbx4': 'T-box transcription factor 4',
 'tbx5a': 'T-box transcription factor 5a',
 'inhbaa': 'inhibin subunit beta Aa',
 'vdra': 'vitamin D receptor a',
 'rbp4': 'retinol binding protein 4, plasma',
 'nccrp1': 'P1, F-box associated domain containing',
 'btg2': 'B-cell translocation gene 2',
 'mitfa': 'melanocyte inducing transcription factor a',
 'ptpn1': 'protein tyrosine phosphatase non-receptor type 1',
 'khdrbs1a': 'KH domain containing, RNA binding, signal transduction associated 1a',
 'nme2b.1': 'NME/NM23 nucleoside diphosphate kinase 2b, tandem duplicate 1',
 'nme2b.2': 'NME/NM23 nucleoside diphosphate kinase 2b, tandem duplicate 2',
 'nme3': 'NME/NM23 nucleoside diphosphate kinase 3',
 'nme7': 'NME/NM23 family member 7',
 'za1': 'ZA1',
 'g3b': 'g3b',
 'elk4': 'ETS transcription factor ELK4'

In [350]:
GENE_GENE_Zebrafish_Zebrafish = pd.read_csv(
    f'{Semi_processed}Zebrafish/GENE_GENE_Zebrafish_Zebrafish_MONARCH.csv'
)

GENE_GENE_Zebrafish_Zebrafish['Head_Detail_name'] = (
    GENE_GENE_Zebrafish_Zebrafish['Head']
    .map(zebra_ncbi_locustag_desc_dict))

GENE_GENE_Zebrafish_Zebrafish['Tail_Detail_name'] = (
    GENE_GENE_Zebrafish_Zebrafish['Tail']
    .map(zebra_ncbi_locustag_desc_dict))

GENE_GENE_Zebrafish_Zebrafish = (
    GENE_GENE_Zebrafish_Zebrafish.rename(columns={
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'}))

# Remove duplicate columns
GENE_GENE_Zebrafish_Zebrafish = (
    GENE_GENE_Zebrafish_Zebrafish.loc[
        :,
        ~GENE_GENE_Zebrafish_Zebrafish.columns.duplicated()])

# Keep only desired columns that exist
GENE_GENE_Zebrafish_Zebrafish = (
    select_cols(GENE_GENE_Zebrafish_Zebrafish))

GENE_GENE_Zebrafish_Zebrafish['Head_ID_IS'] = 'NCBI'
GENE_GENE_Zebrafish_Zebrafish['Tail_ID_IS'] = 'NCBI'

GENE_GENE_Zebrafish_Zebrafish['Relation'] = (
    'Gene_Gene')

GENE_GENE_Zebrafish_Zebrafish['Species'] = (
    'Danio rerio')

save(
    GENE_GENE_Zebrafish_Zebrafish,
    'Zebrafish/GENE_GENE_Zebrafish_Zebrafish.csv'
)

GENE_GENE_Zebrafish_Zebrafish

Saved 151,490 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/GENE_GENE_Zebrafish_Zebrafish.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,KG_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Tail_name,Species
0,ccdc80,Gene_Gene,faua,Gene,interacts_with,Gene,infores:string,Monarch_KG,coiled-coil domain containing 80,FAU ubiquitin like and ribosomal protein S30 f...,Danio rerio,Danio rerio,NCBI,NCBI,ZFIN:ZDB-GENE-030616-56,ZFIN:ZDB-GENE-040426-1700,Danio rerio
1,mcm6l,Gene_Gene,mcm3l,Gene,interacts_with,Gene,infores:string,Monarch_KG,"MCM6 minichromosome maintenance deficient 6, like",MCM3 minichromosome maintenance deficient 3 (S...,Danio rerio,Danio rerio,NCBI,NCBI,ZFIN:ZDB-GENE-050913-141,ZFIN:ZDB-GENE-040121-2,Danio rerio
2,mcm6l,Gene_Gene,rrm1,Gene,interacts_with,Gene,infores:string,Monarch_KG,"MCM6 minichromosome maintenance deficient 6, like",ribonucleotide reductase M1 polypeptide,Danio rerio,Danio rerio,NCBI,NCBI,ZFIN:ZDB-GENE-050913-141,ZFIN:ZDB-GENE-990415-247,Danio rerio
3,mcm6l,Gene_Gene,mcm9,Gene,interacts_with,Gene,infores:string,Monarch_KG,"MCM6 minichromosome maintenance deficient 6, like",minichromosome maintenance 9 homologous recomb...,Danio rerio,Danio rerio,NCBI,NCBI,ZFIN:ZDB-GENE-050913-141,ZFIN:ZDB-GENE-041014-310,Danio rerio
4,mcm6l,Gene_Gene,rad51,Gene,interacts_with,Gene,infores:string,Monarch_KG,"MCM6 minichromosome maintenance deficient 6, like",RAD51 recombinase,Danio rerio,Danio rerio,NCBI,NCBI,ZFIN:ZDB-GENE-050913-141,ZFIN:ZDB-GENE-040426-2286,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151485,bbs4,Gene_Gene,pcm1,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,Bardet-Biedl syndrome 4,pericentriolar material 1,Danio rerio,Danio rerio,NCBI,NCBI,ZFIN:ZDB-GENE-060126-2,ZFIN:ZDB-GENE-030131-428,Danio rerio
151486,bbs4,Gene_Gene,bbip1,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,Bardet-Biedl syndrome 4,BBSome interacting protein 1,Danio rerio,Danio rerio,NCBI,NCBI,ZFIN:ZDB-GENE-060126-2,ZFIN:ZDB-GENE-050208-445,Danio rerio
151487,bbip1,Gene_Gene,bbs4,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,BBSome interacting protein 1,Bardet-Biedl syndrome 4,Danio rerio,Danio rerio,NCBI,NCBI,ZFIN:ZDB-GENE-050208-445,ZFIN:ZDB-GENE-060126-2,Danio rerio
151488,bbip1,Gene_Gene,bbs1,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,BBSome interacting protein 1,Bardet-Biedl syndrome 1,Danio rerio,Danio rerio,NCBI,NCBI,ZFIN:ZDB-GENE-050208-445,ZFIN:ZDB-GENE-060126-1,Danio rerio


In [371]:
Gene_Zebra_Anatomy = pd.read_csv(
    f'{Semi_processed}Zebrafish/Gene_Zebra_Anatomy_MONARCH.csv'
)
Gene_Zebra_Anatomy['Head'] = (
    Gene_Zebra_Anatomy['Head']
    .str.replace('ZFIN:', '', regex=False)
)

Gene_Zebra_Anatomy['Head'] = Gene_Zebra_Anatomy['Head'].map(zebra_ncbi_locustag_dict)
Gene_Zebra_Anatomy['Head_Detail_name'] = Gene_Zebra_Anatomy['Head'].map(zebra_ncbi_locustag_desc_dict)


Gene_Zebra_Anatomy = (
    Gene_Zebra_Anatomy.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'}))

# Remove duplicate columns
Gene_Zebra_Anatomy = (
    Gene_Zebra_Anatomy.loc[
        :,
        ~Gene_Zebra_Anatomy.columns.duplicated()])

# Keep only desired columns that exist
Gene_Zebra_Anatomy = (
    select_cols(Gene_Zebra_Anatomy))

Gene_Zebra_Anatomy['Head_ID_IS'] = 'NCBI'
Gene_Zebra_Anatomy['Tail_ID_IS'] = 'ZFIN_ID'

Gene_Zebra_Anatomy['Relation'] = (
    'Gene_Gene')

Gene_Zebra_Anatomy['Species'] = (
    'Danio rerio')

save(
    Gene_Zebra_Anatomy,
    'Zebrafish/Gene_Zebra_Anatomy.csv'
)
Gene_Zebra_Anatomy

Saved 528,410 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Gene_Zebra_Anatomy.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,myb,Gene_Gene,ZFA:0000033,Gene,expressed_in,AnatomicalEntity,infores:zfin,v-myb avian myeloblastosis viral oncogene homolog,intermediate cell mass of mesoderm,Danio rerio,NaN,NCBI,ZFIN_ID,myb,Danio rerio
1,ptpn12,Gene_Gene,ZFA:0000079,Gene,expressed_in,AnatomicalEntity,infores:zfin,protein tyrosine phosphatase non-receptor type 12,telencephalon,Danio rerio,NaN,NCBI,ZFIN_ID,ptpn12,Danio rerio
2,fgf24,Gene_Gene,ZFA:0001453,Gene,expressed_in,AnatomicalEntity,infores:zfin,fibroblast growth factor 24,pectoral fin field,Danio rerio,NaN,NCBI,ZFIN_ID,fgf24,Danio rerio
3,gata5,Gene_Gene,ZFA:0000038,Gene,expressed_in,AnatomicalEntity,infores:zfin,GATA binding protein 5,margin,Danio rerio,NaN,NCBI,ZFIN_ID,gata5,Danio rerio
4,exo1,Gene_Gene,ZFA:0000098,Gene,expressed_in,AnatomicalEntity,infores:zfin,exonuclease 1,proliferative region,Danio rerio,NaN,NCBI,ZFIN_ID,exo1,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528405,im:6909183,Gene_Gene,ZFA:0001093,Gene,expressed_in,AnatomicalEntity,infores:zfin,im:6909183,unspecified,Danio rerio,NaN,NCBI,ZFIN_ID,im:6909183,Danio rerio
528406,kcnk5b,Gene_Gene,ZFA:0001093,Gene,expressed_in,AnatomicalEntity,infores:zfin,"potassium channel, subfamily K, member 5b",unspecified,Danio rerio,NaN,NCBI,ZFIN_ID,kcnk5b,Danio rerio
528407,pik3r4,Gene_Gene,ZFA:0001094,Gene,expressed_in,AnatomicalEntity,infores:zfin,"phosphoinositide-3-kinase, regulatory subunit 4",whole organism,Danio rerio,NaN,NCBI,ZFIN_ID,pik3r4,Danio rerio
528408,ywhaqb,Gene_Gene,ZFA:0001093,Gene,expressed_in,AnatomicalEntity,infores:zfin,tyrosine 3-monooxygenase/tryptophan 5-monooxyg...,unspecified,Danio rerio,NaN,NCBI,ZFIN_ID,ywhaqb,Danio rerio


In [374]:
Gene_Zebra_BiologicalProcess = pd.read_csv(
    f'{Semi_processed}Zebrafish/Gene_Zebra_BiologicalProcess_MONARCH.csv'
)

Gene_Zebra_BiologicalProcess['Head'] = (
    Gene_Zebra_BiologicalProcess['Head']
    .str.replace('ZFIN:', '', regex=False)
)

Gene_Zebra_BiologicalProcess['Head'] = (
    Gene_Zebra_BiologicalProcess['Head']
    .map(zebra_ncbi_locustag_dict)
)

Gene_Zebra_BiologicalProcess['Head_Detail_name'] = (
    Gene_Zebra_BiologicalProcess['Head']
    .map(zebra_ncbi_locustag_desc_dict)
)

Gene_Zebra_BiologicalProcess = (
    Gene_Zebra_BiologicalProcess.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Zebra_BiologicalProcess = (
    Gene_Zebra_BiologicalProcess.loc[
        :,
        ~Gene_Zebra_BiologicalProcess.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Zebra_BiologicalProcess = (
    select_cols(Gene_Zebra_BiologicalProcess)
)

Gene_Zebra_BiologicalProcess['Head_ID_IS'] = 'NCBI'
Gene_Zebra_BiologicalProcess['Tail_ID_IS'] = 'Quick_GO'

Gene_Zebra_BiologicalProcess['Relation'] = (
    'Gene_BiologicalProcess'
)

Gene_Zebra_BiologicalProcess['Tail_type'] = (
    'BiologicalProcess'
)

Gene_Zebra_BiologicalProcess['Species'] = (
    'Danio rerio'
)

save(
    Gene_Zebra_BiologicalProcess,
    'Zebrafish/Gene_Zebra_BiologicalProcess.csv'
)

Gene_Zebra_BiologicalProcess

Saved 91,909 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Gene_Zebra_BiologicalProcess.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,pbx1a,Gene_BiologicalProcess,GO:0001654,Gene,actively_involved_in,BiologicalProcess,infores:go-central,pre-B-cell leukemia homeobox 1a,eye development,Danio rerio,NaN,NCBI,Quick_GO,pbx1a,Danio rerio
1,zgc:77752,Gene_BiologicalProcess,GO:0060271,Gene,actively_involved_in,BiologicalProcess,infores:go-central,zgc:77752,cilium assembly,Danio rerio,NaN,NCBI,Quick_GO,zgc:77752,Danio rerio
2,pou3f2b,Gene_BiologicalProcess,GO:0006357,Gene,actively_involved_in,BiologicalProcess,infores:go-central,POU class 3 homeobox 2b,regulation of transcription by RNA polymerase II,Danio rerio,NaN,NCBI,Quick_GO,pou3f2b,Danio rerio
3,flvcr1,Gene_BiologicalProcess,GO:0030218,Gene,actively_involved_in,BiologicalProcess,infores:go-central,FLVCR choline and heme transporter 1,erythrocyte differentiation,Danio rerio,NaN,NCBI,Quick_GO,flvcr1,Danio rerio
4,cpdb,Gene_BiologicalProcess,GO:0006518,Gene,actively_involved_in,BiologicalProcess,infores:go-central,"carboxypeptidase D, b",peptide metabolic process,Danio rerio,NaN,NCBI,Quick_GO,cpdb,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91904,c9,Gene_BiologicalProcess,GO:0006958,Gene,acts_upstream_of_or_within,BiologicalProcess,infores:zfin,complement component 9,"complement activation, classical pathway",Danio rerio,NaN,NCBI,Quick_GO,c9,Danio rerio
91905,zgc:112271,Gene_BiologicalProcess,GO:0097428,Gene,acts_upstream_of_or_within,BiologicalProcess,infores:zfin,zgc:112271,protein maturation by iron-sulfur cluster tran...,Danio rerio,NaN,NCBI,Quick_GO,zgc:112271,Danio rerio
91906,rgrb,Gene_BiologicalProcess,GO:0007186,Gene,acts_upstream_of_or_within,BiologicalProcess,infores:zfin,retinal G protein coupled receptor b,G protein-coupled receptor signaling pathway,Danio rerio,NaN,NCBI,Quick_GO,rgrb,Danio rerio
91907,rgrb,Gene_BiologicalProcess,GO:0007186,Gene,acts_upstream_of_or_within,BiologicalProcess,infores:zfin,retinal G protein coupled receptor b,G protein-coupled receptor signaling pathway,Danio rerio,NaN,NCBI,Quick_GO,rgrb,Danio rerio


In [376]:
Gene_Zebra_CellularComponent = pd.read_csv(
    f'{Semi_processed}Zebrafish/Gene_Zebra_CellularComponent_MONARCH.csv'
)

Gene_Zebra_CellularComponent['Head'] = (
    Gene_Zebra_CellularComponent['Head']
    .str.replace('ZFIN:', '', regex=False)
)

Gene_Zebra_CellularComponent['Head'] = (
    Gene_Zebra_CellularComponent['Head']
    .map(zebra_ncbi_locustag_dict)
)

Gene_Zebra_CellularComponent['Head_Detail_name'] = (
    Gene_Zebra_CellularComponent['Head']
    .map(zebra_ncbi_locustag_desc_dict)
)

Gene_Zebra_CellularComponent = (
    Gene_Zebra_CellularComponent.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Zebra_CellularComponent = (
    Gene_Zebra_CellularComponent.loc[
        :,
        ~Gene_Zebra_CellularComponent.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Zebra_CellularComponent = (
    select_cols(Gene_Zebra_CellularComponent)
)

Gene_Zebra_CellularComponent['Head_ID_IS'] = 'NCBI'
Gene_Zebra_CellularComponent['Tail_ID_IS'] = 'Quick_GO'

Gene_Zebra_CellularComponent['Relation'] = (
    'Gene_CellularComponent'
)

Gene_Zebra_CellularComponent['Tail_type'] = (
    'CellularComponent'
)

Gene_Zebra_CellularComponent['Species'] = (
    'Danio rerio'
)

save(
    Gene_Zebra_CellularComponent,
    'Zebrafish/Gene_Zebra_CellularComponent.csv'
)

Gene_Zebra_CellularComponent

Saved 67,369 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Gene_Zebra_CellularComponent.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,plxnd1,Gene_CellularComponent,GO:0002116,Gene,part_of,CellularComponent,infores:go-central,plexin D1,semaphorin receptor complex,Danio rerio,NaN,NCBI,Quick_GO,plxnd1,Danio rerio
1,unc119b,Gene_CellularComponent,GO:0005929,Gene,is_active_in,CellularComponent,infores:go-central,unc-119 lipid binding chaperone B,cilium,Danio rerio,NaN,NCBI,Quick_GO,unc119b,Danio rerio
2,scarb2a,Gene_CellularComponent,GO:0005886,Gene,is_active_in,CellularComponent,infores:go-central,"scavenger receptor class B, member 2a",plasma membrane,Danio rerio,NaN,NCBI,Quick_GO,scarb2a,Danio rerio
3,exd2,Gene_CellularComponent,GO:0005737,Gene,is_active_in,CellularComponent,infores:go-central,exonuclease 3'-5' domain containing 2,cytoplasm,Danio rerio,NaN,NCBI,Quick_GO,exd2,Danio rerio
4,zgc:110045,Gene_CellularComponent,GO:0005737,Gene,is_active_in,CellularComponent,infores:go-central,zgc:110045,cytoplasm,Danio rerio,NaN,NCBI,Quick_GO,zgc:110045,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67364,tor1l1,Gene_CellularComponent,GO:0042995,Gene,located_in,CellularComponent,infores:zfin,torsin family 1 like 1,cell projection,Danio rerio,NaN,NCBI,Quick_GO,tor1l1,Danio rerio
67365,cenpq,Gene_CellularComponent,GO:0000775,Gene,located_in,CellularComponent,infores:zfin,centromere protein Q,"chromosome, centromeric region",Danio rerio,NaN,NCBI,Quick_GO,cenpq,Danio rerio
67366,cenpq,Gene_CellularComponent,GO:0005694,Gene,located_in,CellularComponent,infores:zfin,centromere protein Q,chromosome,Danio rerio,NaN,NCBI,Quick_GO,cenpq,Danio rerio
67367,cenpq,Gene_CellularComponent,GO:0005634,Gene,located_in,CellularComponent,infores:zfin,centromere protein Q,nucleus,Danio rerio,NaN,NCBI,Quick_GO,cenpq,Danio rerio


In [378]:
Gene_Zebra_MolecularActivity = pd.read_csv(
    f'{Semi_processed}Zebrafish/Gene_Zebra_MolecularActivity_MONARCH.csv'
)

Gene_Zebra_MolecularActivity['Head'] = (
    Gene_Zebra_MolecularActivity['Head']
    .str.replace('ZFIN:', '', regex=False)
)

Gene_Zebra_MolecularActivity['Head'] = (
    Gene_Zebra_MolecularActivity['Head']
    .map(zebra_ncbi_locustag_dict)
)

Gene_Zebra_MolecularActivity['Head_Detail_name'] = (
    Gene_Zebra_MolecularActivity['Head']
    .map(zebra_ncbi_locustag_desc_dict)
)

Gene_Zebra_MolecularActivity = (
    Gene_Zebra_MolecularActivity.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Zebra_MolecularActivity = (
    Gene_Zebra_MolecularActivity.loc[
        :,
        ~Gene_Zebra_MolecularActivity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Zebra_MolecularActivity = (
    select_cols(Gene_Zebra_MolecularActivity)
)

Gene_Zebra_MolecularActivity['Head_ID_IS'] = 'NCBI'
Gene_Zebra_MolecularActivity['Tail_ID_IS'] = 'Quick_GO'

Gene_Zebra_MolecularActivity['Relation'] = (
    'Gene_MolecularFunction'
)

Gene_Zebra_MolecularActivity['Tail_type'] = (
    'MolecularFunction'
)

Gene_Zebra_MolecularActivity['Species'] = (
    'Danio rerio'
)

save(
    Gene_Zebra_MolecularActivity,
    'Zebrafish/Gene_Zebra_MolecularFunction.csv'
)

Gene_Zebra_MolecularActivity

Saved 87,998 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Gene_Zebra_MolecularFunction.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,dnm2a,Gene_MolecularFunction,GO:0003924,Gene,enables,MolecularFunction,infores:go-central,dynamin 2a,GTPase activity,Danio rerio,NaN,NCBI,Quick_GO,dnm2a,Danio rerio
1,triobpa,Gene_MolecularFunction,GO:0051015,Gene,enables,MolecularFunction,infores:go-central,TRIO and F-actin binding protein a,actin filament binding,Danio rerio,NaN,NCBI,Quick_GO,triobpa,Danio rerio
2,creb3l3l,Gene_MolecularFunction,GO:0000981,Gene,enables,MolecularFunction,infores:go-central,cAMP responsive element binding protein 3-like...,"DNA-binding transcription factor activity, RNA...",Danio rerio,NaN,NCBI,Quick_GO,creb3l3l,Danio rerio
3,zgc:158412,Gene_MolecularFunction,GO:0043565,Gene,enables,MolecularFunction,infores:go-central,zgc:158412,sequence-specific DNA binding,Danio rerio,NaN,NCBI,Quick_GO,zgc:158412,Danio rerio
4,gbx1,Gene_MolecularFunction,GO:0000977,Gene,enables,MolecularFunction,infores:go-central,gastrulation brain homeobox 1,RNA polymerase II transcription regulatory reg...,Danio rerio,NaN,NCBI,Quick_GO,gbx1,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87993,tbxas1,Gene_MolecularFunction,GO:0004796,Gene,enables,MolecularFunction,infores:zfin,thromboxane A synthase 1 (platelet),thromboxane-A synthase activity,Danio rerio,NaN,NCBI,Quick_GO,tbxas1,Danio rerio
87994,crot,Gene_MolecularFunction,GO:0008458,Gene,enables,MolecularFunction,infores:zfin,carnitine O-octanoyltransferase,carnitine O-octanoyltransferase activity,Danio rerio,NaN,NCBI,Quick_GO,crot,Danio rerio
87995,echdc1,Gene_MolecularFunction,GO:0004492,Gene,enables,MolecularFunction,infores:zfin,enoyl CoA hydratase domain containing 1,methyl/ethyl malonyl-CoA decarboxylase activity,Danio rerio,NaN,NCBI,Quick_GO,echdc1,Danio rerio
87996,ssh1b,Gene_MolecularFunction,GO:0017018,Gene,enables,MolecularFunction,infores:zfin,slingshot protein phosphatase 1b,myosin phosphatase activity,Danio rerio,NaN,NCBI,Quick_GO,ssh1b,Danio rerio


In [380]:
Gene_Zebra_Phenotype = pd.read_csv(
    f'{Semi_processed}Zebrafish/Gene_Zebra_Phenotype_MONARCH.csv'
)

Gene_Zebra_Phenotype['Head'] = (
    Gene_Zebra_Phenotype['Head']
    .str.replace('ZFIN:', '', regex=False)
)

Gene_Zebra_Phenotype['Head'] = (
    Gene_Zebra_Phenotype['Head']
    .map(zebra_ncbi_locustag_dict)
)

Gene_Zebra_Phenotype['Head_Detail_name'] = (
    Gene_Zebra_Phenotype['Head']
    .map(zebra_ncbi_locustag_desc_dict)
)

Gene_Zebra_Phenotype = (
    Gene_Zebra_Phenotype.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Zebra_Phenotype = (
    Gene_Zebra_Phenotype.loc[
        :,
        ~Gene_Zebra_Phenotype.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Zebra_Phenotype = (
    select_cols(Gene_Zebra_Phenotype)
)

Gene_Zebra_Phenotype['Head_ID_IS'] = 'NCBI'
Gene_Zebra_Phenotype['Tail_ID_IS'] = 'ZFIN_ID'

Gene_Zebra_Phenotype['Relation'] = (
    'Gene_Phenotype'
)

Gene_Zebra_Phenotype['Tail_type'] = (
    'Phenotype'
)

Gene_Zebra_Phenotype['Species'] = (
    'Danio rerio'
)

save(
    Gene_Zebra_Phenotype,
    'Zebrafish/Gene_Zebra_Phenotype.csv'
)

Gene_Zebra_Phenotype

Saved 156,719 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Gene_Zebra_Phenotype.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,fbn2b,Gene_Phenotype,ZP:0001452,Gene,has_phenotype,Phenotype,infores:zfin,fibrillin 2b,"trunk edematous, abnormal",Danio rerio,NaN,NCBI,ZFIN_ID,fbn2b,Danio rerio
1,fbn2b,Gene_Phenotype,ZP:0009282,Gene,has_phenotype,Phenotype,infores:zfin,fibrillin 2b,"caudal vein plexus disorganized, abnormal",Danio rerio,NaN,NCBI,ZFIN_ID,fbn2b,Danio rerio
2,fbn2b,Gene_Phenotype,ZP:0001556,Gene,has_phenotype,Phenotype,infores:zfin,fibrillin 2b,"notochord kinked, abnormal",Danio rerio,NaN,NCBI,ZFIN_ID,fbn2b,Danio rerio
3,fbn2b,Gene_Phenotype,ZP:0009277,Gene,has_phenotype,Phenotype,infores:zfin,fibrillin 2b,"caudal vein structure, cavities, abnormal",Danio rerio,NaN,NCBI,ZFIN_ID,fbn2b,Danio rerio
4,fbn2b,Gene_Phenotype,ZP:0009275,Gene,has_phenotype,Phenotype,infores:zfin,fibrillin 2b,"median fin fold attenuate, abnormal",Danio rerio,NaN,NCBI,ZFIN_ID,fbn2b,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156714,fbn2b,Gene_Phenotype,ZP:0009277,Gene,has_phenotype,Phenotype,infores:zfin,fibrillin 2b,"caudal vein structure, cavities, abnormal",Danio rerio,NaN,NCBI,ZFIN_ID,fbn2b,Danio rerio
156715,fbn2b,Gene_Phenotype,ZP:0009275,Gene,has_phenotype,Phenotype,infores:zfin,fibrillin 2b,"median fin fold attenuate, abnormal",Danio rerio,NaN,NCBI,ZFIN_ID,fbn2b,Danio rerio
156716,fbn2b,Gene_Phenotype,ZP:0009275,Gene,has_phenotype,Phenotype,infores:zfin,fibrillin 2b,"median fin fold attenuate, abnormal",Danio rerio,NaN,NCBI,ZFIN_ID,fbn2b,Danio rerio
156717,fbn2b,Gene_Phenotype,ZP:0001556,Gene,has_phenotype,Phenotype,infores:zfin,fibrillin 2b,"notochord kinked, abnormal",Danio rerio,NaN,NCBI,ZFIN_ID,fbn2b,Danio rerio


In [384]:
Zebra_AnatomicalEntity_AnatomicalEntity = pd.read_csv(
    f'{Semi_processed}Zebrafish/Zebra_AnatomicalEntity_AnatomicalEntity_MONARCH.csv'
)

Zebra_AnatomicalEntity_AnatomicalEntity = (
    Zebra_AnatomicalEntity_AnatomicalEntity.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Zebra_AnatomicalEntity_AnatomicalEntity = (
    Zebra_AnatomicalEntity_AnatomicalEntity.loc[
        :,
        ~Zebra_AnatomicalEntity_AnatomicalEntity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Zebra_AnatomicalEntity_AnatomicalEntity = (
    select_cols(Zebra_AnatomicalEntity_AnatomicalEntity)
)

Zebra_AnatomicalEntity_AnatomicalEntity['Head_ID_IS'] = 'ZFIN_ID'
Zebra_AnatomicalEntity_AnatomicalEntity['Tail_ID_IS'] = 'ZFIN_ID'

Zebra_AnatomicalEntity_AnatomicalEntity['Head_type'] = (
    'Anatomy'
)
Zebra_AnatomicalEntity_AnatomicalEntity['Tail_type'] = (
    'Anatomy'
)
Zebra_AnatomicalEntity_AnatomicalEntity['Relation'] = (
    'Anatomy_Anatomy'
)

Zebra_AnatomicalEntity_AnatomicalEntity['Species'] = (
    'Danio rerio'
)

save(
    Zebra_AnatomicalEntity_AnatomicalEntity,
    'Zebrafish/Zebra_AnatomicalEntity_AnatomicalEntity.csv'
)

Zebra_AnatomicalEntity_AnatomicalEntity

Saved 5,951 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Zebra_AnatomicalEntity_AnatomicalEntity.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,ZFA:0000000,Anatomy_Anatomy,ZFA:0001105,Anatomy,subclass_of,Anatomy,infores:zfa,Brachet's cleft,embryonic structure,Danio rerio,Danio rerio,ZFIN_ID,ZFIN_ID,Danio rerio
1,ZFA:0000001,Anatomy_Anatomy,ZFA:0001105,Anatomy,subclass_of,Anatomy,infores:zfa,Kupffer's vesicle,embryonic structure,Danio rerio,Danio rerio,ZFIN_ID,ZFIN_ID,Danio rerio
2,ZFA:0000001,Anatomy_Anatomy,ZFA:0000077,Anatomy,related_to,Anatomy,infores:zfa,Kupffer's vesicle,tail bud,Danio rerio,Danio rerio,ZFIN_ID,ZFIN_ID,Danio rerio
3,ZFA:0000001,Anatomy_Anatomy,ZFA:0000023,Anatomy,related_to,Anatomy,infores:zfa,Kupffer's vesicle,forerunner cell group,Danio rerio,Danio rerio,ZFIN_ID,ZFIN_ID,Danio rerio
4,ZFA:0000003,Anatomy_Anatomy,ZFA:0009000,Anatomy,subclass_of,Anatomy,infores:zfa,adaxial cell,cell,Danio rerio,Danio rerio,ZFIN_ID,ZFIN_ID,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5946,ZFA:0009401,Anatomy_Anatomy,ZFA:0000035,Anatomy,related_to,Anatomy,infores:zfa,lens fiber cell,lens,Danio rerio,Danio rerio,ZFIN_ID,ZFIN_ID,Danio rerio
5947,ZFA:0009402,Anatomy_Anatomy,ZFA:0009000,Anatomy,subclass_of,Anatomy,infores:zfa,heart valve cell,cell,Danio rerio,Danio rerio,ZFIN_ID,ZFIN_ID,Danio rerio
5948,ZFA:0009402,Anatomy_Anatomy,ZFA:0005065,Anatomy,related_to,Anatomy,infores:zfa,heart valve cell,heart valve,Danio rerio,Danio rerio,ZFIN_ID,ZFIN_ID,Danio rerio
5949,ZFA:0009403,Anatomy_Anatomy,ZFA:0009402,Anatomy,subclass_of,Anatomy,infores:zfa,heart valve interstitial cell,heart valve cell,Danio rerio,Danio rerio,ZFIN_ID,ZFIN_ID,Danio rerio


In [387]:
Zebra_Gene_Pathway = pd.read_csv(
    f'{Semi_processed}Zebrafish/Zebra_Gene_Pathway_MONARCH.csv'
)

Zebra_Gene_Pathway['Head'] = (
    Zebra_Gene_Pathway['Head']
    .str.replace('ZFIN:', '', regex=False)
)

Zebra_Gene_Pathway['Head'] = (
    Zebra_Gene_Pathway['Head']
    .map(zebra_ncbi_locustag_dict)
)

Zebra_Gene_Pathway['Head_Detail_name'] = (
    Zebra_Gene_Pathway['Head']
    .map(zebra_ncbi_locustag_desc_dict)
)

Zebra_Gene_Pathway = (
    Zebra_Gene_Pathway.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Zebra_Gene_Pathway = (
    Zebra_Gene_Pathway.loc[
        :,
        ~Zebra_Gene_Pathway.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Zebra_Gene_Pathway = (
    select_cols(Zebra_Gene_Pathway)
)

Zebra_Gene_Pathway['Head_ID_IS'] = 'NCBI'

Zebra_Gene_Pathway['Relation'] = (
    'Gene_Pathway'
)

Zebra_Gene_Pathway['Species'] = (
    'Danio rerio'
)

save(Zebra_Gene_Pathway,'Zebrafish/Zebra_Gene_Pathway.csv')

Zebra_Gene_Pathway

Saved 10,398 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Zebra_Gene_Pathway.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,parvaa,Gene_Pathway,R-HSA-446343,Gene,participates_in,Pathway,infores:reactome,"parvin, alpha a",Localization of the PINCH-ILK-PARVIN complex t...,Danio rerio,Homo sapiens,NCBI,Reactome,parvaa,Danio rerio
1,parvaa,Gene_Pathway,R-HSA-446388,Gene,participates_in,Pathway,infores:reactome,"parvin, alpha a",Regulation of cytoskeletal remodeling and cell...,Danio rerio,Homo sapiens,NCBI,Reactome,parvaa,Danio rerio
2,oxct1a,Gene_Pathway,R-HSA-77108,Gene,participates_in,Pathway,infores:reactome,3-oxoacid CoA transferase 1a,Utilization of Ketone Bodies,Danio rerio,Homo sapiens,NCBI,Reactome,oxct1a,Danio rerio
3,ctdp1,Gene_Pathway,R-HSA-6796648,Gene,participates_in,Pathway,infores:reactome,"CTD (carboxy-terminal domain, RNA polymerase I...",TP53 Regulates Transcription of DNA Repair Genes,Danio rerio,Homo sapiens,NCBI,Reactome,ctdp1,Danio rerio
4,sumo2b,Gene_Pathway,R-HSA-196791,Gene,participates_in,Pathway,infores:reactome,small ubiquitin like modifier 2b,Vitamin D (calciferol) metabolism,Danio rerio,Homo sapiens,NCBI,Reactome,sumo2b,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10393,tyrp1b,Gene_Pathway,R-HSA-5662702,Gene,participates_in,Pathway,infores:reactome,tyrosinase-related protein 1b,Melanin biosynthesis,Danio rerio,Homo sapiens,NCBI,Reactome,tyrp1b,Danio rerio
10394,rab30,Gene_Pathway,R-HSA-8873719,Gene,participates_in,Pathway,infores:reactome,"RAB30, member RAS oncogene family",RAB geranylgeranylation,Danio rerio,Homo sapiens,NCBI,Reactome,rab30,Danio rerio
10395,ikbke,Gene_Pathway,R-HSA-9860927,Gene,participates_in,Pathway,infores:reactome,inhibitor of nuclear factor kappa B kinase sub...,"Turbulent (oscillatory, disturbed) flow shear ...",Danio rerio,Homo sapiens,NCBI,Reactome,ikbke,Danio rerio
10396,rac3a,Gene_Pathway,R-HSA-4086400,Gene,participates_in,Pathway,infores:reactome,Rac family small GTPase 3a,PCP/CE pathway,Danio rerio,Homo sapiens,NCBI,Reactome,rac3a,Danio rerio


In [389]:
# Zebra_Genotype_Disease = pd.read_csv(  #not keeping Genotype node
#     f'{Semi_processed}Zebrafish/Zebra_Genotype_Disease_Monarch.csv'
# )
# Zebra_Genotype_Disease

In [392]:
Zebra_Phenotype_CellularComponent = pd.read_csv(
    f'{Semi_processed}Zebrafish/Zebra_Phenotype_CellularComponent_Monarch.csv'
)

Zebra_Phenotype_CellularComponent = (
    Zebra_Phenotype_CellularComponent.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Zebra_Phenotype_CellularComponent = (
    Zebra_Phenotype_CellularComponent.loc[
        :,
        ~Zebra_Phenotype_CellularComponent.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Zebra_Phenotype_CellularComponent = (
    select_cols(Zebra_Phenotype_CellularComponent)
)

Zebra_Phenotype_CellularComponent['Head_ID_IS'] = 'ZFIN_ID'
Zebra_Phenotype_CellularComponent['Tail_ID_IS'] = 'Quick_GO'

Zebra_Phenotype_CellularComponent['Relation'] = (
    'Phenotype_CellularComponent'
)

Zebra_Phenotype_CellularComponent['Head_type'] = (
    'Phenotype'
)

Zebra_Phenotype_CellularComponent['Tail_type'] = (
    'CellularComponent'
)

Zebra_Phenotype_CellularComponent['Species'] = (
    'Danio rerio'
)

save(
    Zebra_Phenotype_CellularComponent,
    'Zebrafish/Zebra_Phenotype_CellularComponent.csv'
)

Zebra_Phenotype_CellularComponent

Saved 5,378 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Zebra_Phenotype_CellularComponent.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,ZP:0019682,Phenotype_CellularComponent,GO:0030424,Phenotype,related_to,CellularComponent,infores:upheno,axon commissure of the tract of the commissure...,axon,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
1,ZP:0019700,Phenotype_CellularComponent,GO:0044297,Phenotype,related_to,CellularComponent,infores:upheno,"cell body photoreceptor cell mislocalised, abn...",cell body,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
2,ZP:0019701,Phenotype_CellularComponent,GO:0060091,Phenotype,related_to,CellularComponent,infores:upheno,"kinocilium hair cell disorganized, abnormal",kinocilium,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
3,ZP:0019702,Phenotype_CellularComponent,GO:0005929,Phenotype,related_to,CellularComponent,infores:upheno,"cilium olfactory pit swollen, abnormal",cilium,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
4,ZP:0019703,Phenotype_CellularComponent,GO:0060091,Phenotype,related_to,CellularComponent,infores:upheno,"kinocilium hair cell decreased length, abnormal",kinocilium,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5373,ZP:0019604,Phenotype_CellularComponent,GO:0045495,Phenotype,related_to,CellularComponent,infores:upheno,pole plasm oocyte stage I increased distributi...,pole plasm,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
5374,ZP:0019605,Phenotype_CellularComponent,GO:0031514,Phenotype,related_to,CellularComponent,infores:upheno,motile cilium Kupffer's vesicle increased leng...,motile cilium,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
5375,ZP:0019607,Phenotype_CellularComponent,GO:0005892,Phenotype,related_to,CellularComponent,infores:upheno,acetylcholine-gated channel complex myotome di...,acetylcholine-gated channel complex,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
5376,ZP:0019608,Phenotype_CellularComponent,GO:0005892,Phenotype,related_to,CellularComponent,infores:upheno,acetylcholine-gated channel complex myotome mi...,acetylcholine-gated channel complex,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio


In [394]:
Zebra_Phenotype_ChemicalEntity = pd.read_csv(
    f'{Semi_processed}Zebrafish/Zebra_Phenotype_ChemicalEntity_Monarch.csv'
)

Zebra_Phenotype_ChemicalEntity = (
    Zebra_Phenotype_ChemicalEntity.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Zebra_Phenotype_ChemicalEntity = (
    Zebra_Phenotype_ChemicalEntity.loc[
        :,
        ~Zebra_Phenotype_ChemicalEntity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Zebra_Phenotype_ChemicalEntity = (
    select_cols(Zebra_Phenotype_ChemicalEntity)
)

Zebra_Phenotype_ChemicalEntity['Head_ID_IS'] = 'ZFIN_ID'
Zebra_Phenotype_ChemicalEntity['Tail_ID_IS'] = 'PubChem'

Zebra_Phenotype_ChemicalEntity['Relation'] = (
    'Phenotype_ChemicalEntity'
)

Zebra_Phenotype_ChemicalEntity['Head_type'] = (
    'Phenotype'
)

Zebra_Phenotype_ChemicalEntity['Tail_type'] = (
    'ChemicalEntity'
)

Zebra_Phenotype_ChemicalEntity['Tail_IUPAC_name'] = (
    Zebra_Phenotype_ChemicalEntity['Tail']
    .astype(str)
    .map(Pubchem_IUPAC_CID_Dict)
)

Zebra_Phenotype_ChemicalEntity['Species'] = (
    'Danio rerio'
)

save(
    Zebra_Phenotype_ChemicalEntity,
    'Zebrafish/Zebra_Phenotype_ChemicalEntity.csv'
)

Zebra_Phenotype_ChemicalEntity

Saved 702 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Zebra_Phenotype_ChemicalEntity.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Tail_IUPAC_name,Species
0,ZP:0020710,Phenotype_ChemicalEntity,951,Phenotype,related_to,ChemicalEntity,infores:upheno,"noradrenaline brain decreased amount, abnormal",noradrenaline,Danio rerio,NaN,ZFIN_ID,PubChem,"4-(2-amino-1-hydroxyethyl)benzene-1,2-diol",Danio rerio
1,ZP:0020711,Phenotype_ChemicalEntity,5202,Phenotype,related_to,ChemicalEntity,infores:upheno,"serotonin brain decreased amount, abnormal",serotonin,Danio rerio,NaN,ZFIN_ID,PubChem,3-(2-aminoethyl)-1H-indol-5-ol,Danio rerio
2,ZP:0020712,Phenotype_ChemicalEntity,1826,Phenotype,related_to,ChemicalEntity,infores:upheno,(5-hydroxyindol-3-yl)acetic acid brain increas...,(5-hydroxyindol-3-yl)acetic acid,Danio rerio,NaN,ZFIN_ID,PubChem,2-(5-hydroxy-1H-indol-3-yl)acetic acid,Danio rerio
3,ZP:0020713,Phenotype_ChemicalEntity,681,Phenotype,related_to,ChemicalEntity,infores:upheno,"dopamine brain decreased amount, abnormal",dopamine,Danio rerio,NaN,ZFIN_ID,PubChem,"4-(2-aminoethyl)benzene-1,2-diol",Danio rerio
4,ZP:0021626,Phenotype_ChemicalEntity,681,Phenotype,related_to,ChemicalEntity,infores:upheno,"dopamine whole organism amount, abnormal",dopamine,Danio rerio,NaN,ZFIN_ID,PubChem,"4-(2-aminoethyl)benzene-1,2-diol",Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
697,ZP:0018886,Phenotype_ChemicalEntity,617,Phenotype,related_to,ChemicalEntity,infores:upheno,"serine liver decreased amount, abnormal",serine,Danio rerio,NaN,ZFIN_ID,PubChem,2-amino-3-hydroxypropanoic acid,Danio rerio
698,ZP:0018888,Phenotype_ChemicalEntity,876,Phenotype,related_to,ChemicalEntity,infores:upheno,"methionine liver decreased amount, abnormal",methionine,Danio rerio,NaN,ZFIN_ID,PubChem,2-amino-4-methylsulfanylbutanoic acid,Danio rerio
699,ZP:0019009,Phenotype_ChemicalEntity,5538,Phenotype,related_to,ChemicalEntity,infores:upheno,"retinoic acid whole organism increased amount,...",retinoic acid,Danio rerio,NaN,ZFIN_ID,PubChem,"3,7-dimethyl-9-(2,6,6-trimethylcyclohexen-1-yl...",Danio rerio
700,ZP:0019312,Phenotype_ChemicalEntity,24742074,Phenotype,related_to,ChemicalEntity,infores:upheno,"1-phosphatidyl-1D-myo-inositol 4,5-bisphosphat...","1-phosphatidyl-1D-myo-inositol 4,5-bisphosphate",Danio rerio,NaN,ZFIN_ID,PubChem,"[(1R,2S,3R,4R,5S,6R)-2,3,5-trihydroxy-4-[[2-[(...",Danio rerio


In [397]:
Zebra_PhenotypicFeature_BiologicalProcess = pd.read_csv(
    f'{Semi_processed}Zebrafish/Zebra_PhenotypicFeature_BiologicalProcess_Monarch.csv'
)

Zebra_PhenotypicFeature_BiologicalProcess = (
    Zebra_PhenotypicFeature_BiologicalProcess.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Zebra_PhenotypicFeature_BiologicalProcess = (
    Zebra_PhenotypicFeature_BiologicalProcess.loc[
        :,
        ~Zebra_PhenotypicFeature_BiologicalProcess.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Zebra_PhenotypicFeature_BiologicalProcess = (
    select_cols(Zebra_PhenotypicFeature_BiologicalProcess)
)

Zebra_PhenotypicFeature_BiologicalProcess['Head_ID_IS'] = 'ZFIN_ID'
Zebra_PhenotypicFeature_BiologicalProcess['Tail_ID_IS'] = 'Quick_GO'

Zebra_PhenotypicFeature_BiologicalProcess['Relation'] = (
    'Phenotype_BiologicalProcess'
)

Zebra_PhenotypicFeature_BiologicalProcess['Head_type'] = (
    'Phenotype'
)

Zebra_PhenotypicFeature_BiologicalProcess['Tail_type'] = (
    'BiologicalProcess'
)

Zebra_PhenotypicFeature_BiologicalProcess['Species'] = (
    'Danio rerio'
)

save(
    Zebra_PhenotypicFeature_BiologicalProcess,
    'Zebrafish/Zebra_PhenotypicFeature_BiologicalProcess.csv'
)

Zebra_PhenotypicFeature_BiologicalProcess

Saved 10,674 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Zebra_PhenotypicFeature_BiologicalProcess.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,ZP:0019661,Phenotype_BiologicalProcess,GO:0001755,Phenotype,related_to,BiologicalProcess,infores:upheno,"trunk delayed neural crest cell migration, abn...",neural crest cell migration,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
1,ZP:0019668,Phenotype_BiologicalProcess,GO:0030902,Phenotype,related_to,BiologicalProcess,infores:upheno,"hindbrain development process quality, abnormal",hindbrain development,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
2,ZP:0019677,Phenotype_BiologicalProcess,GO:0007409,Phenotype,related_to,BiologicalProcess,infores:upheno,"motor neuron disrupted axonogenesis, abnormal",axonogenesis,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
3,ZP:0019678,Phenotype_BiologicalProcess,GO:0061564,Phenotype,related_to,BiologicalProcess,infores:upheno,"motor neuron disrupted axon development, abnormal",axon development,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
4,ZP:0019680,Phenotype_BiologicalProcess,GO:0071679,Phenotype,related_to,BiologicalProcess,infores:upheno,commissure of the tract of the commissure of t...,commissural neuron axon guidance,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10669,ZP:0019642,Phenotype_BiologicalProcess,GO:0030431,Phenotype,related_to,BiologicalProcess,infores:upheno,"sleep behavioral quality of a process, abnormal",sleep,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
10670,ZP:0019644,Phenotype_BiologicalProcess,GO:0042063,Phenotype,related_to,BiologicalProcess,infores:upheno,"gliogenesis decreased process quality, abnormal",gliogenesis,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
10671,ZP:0019645,Phenotype_BiologicalProcess,GO:0031017,Phenotype,related_to,BiologicalProcess,infores:upheno,"exocrine pancreas development arrested, abnormal",exocrine pancreas development,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
10672,ZP:0019646,Phenotype_BiologicalProcess,GO:0007411,Phenotype,related_to,BiologicalProcess,infores:upheno,motor neuron decreased process quality axon gu...,axon guidance,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio


In [400]:
Zebra_PhenotypicFeature_MolecularActivity = pd.read_csv(
    f'{Semi_processed}Zebrafish/Zebra_PhenotypicFeature_MolecularActivity_Monarch.csv'
)
Zebra_PhenotypicFeature_MolecularActivity = (
    Zebra_PhenotypicFeature_MolecularActivity.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Zebra_PhenotypicFeature_MolecularActivity = (
    Zebra_PhenotypicFeature_MolecularActivity.loc[
        :,
        ~Zebra_PhenotypicFeature_MolecularActivity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Zebra_PhenotypicFeature_MolecularActivity = (
    select_cols(Zebra_PhenotypicFeature_MolecularActivity)
)

Zebra_PhenotypicFeature_MolecularActivity['Head_ID_IS'] = 'ZFIN_ID'
Zebra_PhenotypicFeature_MolecularActivity['Tail_ID_IS'] = 'Quick_GO'

Zebra_PhenotypicFeature_MolecularActivity['Relation'] = (
    'Phenotype_MolecularFunction'
)

Zebra_PhenotypicFeature_MolecularActivity['Head_type'] = (
    'Phenotype'
)

Zebra_PhenotypicFeature_MolecularActivity['Tail_type'] = (
    'MolecularFunction'
)

Zebra_PhenotypicFeature_MolecularActivity['Species'] = (
    'Danio rerio'
)

save(
    Zebra_PhenotypicFeature_MolecularActivity,
    'Zebrafish/Zebra_PhenotypicFeature_MolecularFunction.csv'
)
Zebra_PhenotypicFeature_MolecularActivity

Saved 455 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Zebrafish/Zebra_PhenotypicFeature_MolecularFunction.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,ZP:0019874,Phenotype_MolecularFunction,GO:0004708,Phenotype,related_to,MolecularFunction,infores:upheno,"MAP kinase kinase activity process quality, ab...",MAP kinase kinase activity,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
1,ZP:0019951,Phenotype_MolecularFunction,GO:0072320,Phenotype,related_to,MolecularFunction,infores:upheno,volume-sensitive chloride channel activity dec...,volume-sensitive chloride channel activity,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
2,ZP:0020011,Phenotype_MolecularFunction,GO:0097153,Phenotype,related_to,MolecularFunction,infores:upheno,cysteine-type endopeptidase activity involved ...,obsolete cysteine-type endopeptidase activity ...,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
3,ZP:0020439,Phenotype_MolecularFunction,GO:0008475,Phenotype,related_to,MolecularFunction,infores:upheno,procollagen-lysine 5-dioxygenase activity disr...,procollagen-lysine 5-dioxygenase activity,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
4,ZP:0021430,Phenotype_MolecularFunction,GO:0008086,Phenotype,related_to,MolecularFunction,infores:upheno,light-activated voltage-gated calcium channel ...,light-activated voltage-gated calcium channel ...,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,ZP:0019079,Phenotype_MolecularFunction,GO:0004800,Phenotype,related_to,MolecularFunction,infores:upheno,"thyroxine 5'-deiodinase activity disrupted, ab...",thyroxine 5'-deiodinase activity,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
451,ZP:0019080,Phenotype_MolecularFunction,GO:0004800,Phenotype,related_to,MolecularFunction,infores:upheno,thyroxine 5'-deiodinase activity decreased pro...,thyroxine 5'-deiodinase activity,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
452,ZP:0019486,Phenotype_MolecularFunction,GO:0004348,Phenotype,related_to,MolecularFunction,infores:upheno,glucosylceramidase activity decreased process ...,glucosylceramidase activity,Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio
453,ZP:0019568,Phenotype_MolecularFunction,GO:0016772,Phenotype,related_to,MolecularFunction,infores:upheno,"transferase activity, transferring phosphorus-...","transferase activity, transferring phosphorus-...",Danio rerio,NaN,ZFIN_ID,Quick_GO,Danio rerio


# Mouse

In [408]:
mouse_ncbi = pd.read_csv(f'{BASE_PATH}databases_for_mapping/ncbi/Mus_musculus.gene_info',sep = '\t')
# mouse_ncbi['LocusTag'] = mouse_ncbi['LocusTag'].str.replace('Dmel_', '', regex=False)
# Extract Zfin ID
mouse_ncbi['MGI_ID'] = (
    mouse_ncbi['dbXrefs']
    .str.extract(r'MGI:(MGI:\d+)')
)

# Keep only useful columns
mouse_ncbi_clean = mouse_ncbi[
    [
        'Symbol',
        'LocusTag',
        'MGI_ID',
        'description'
    ]
]

# # Remove duplicates
mouse_ncbi_clean = mouse_ncbi_clean.drop_duplicates()

# # Remove rows with missing IDs
mouse_ncbi_clean = mouse_ncbi_clean[
    ~mouse_ncbi_clean['description'].isna()
]

mouse_ncbi_locustag_dict = dict(zip(mouse_ncbi_clean['MGI_ID'],mouse_ncbi_clean['Symbol']))
mouse_ncbi_locustag_desc_dict = dict(zip(mouse_ncbi_clean['Symbol'],mouse_ncbi_clean['description']))
mouse_ncbi_mgi_desc_dict = dict(zip(mouse_ncbi_clean['MGI_ID'],mouse_ncbi_clean['description']))
mouse_ncbi_locustag_desc_dict
mouse_ncbi_locustag_dict

,Symbol,LocusTag,MGI_ID,description
0,Pzp,-,MGI:87854,"PZP, alpha-2-macroglobulin like"
1,Aanat,-,MGI:1328365,arylalkylamine N-acetyltransferase
2,Aatk,-,MGI:1197518,apoptosis-associated tyrosine kinase
3,Abca1,-,MGI:99607,"ATP-binding cassette, sub-family A member 1"
4,Abca4,-,MGI:109424,"ATP-binding cassette, sub-family A member 4"
...,...,...,...,...
112351,ND6,-,NaN,ND6
112352,ATP8,-,NaN,ATP8
112353,ND3,-,NaN,ND3
112354,COX1,-,NaN,COX1


In [413]:
GENE_GENE_Mouse_Mouse= pd.read_csv(
    f'{Semi_processed}Mouse/GENE_GENE_Mouse_Mouse_MONARCH.csv'
)


# GENE_GENE_Mouse_Mouse['Head'] = (
#     GENE_GENE_Mouse_Mouse['Head']
#     .str.replace('WB:', '', regex=False)
# )

# GENE_GENE_Mouse_Mouse['Tail'] = (
#     GENE_GENE_Mouse_Mouse['Tail']
#     .str.replace('WB:', '', regex=False)
# )

# GENE_GENE_Mouse_Mouse['Head'] = (
#     GENE_GENE_Mouse_Mouse['Head']
#     .map(celegans_ncbi_locustag_dict)
# )

# GENE_GENE_Mouse_Mouse['Tail'] = (
#     GENE_GENE_Mouse_Mouse['Tail']
#     .map(celegans_ncbi_locustag_dict)
# )

GENE_GENE_Mouse_Mouse['Head_Detail_name'] = (
    GENE_GENE_Mouse_Mouse['Head']
    .map(mouse_ncbi_locustag_desc_dict)
)

GENE_GENE_Mouse_Mouse['Tail_Detail_name'] = (
    GENE_GENE_Mouse_Mouse['Tail']
    .map(mouse_ncbi_locustag_desc_dict)
)

GENE_GENE_Mouse_Mouse = (
    GENE_GENE_Mouse_Mouse.rename(columns={
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
GENE_GENE_Mouse_Mouse = (
    GENE_GENE_Mouse_Mouse.loc[
        :,
        ~GENE_GENE_Mouse_Mouse.columns.duplicated()
    ]
)

# Keep only desired columns that exist
GENE_GENE_Mouse_Mouse = (
    select_cols(GENE_GENE_Mouse_Mouse)
)

GENE_GENE_Mouse_Mouse['Head_ID_IS'] = 'NCBI_ID'
GENE_GENE_Mouse_Mouse['Tail_ID_IS'] = 'NCBI_ID'

GENE_GENE_Mouse_Mouse['Relation'] = (
    'Gene_Gene'
)

GENE_GENE_Mouse_Mouse['Species'] = (
    'Mus musculus'
)

save(GENE_GENE_Mouse_Mouse,'Mouse/GENE_GENE_Mouse_Mouse.csv')

GENE_GENE_Mouse_Mouse

Saved 263,909 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Mouse/GENE_GENE_Mouse_Mouse.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,KG_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Tail_name,Species
0,Gnai3,Gene_Gene,Rgs4,Gene,interacts_with,Gene,infores:string,Monarch_KG,G protein subunit alpha i3,regulator of G-protein signaling 4,Mus musculus,Mus musculus,NCBI_ID,NCBI_ID,MGI:95773,MGI:108409,Mus musculus
1,Gnai3,Gene_Gene,Gnb4,Gene,interacts_with,Gene,infores:string,Monarch_KG,G protein subunit alpha i3,guanine nucleotide binding protein (G protein)...,Mus musculus,Mus musculus,NCBI_ID,NCBI_ID,MGI:95773,MGI:104581,Mus musculus
2,Gnai3,Gene_Gene,Gnaq,Gene,interacts_with,Gene,infores:string,Monarch_KG,G protein subunit alpha i3,"guanine nucleotide binding protein, alpha q po...",Mus musculus,Mus musculus,NCBI_ID,NCBI_ID,MGI:95773,MGI:95776,Mus musculus
3,Gnai3,Gene_Gene,Nucb1,Gene,interacts_with,Gene,infores:string,Monarch_KG,G protein subunit alpha i3,nucleobindin 1,Mus musculus,Mus musculus,NCBI_ID,NCBI_ID,MGI:95773,MGI:97388,Mus musculus
4,Gnai3,Gene_Gene,Htr1f,Gene,interacts_with,Gene,infores:string,Monarch_KG,G protein subunit alpha i3,5-hydroxytryptamine (serotonin) receptor 1F,Mus musculus,Mus musculus,NCBI_ID,NCBI_ID,MGI:95773,MGI:99842,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263904,Spop,Gene_Gene,Spop,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,speckle-type BTB/POZ protein,speckle-type BTB/POZ protein,Mus musculus,Mus musculus,NCBI_ID,NCBI_ID,MGI:1343085,MGI:1343085,Mus musculus
263905,Pak1,Gene_Gene,Rac1,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,p21 (RAC1) activated kinase 1,Rac family small GTPase 1,Mus musculus,Mus musculus,NCBI_ID,NCBI_ID,MGI:1339975,MGI:97845,Mus musculus
263906,Wdr4,Gene_Gene,Arhgap17,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,WD repeat domain 4,Rho GTPase activating protein 17,Mus musculus,Mus musculus,NCBI_ID,NCBI_ID,MGI:1889002,MGI:1917747,Mus musculus
263907,Nhlrc1,Gene_Gene,Epm2a,Gene,interacts_with,Gene,infores:biogrid,Monarch_KG,NHL repeat containing 1,"epilepsy, progressive myoclonic epilepsy, type...",Mus musculus,Mus musculus,NCBI_ID,NCBI_ID,MGI:2145264,MGI:1341085,Mus musculus


In [417]:
Gene_Mouse_BiologicalProcess = pd.read_csv(
    f'{Semi_processed}Mouse/Gene_Mouse_BiologicalProcess_MONARCH.csv'
)

Gene_Mouse_BiologicalProcess['Head'] = (
    Gene_Mouse_BiologicalProcess['Head']
    .map(mouse_ncbi_locustag_dict)
)


Gene_Mouse_BiologicalProcess['Head_Detail_name'] = (
    Gene_Mouse_BiologicalProcess['Head']
    .map(mouse_ncbi_locustag_desc_dict)
)

Gene_Mouse_BiologicalProcess = (
    Gene_Mouse_BiologicalProcess.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Mouse_BiologicalProcess = (
    Gene_Mouse_BiologicalProcess.loc[
        :,
        ~Gene_Mouse_BiologicalProcess.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Mouse_BiologicalProcess = (
    select_cols(Gene_Mouse_BiologicalProcess)
)

Gene_Mouse_BiologicalProcess['Head_ID_IS'] = 'NCBI_ID'
Gene_Mouse_BiologicalProcess['Tail_ID_IS'] = 'Quick_GO'

Gene_Mouse_BiologicalProcess['Relation'] = (
    'Gene_BiologicalProcess'
)

Gene_Mouse_BiologicalProcess['Tail_type'] = (
    'BiologicalProcess'
)

Gene_Mouse_BiologicalProcess['Species'] = (
    'Mus musculus'
)

save(
    Gene_Mouse_BiologicalProcess,
    'Mouse/Gene_Mouse_BiologicalProcess.csv'
)

Gene_Mouse_BiologicalProcess

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,Ptk2b,Gene_BiologicalProcess,GO:0045860,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,PTK2 protein tyrosine kinase 2 beta,positive regulation of protein kinase activity,Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
1,Ptk2b,Gene_BiologicalProcess,GO:0046330,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,PTK2 protein tyrosine kinase 2 beta,positive regulation of JNK cascade,Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
2,Ptk2b,Gene_BiologicalProcess,GO:0048010,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,PTK2 protein tyrosine kinase 2 beta,vascular endothelial growth factor receptor si...,Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
3,Ptk2b,Gene_BiologicalProcess,GO:0048041,Gene,actively_involved_in,BiologicalProcess,infores:ensembl,PTK2 protein tyrosine kinase 2 beta,focal adhesion assembly,Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
4,Ptk2b,Gene_BiologicalProcess,GO:0051279,Gene,actively_involved_in,BiologicalProcess,infores:uniprot,PTK2 protein tyrosine kinase 2 beta,regulation of release of sequestered calcium i...,Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257610,4930559C10Rik,Gene_BiologicalProcess,GO:0008150,Gene,actively_involved_in,BiologicalProcess,infores:mgi,RIKEN cDNA 4930559C10 gene,biological_process,Mus musculus,NaN,NCBI_ID,Quick_GO,4930559C10Rik,Mus musculus
257611,4930554N03Rik,Gene_BiologicalProcess,GO:0008150,Gene,actively_involved_in,BiologicalProcess,infores:mgi,RIKEN cDNA 4930554N03 gene,biological_process,Mus musculus,NaN,NCBI_ID,Quick_GO,4930554N03Rik,Mus musculus
257612,4930547M16Rik,Gene_BiologicalProcess,GO:0008150,Gene,actively_involved_in,BiologicalProcess,infores:mgi,RIKEN cDNA 4930547M16 gene,biological_process,Mus musculus,NaN,NCBI_ID,Quick_GO,4930547M16Rik,Mus musculus
257613,4930548F15Rik,Gene_BiologicalProcess,GO:0008150,Gene,actively_involved_in,BiologicalProcess,infores:mgi,RIKEN cDNA 4930548F15 gene,biological_process,Mus musculus,NaN,NCBI_ID,Quick_GO,4930548F15Rik,Mus musculus


In [419]:
Gene_Mouse_CellularComponent = pd.read_csv(
    f'{Semi_processed}Mouse/Gene_Mouse_CellularComponent_MONARCH.csv'
)

Gene_Mouse_CellularComponent['Head'] = (
    Gene_Mouse_CellularComponent['Head']
    .map(mouse_ncbi_locustag_dict)
)

Gene_Mouse_CellularComponent['Head_Detail_name'] = (
    Gene_Mouse_CellularComponent['Head']
    .map(mouse_ncbi_locustag_desc_dict)
)

Gene_Mouse_CellularComponent = (
    Gene_Mouse_CellularComponent.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Mouse_CellularComponent = (
    Gene_Mouse_CellularComponent.loc[
        :,
        ~Gene_Mouse_CellularComponent.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Mouse_CellularComponent = (
    select_cols(Gene_Mouse_CellularComponent)
)

Gene_Mouse_CellularComponent['Head_ID_IS'] = 'NCBI_ID'
Gene_Mouse_CellularComponent['Tail_ID_IS'] = 'Quick_GO'

Gene_Mouse_CellularComponent['Relation'] = (
    'Gene_CellularComponent'
)

Gene_Mouse_CellularComponent['Tail_type'] = (
    'CellularComponent'
)

Gene_Mouse_CellularComponent['Species'] = (
    'Mus musculus'
)

save(
    Gene_Mouse_CellularComponent,
    'Mouse/Gene_Mouse_CellularComponent.csv'
)

Gene_Mouse_CellularComponent

Saved 174,588 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Mouse/Gene_Mouse_CellularComponent.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,Ptk2b,Gene_CellularComponent,GO:0017146,Gene,part_of,CellularComponent,infores:alzheimers-university-of-toronto,PTK2 protein tyrosine kinase 2 beta,NMDA selective glutamate receptor complex,Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
1,Ptk2b,Gene_CellularComponent,GO:0098793,Gene,is_active_in,CellularComponent,infores:syngo,PTK2 protein tyrosine kinase 2 beta,presynapse,Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
2,Ptk2b,Gene_CellularComponent,GO:0098978,Gene,is_active_in,CellularComponent,infores:syngo,PTK2 protein tyrosine kinase 2 beta,glutamatergic synapse,Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
3,Ptk2b,Gene_CellularComponent,GO:0098978,Gene,is_active_in,CellularComponent,infores:syngo,PTK2 protein tyrosine kinase 2 beta,glutamatergic synapse,Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
4,Ptk2b,Gene_CellularComponent,GO:0099092,Gene,is_active_in,CellularComponent,infores:syngo,PTK2 protein tyrosine kinase 2 beta,"postsynaptic density, intracellular component",Mus musculus,NaN,NCBI_ID,Quick_GO,Ptk2b,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174583,4930554N03Rik,Gene_CellularComponent,GO:0005575,Gene,is_active_in,CellularComponent,infores:mgi,RIKEN cDNA 4930554N03 gene,cellular_component,Mus musculus,NaN,NCBI_ID,Quick_GO,4930554N03Rik,Mus musculus
174584,4930547M16Rik,Gene_CellularComponent,GO:0005575,Gene,is_active_in,CellularComponent,infores:mgi,RIKEN cDNA 4930547M16 gene,cellular_component,Mus musculus,NaN,NCBI_ID,Quick_GO,4930547M16Rik,Mus musculus
174585,4930548F15Rik,Gene_CellularComponent,GO:0005575,Gene,is_active_in,CellularComponent,infores:mgi,RIKEN cDNA 4930548F15 gene,cellular_component,Mus musculus,NaN,NCBI_ID,Quick_GO,4930548F15Rik,Mus musculus
174586,4930556N09Rik,Gene_CellularComponent,GO:0005575,Gene,is_active_in,CellularComponent,infores:mgi,RIKEN cDNA 4930556N09 gene,cellular_component,Mus musculus,NaN,NCBI_ID,Quick_GO,4930556N09Rik,Mus musculus


In [421]:
Gene_Mouse_MolecularActivity = pd.read_csv(
    f'{Semi_processed}Mouse/Gene_Mouse_MolecularActivity_MONARCH.csv'
)

Gene_Mouse_MolecularActivity['Head'] = (
    Gene_Mouse_MolecularActivity['Head']
    .map(mouse_ncbi_locustag_dict)
)

Gene_Mouse_MolecularActivity['Head_Detail_name'] = (
    Gene_Mouse_MolecularActivity['Head']
    .map(mouse_ncbi_locustag_desc_dict)
)

Gene_Mouse_MolecularActivity = (
    Gene_Mouse_MolecularActivity.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Mouse_MolecularActivity = (
    Gene_Mouse_MolecularActivity.loc[
        :,
        ~Gene_Mouse_MolecularActivity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Mouse_MolecularActivity = (
    select_cols(Gene_Mouse_MolecularActivity)
)

Gene_Mouse_MolecularActivity['Head_ID_IS'] = 'NCBI_ID'
Gene_Mouse_MolecularActivity['Tail_ID_IS'] = 'Quick_GO'

Gene_Mouse_MolecularActivity['Relation'] = (
    'Gene_MolecularFunction'
)

Gene_Mouse_MolecularActivity['Tail_type'] = (
    'MolecularFunction'
)

Gene_Mouse_MolecularActivity['Species'] = ('Mus musculus')

save(Gene_Mouse_MolecularActivity,'Mouse/Gene_Mouse_MolecularFunction.csv')

Gene_Mouse_MolecularActivity

Saved 162,670 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Mouse/Gene_Mouse_MolecularFunction.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,Chaf1a,Gene_MolecularFunction,GO:0005515,Gene,enables,MolecularFunction,infores:intact,"chromatin assembly factor 1, subunit A",protein binding,Mus musculus,NaN,NCBI_ID,Quick_GO,Chaf1a,Mus musculus
1,Chaf1a,Gene_MolecularFunction,GO:0005515,Gene,enables,MolecularFunction,infores:intact,"chromatin assembly factor 1, subunit A",protein binding,Mus musculus,NaN,NCBI_ID,Quick_GO,Chaf1a,Mus musculus
2,Chaf1a,Gene_MolecularFunction,GO:0042802,Gene,enables,MolecularFunction,infores:ensembl,"chromatin assembly factor 1, subunit A",identical protein binding,Mus musculus,NaN,NCBI_ID,Quick_GO,Chaf1a,Mus musculus
3,Chaf1a,Gene_MolecularFunction,GO:0070087,Gene,enables,MolecularFunction,infores:ensembl,"chromatin assembly factor 1, subunit A",chromo shadow domain binding,Mus musculus,NaN,NCBI_ID,Quick_GO,Chaf1a,Mus musculus
4,Sult1b1,Gene_MolecularFunction,GO:0004062,Gene,enables,MolecularFunction,infores:uniprot,"sulfotransferase family 1B, member 1",aryl sulfotransferase activity,Mus musculus,NaN,NCBI_ID,Quick_GO,Sult1b1,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162665,4930547M16Rik,Gene_MolecularFunction,GO:0003674,Gene,enables,MolecularFunction,infores:mgi,RIKEN cDNA 4930547M16 gene,molecular_function,Mus musculus,NaN,NCBI_ID,Quick_GO,4930547M16Rik,Mus musculus
162666,4930548F15Rik,Gene_MolecularFunction,GO:0003674,Gene,enables,MolecularFunction,infores:mgi,RIKEN cDNA 4930548F15 gene,molecular_function,Mus musculus,NaN,NCBI_ID,Quick_GO,4930548F15Rik,Mus musculus
162667,4930548F15Rik,Gene_MolecularFunction,GO:0003674,Gene,enables,MolecularFunction,infores:mgi,RIKEN cDNA 4930548F15 gene,molecular_function,Mus musculus,NaN,NCBI_ID,Quick_GO,4930548F15Rik,Mus musculus
162668,4930556N09Rik,Gene_MolecularFunction,GO:0003674,Gene,enables,MolecularFunction,infores:mgi,RIKEN cDNA 4930556N09 gene,molecular_function,Mus musculus,NaN,NCBI_ID,Quick_GO,4930556N09Rik,Mus musculus


In [424]:
Gene_Mouse_Phenotype = pd.read_csv(
    f'{Semi_processed}Mouse/Gene_Mouse_Phenotype_MONARCH.csv'
)

Gene_Mouse_Phenotype['Head'] = (
    Gene_Mouse_Phenotype['Head']
    .map(mouse_ncbi_locustag_dict)
)

Gene_Mouse_Phenotype['Head_Detail_name'] = (
    Gene_Mouse_Phenotype['Head']
    .map(mouse_ncbi_locustag_desc_dict)
)

Gene_Mouse_Phenotype = (
    Gene_Mouse_Phenotype.rename(columns={
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Gene_Mouse_Phenotype = (
    Gene_Mouse_Phenotype.loc[
        :,
        ~Gene_Mouse_Phenotype.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Gene_Mouse_Phenotype = (
    select_cols(Gene_Mouse_Phenotype)
)

Gene_Mouse_Phenotype['Head_ID_IS'] = 'NCBI_ID'
Gene_Mouse_Phenotype['Tail_ID_IS'] = 'MGI'

Gene_Mouse_Phenotype['Relation'] = (
    'Gene_Phenotype'
)

Gene_Mouse_Phenotype['Tail_type'] = (
    'Phenotype'
)

Gene_Mouse_Phenotype['Species'] = (
    'Mus musculus'
)

save(Gene_Mouse_Phenotype,'Mouse/Gene_Mouse_Phenotype.csv')

Gene_Mouse_Phenotype

Saved 286,524 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Mouse/Gene_Mouse_Phenotype.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Head_name,Species
0,Gabarapl2,Gene_Phenotype,MP:0005016,Gene,has_phenotype,Phenotype,infores:mgi,GABA type A receptor associated protein like 2,decreased lymphocyte cell number,Mus musculus,NaN,NCBI_ID,MGI,Gabarapl2,Mus musculus
1,Gabarapl2,Gene_Phenotype,MP:0005419,Gene,has_phenotype,Phenotype,infores:mgi,GABA type A receptor associated protein like 2,decreased circulating serum albumin level,Mus musculus,NaN,NCBI_ID,MGI,Gabarapl2,Mus musculus
2,Gabarapl2,Gene_Phenotype,MP:0011100,Gene,has_phenotype,Phenotype,infores:mgi,GABA type A receptor associated protein like 2,"preweaning lethality, complete penetrance",Mus musculus,NaN,NCBI_ID,MGI,Gabarapl2,Mus musculus
3,Osr2,Gene_Phenotype,MP:0000030,Gene,has_phenotype,Phenotype,infores:mgi,odd-skipped related 2,abnormal tympanic ring morphology,Mus musculus,NaN,NCBI_ID,MGI,Osr2,Mus musculus
4,Osr2,Gene_Phenotype,MP:0000030,Gene,has_phenotype,Phenotype,infores:mgi,odd-skipped related 2,abnormal tympanic ring morphology,Mus musculus,NaN,NCBI_ID,MGI,Osr2,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286519,Msh3,Gene_Phenotype,MP:0002135,Gene,has_phenotype,Phenotype,infores:mgi,mutS homolog 3,abnormal kidney morphology,Mus musculus,NaN,NCBI_ID,MGI,Msh3,Mus musculus
286520,Msh3,Gene_Phenotype,MP:0002699,Gene,has_phenotype,Phenotype,infores:mgi,mutS homolog 3,abnormal vitreous body morphology,Mus musculus,NaN,NCBI_ID,MGI,Msh3,Mus musculus
286521,Msh3,Gene_Phenotype,MP:0003068,Gene,has_phenotype,Phenotype,infores:mgi,mutS homolog 3,enlarged kidney,Mus musculus,NaN,NCBI_ID,MGI,Msh3,Mus musculus
286522,Htr2a,Gene_Phenotype,MP:0000479,Gene,has_phenotype,Phenotype,infores:mgi,5-hydroxytryptamine (serotonin) receptor 2A,abnormal enterocyte morphology,Mus musculus,NaN,NCBI_ID,MGI,Htr2a,Mus musculus


In [429]:
Mouse_AnatomicalEntity_AnatomicalEntity = pd.read_csv(
    f'{Semi_processed}Mouse/Mouse_AnatomicalEntity_AnatomicalEntity_MONARCH.csv'
)

Mouse_AnatomicalEntity_AnatomicalEntity = (
    Mouse_AnatomicalEntity_AnatomicalEntity.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Mouse_AnatomicalEntity_AnatomicalEntity = (
    Mouse_AnatomicalEntity_AnatomicalEntity.loc[
        :,
        ~Mouse_AnatomicalEntity_AnatomicalEntity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Mouse_AnatomicalEntity_AnatomicalEntity = (
    select_cols(Mouse_AnatomicalEntity_AnatomicalEntity)
)

Mouse_AnatomicalEntity_AnatomicalEntity['Head_ID_IS'] = 'EMAPA'
Mouse_AnatomicalEntity_AnatomicalEntity['Tail_ID_IS'] = 'EMAPA'

Mouse_AnatomicalEntity_AnatomicalEntity['Relation'] = (
    'Anatomy_Anatomy'
)
Mouse_AnatomicalEntity_AnatomicalEntity['Head_type'] = (
    'Anatomy'
)
Mouse_AnatomicalEntity_AnatomicalEntity['Tail_type'] = (
    'Anatomy'
)

Mouse_AnatomicalEntity_AnatomicalEntity['Species'] = (
    'Mus musculus'
)

save(
    Mouse_AnatomicalEntity_AnatomicalEntity,
    'Mouse/Mouse_AnatomicalEntity_AnatomicalEntity.csv'
)

Mouse_AnatomicalEntity_AnatomicalEntity

Saved 4,579 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Mouse/Mouse_AnatomicalEntity_AnatomicalEntity.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,EMAPA:16032,Anatomy_Anatomy,EMAPA:31859,Anatomy,subclass_of,Anatomy,infores:emapa,first polar body,polar body,Mus musculus,Mus musculus,EMAPA,EMAPA,Mus musculus
1,EMAPA:16033,Anatomy_Anatomy,EMAPA:36032,Anatomy,subclass_of,Anatomy,infores:emapa,1-cell stage embryo,early embryo,Mus musculus,Mus musculus,EMAPA,EMAPA,Mus musculus
2,EMAPA:16034,Anatomy_Anatomy,EMAPA:31859,Anatomy,subclass_of,Anatomy,infores:emapa,second polar body,polar body,Mus musculus,Mus musculus,EMAPA,EMAPA,Mus musculus
3,EMAPA:16036,Anatomy_Anatomy,EMAPA:36032,Anatomy,subclass_of,Anatomy,infores:emapa,2-cell stage embryo,early embryo,Mus musculus,Mus musculus,EMAPA,EMAPA,Mus musculus
4,EMAPA:16037,Anatomy_Anatomy,EMAPA:36032,Anatomy,subclass_of,Anatomy,infores:emapa,4-8 cell stage embryo,early embryo,Mus musculus,Mus musculus,EMAPA,EMAPA,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4574,EMAPA:38234,Anatomy_Anatomy,EMAPA:35263,Anatomy,subclass_of,Anatomy,infores:emapa,cranial suture mesenchyme,cranial suture,Mus musculus,Mus musculus,EMAPA,EMAPA,Mus musculus
4575,EMAPA:38235,Anatomy_Anatomy,EMAPA:35263,Anatomy,subclass_of,Anatomy,infores:emapa,cranial suture osteogenic front,cranial suture,Mus musculus,Mus musculus,EMAPA,EMAPA,Mus musculus
4576,EMAPA:38236,Anatomy_Anatomy,EMAPA:38235,Anatomy,subclass_of,Anatomy,infores:emapa,coronal suture osteogenic front,cranial suture osteogenic front,Mus musculus,Mus musculus,EMAPA,EMAPA,Mus musculus
4577,EMAPA:38237,Anatomy_Anatomy,EMAPA:38235,Anatomy,subclass_of,Anatomy,infores:emapa,lambdoid suture osteogenic front,cranial suture osteogenic front,Mus musculus,Mus musculus,EMAPA,EMAPA,Mus musculus


In [430]:
# Mouse_Genotype_Disease = pd.read_csv(
#     f'{Semi_processed}Mouse/Mouse_Genotype_Disease_Monarch.csv'
# )

In [433]:
Mouse_Phenotype_CellularComponent = pd.read_csv(
    f'{Semi_processed}Mouse/Mouse_Phenotype_CellularComponent_Monarch.csv'
)

Mouse_Phenotype_CellularComponent = (
    Mouse_Phenotype_CellularComponent.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Mouse_Phenotype_CellularComponent = (
    Mouse_Phenotype_CellularComponent.loc[
        :,
        ~Mouse_Phenotype_CellularComponent.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Mouse_Phenotype_CellularComponent = (
    select_cols(Mouse_Phenotype_CellularComponent)
)

Mouse_Phenotype_CellularComponent['Head_ID_IS'] = 'MGI'
Mouse_Phenotype_CellularComponent['Tail_ID_IS'] = 'Quick_GO'

Mouse_Phenotype_CellularComponent['Relation'] = (
    'Phenotype_CellularComponent'
)

Mouse_Phenotype_CellularComponent['Head_type'] = (
    'Phenotype'
)

Mouse_Phenotype_CellularComponent['Tail_type'] = (
    'CellularComponent'
)

Mouse_Phenotype_CellularComponent['Species'] = (
    'Mus musculus'
)

save(
    Mouse_Phenotype_CellularComponent,
    'Mouse/Mouse_Phenotype_CellularComponent.csv'
)

Mouse_Phenotype_CellularComponent

Saved 354 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Mouse/Mouse_Phenotype_CellularComponent.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,MP:0000731,Phenotype_CellularComponent,GO:0005581,Phenotype,related_to,CellularComponent,infores:upheno,increased collagen deposition in the muscles,collagen trimer,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
1,MP:0001053,Phenotype_CellularComponent,GO:0031594,Phenotype,related_to,CellularComponent,infores:upheno,abnormal neuromuscular synapse morphology,neuromuscular junction,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
2,MP:0001805,Phenotype_CellularComponent,GO:0071736,Phenotype,related_to,CellularComponent,infores:upheno,decreased IgG level,"IgG immunoglobulin complex, circulating",Mus musculus,NaN,MGI,Quick_GO,Mus musculus
3,MP:0001806,Phenotype_CellularComponent,GO:0071754,Phenotype,related_to,CellularComponent,infores:upheno,decreased IgM level,"IgM immunoglobulin complex, circulating",Mus musculus,NaN,MGI,Quick_GO,Mus musculus
4,MP:0001807,Phenotype_CellularComponent,GO:0071746,Phenotype,related_to,CellularComponent,infores:upheno,decreased IgA level,"IgA immunoglobulin complex, circulating",Mus musculus,NaN,MGI,Quick_GO,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
349,MP:0031388,Phenotype_CellularComponent,GO:0002177,Phenotype,related_to,CellularComponent,infores:upheno,absent manchette,manchette,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
350,MP:0031408,Phenotype_CellularComponent,GO:0061827,Phenotype,related_to,CellularComponent,infores:upheno,multi-headed sperm,sperm head,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
351,MP:0031409,Phenotype_CellularComponent,GO:0061827,Phenotype,related_to,CellularComponent,infores:upheno,double-headed sperm,sperm head,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
352,MP:0031410,Phenotype_CellularComponent,GO:0036126,Phenotype,related_to,CellularComponent,infores:upheno,biflagellated sperm,sperm flagellum,Mus musculus,NaN,MGI,Quick_GO,Mus musculus


In [434]:
Mouse_Phenotype_ChemicalEntity = pd.read_csv(
    f'{Semi_processed}Mouse/Mouse_Phenotype_ChemicalEntity_Monarch.csv'
)

Mouse_Phenotype_ChemicalEntity = (
    Mouse_Phenotype_ChemicalEntity.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Mouse_Phenotype_ChemicalEntity = (
    Mouse_Phenotype_ChemicalEntity.loc[
        :,
        ~Mouse_Phenotype_ChemicalEntity.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Mouse_Phenotype_ChemicalEntity = (
    select_cols(Mouse_Phenotype_ChemicalEntity)
)

Mouse_Phenotype_ChemicalEntity['Head_ID_IS'] = 'MGI'
Mouse_Phenotype_ChemicalEntity['Tail_ID_IS'] = 'PubChem'

Mouse_Phenotype_ChemicalEntity['Relation'] = (
    'Phenotype_ChemicalEntity'
)

Mouse_Phenotype_ChemicalEntity['Head_type'] = (
    'Phenotype'
)

Mouse_Phenotype_ChemicalEntity['Tail_type'] = (
    'ChemicalEntity'
)

Mouse_Phenotype_ChemicalEntity['Tail_IUPAC_name'] = (
    Mouse_Phenotype_ChemicalEntity['Tail']
    .astype(str)
    .map(Pubchem_IUPAC_CID_Dict)
)

Mouse_Phenotype_ChemicalEntity['Species'] = (
    'Mus musculus'
)

save(
    Mouse_Phenotype_ChemicalEntity,
    'Mouse/Mouse_Phenotype_ChemicalEntity.csv'
)

Mouse_Phenotype_ChemicalEntity

Saved 451 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Mouse/Mouse_Phenotype_ChemicalEntity.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Tail_IUPAC_name,Species
0,MP:0000180,Phenotype_ChemicalEntity,5997,Phenotype,related_to,ChemicalEntity,infores:upheno,abnormal circulating cholesterol level,cholesterol,Mus musculus,NaN,MGI,PubChem,"(3S,8S,9S,10R,13R,14S,17R)-10,13-dimethyl-17-[...",Mus musculus
1,MP:0000194,Phenotype_ChemicalEntity,5460341,Phenotype,related_to,ChemicalEntity,infores:upheno,increased circulating calcium level,calcium atom,Mus musculus,NaN,MGI,PubChem,calcium,Mus musculus
2,MP:0000195,Phenotype_ChemicalEntity,5460341,Phenotype,related_to,ChemicalEntity,infores:upheno,decreased circulating calcium level,calcium atom,Mus musculus,NaN,MGI,PubChem,calcium,Mus musculus
3,MP:0000362,Phenotype_ChemicalEntity,774,Phenotype,related_to,ChemicalEntity,infores:upheno,decreased mast cell histamine storage,histamine,Mus musculus,NaN,MGI,PubChem,2-(1H-imidazol-5-yl)ethanamine,Mus musculus
4,MP:0000676,Phenotype_ChemicalEntity,962,Phenotype,related_to,ChemicalEntity,infores:upheno,abnormal body water content,water,Mus musculus,NaN,MGI,PubChem,oxidane,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
446,MP:0031655,Phenotype_ChemicalEntity,5462224,Phenotype,related_to,ChemicalEntity,infores:upheno,abnormal brain magnesium level,magnesium atom,Mus musculus,NaN,MGI,PubChem,magnesium,Mus musculus
447,MP:0031656,Phenotype_ChemicalEntity,5462224,Phenotype,related_to,ChemicalEntity,infores:upheno,decreased brain magnesium level,magnesium atom,Mus musculus,NaN,MGI,PubChem,magnesium,Mus musculus
448,MP:0031657,Phenotype_ChemicalEntity,5462224,Phenotype,related_to,ChemicalEntity,infores:upheno,increased brain magnesium level,magnesium atom,Mus musculus,NaN,MGI,PubChem,magnesium,Mus musculus
449,MP:0031658,Phenotype_ChemicalEntity,5462224,Phenotype,related_to,ChemicalEntity,infores:upheno,decreased magnesium level,magnesium atom,Mus musculus,NaN,MGI,PubChem,magnesium,Mus musculus


In [435]:
Mouse_PhenotypicFeature_BiologicalProcess = pd.read_csv(
    f'{Semi_processed}Mouse/Mouse_PhenotypicFeature_BiologicalProcess_Monarch.csv'
)

Mouse_PhenotypicFeature_BiologicalProcess = (
    Mouse_PhenotypicFeature_BiologicalProcess.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Mouse_PhenotypicFeature_BiologicalProcess = (
    Mouse_PhenotypicFeature_BiologicalProcess.loc[
        :,
        ~Mouse_PhenotypicFeature_BiologicalProcess.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Mouse_PhenotypicFeature_BiologicalProcess = (
    select_cols(Mouse_PhenotypicFeature_BiologicalProcess)
)

Mouse_PhenotypicFeature_BiologicalProcess['Head_ID_IS'] = 'MGI'
Mouse_PhenotypicFeature_BiologicalProcess['Tail_ID_IS'] = 'Quick_GO'

Mouse_PhenotypicFeature_BiologicalProcess['Relation'] = (
    'Phenotype_BiologicalProcess'
)

Mouse_PhenotypicFeature_BiologicalProcess['Head_type'] = (
    'Phenotype'
)

Mouse_PhenotypicFeature_BiologicalProcess['Tail_type'] = (
    'BiologicalProcess'
)

Mouse_PhenotypicFeature_BiologicalProcess['Species'] = (
    'Mus musculus'
)

save(
    Mouse_PhenotypicFeature_BiologicalProcess,
    'Mouse/Mouse_PhenotypicFeature_BiologicalProcess.csv'
)

Mouse_PhenotypicFeature_BiologicalProcess

Saved 1,493 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Mouse/Mouse_PhenotypicFeature_BiologicalProcess.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,MP:0000015,Phenotype_BiologicalProcess,GO:0043473,Phenotype,related_to,BiologicalProcess,infores:upheno,abnormal ear pigmentation,pigmentation,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
1,MP:0000052,Phenotype_BiologicalProcess,GO:0060185,Phenotype,related_to,BiologicalProcess,infores:upheno,early ear unfolding,outer ear unfolding,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
2,MP:0000054,Phenotype_BiologicalProcess,GO:0060186,Phenotype,related_to,BiologicalProcess,infores:upheno,delayed ear emergence,outer ear emergence,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
3,MP:0000060,Phenotype_BiologicalProcess,GO:0001503,Phenotype,related_to,BiologicalProcess,infores:upheno,delayed bone ossification,ossification,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
4,MP:0000064,Phenotype_BiologicalProcess,GO:0045453,Phenotype,related_to,BiologicalProcess,infores:upheno,failure of bone resorption,bone resorption,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1488,MP:0031633,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,decreased heart left ventricle muscle contract...,cardiac muscle contraction,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
1489,MP:0031634,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,decreased heart right ventricle muscle contrac...,cardiac muscle contraction,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
1490,MP:0031635,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,increased heart left ventricle muscle contract...,cardiac muscle contraction,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
1491,MP:0031636,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,increased heart right ventricle muscle contrac...,cardiac muscle contraction,Mus musculus,NaN,MGI,Quick_GO,Mus musculus


In [437]:
Mouse_PhenotypicFeature_BiologicalProcess = pd.read_csv(
    f'{Semi_processed}Mouse/Mouse_PhenotypicFeature_BiologicalProcess_Monarch.csv'
)

Mouse_PhenotypicFeature_BiologicalProcess = (
    Mouse_PhenotypicFeature_BiologicalProcess.rename(columns={
        'Head_name': 'Head_Detail_name',
        'Tail_name': 'Tail_Detail_name',
        'Head_type_clean': 'Head_type',
        'Tail_type_clean': 'Tail_type'
    })
)

# Remove duplicate columns
Mouse_PhenotypicFeature_BiologicalProcess = (
    Mouse_PhenotypicFeature_BiologicalProcess.loc[
        :,
        ~Mouse_PhenotypicFeature_BiologicalProcess.columns.duplicated()
    ]
)

# Keep only desired columns that exist
Mouse_PhenotypicFeature_BiologicalProcess = (
    select_cols(Mouse_PhenotypicFeature_BiologicalProcess)
)

Mouse_PhenotypicFeature_BiologicalProcess['Head_ID_IS'] = 'MGI'
Mouse_PhenotypicFeature_BiologicalProcess['Tail_ID_IS'] = 'Quick_GO'

Mouse_PhenotypicFeature_BiologicalProcess['Relation'] = (
    'Phenotype_BiologicalProcess'
)

Mouse_PhenotypicFeature_BiologicalProcess['Head_type'] = (
    'Phenotype'
)

Mouse_PhenotypicFeature_BiologicalProcess['Tail_type'] = (
    'BiologicalProcess'
)

Mouse_PhenotypicFeature_BiologicalProcess['Species'] = (
    'Mus musculus'
)

save(
    Mouse_PhenotypicFeature_BiologicalProcess,
    'Mouse/Mouse_PhenotypicFeature_BiologicalProcess.csv'
)

Mouse_PhenotypicFeature_BiologicalProcess

Saved 1,493 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/Monarch/Monarch_final/Mouse/Mouse_PhenotypicFeature_BiologicalProcess.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,Relation_Source,Head_Detail_name,Tail_Detail_name,Head_taxon_name,Tail_taxon_name,Head_ID_IS,Tail_ID_IS,Species
0,MP:0000015,Phenotype_BiologicalProcess,GO:0043473,Phenotype,related_to,BiologicalProcess,infores:upheno,abnormal ear pigmentation,pigmentation,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
1,MP:0000052,Phenotype_BiologicalProcess,GO:0060185,Phenotype,related_to,BiologicalProcess,infores:upheno,early ear unfolding,outer ear unfolding,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
2,MP:0000054,Phenotype_BiologicalProcess,GO:0060186,Phenotype,related_to,BiologicalProcess,infores:upheno,delayed ear emergence,outer ear emergence,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
3,MP:0000060,Phenotype_BiologicalProcess,GO:0001503,Phenotype,related_to,BiologicalProcess,infores:upheno,delayed bone ossification,ossification,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
4,MP:0000064,Phenotype_BiologicalProcess,GO:0045453,Phenotype,related_to,BiologicalProcess,infores:upheno,failure of bone resorption,bone resorption,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1488,MP:0031633,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,decreased heart left ventricle muscle contract...,cardiac muscle contraction,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
1489,MP:0031634,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,decreased heart right ventricle muscle contrac...,cardiac muscle contraction,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
1490,MP:0031635,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,increased heart left ventricle muscle contract...,cardiac muscle contraction,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
1491,MP:0031636,Phenotype_BiologicalProcess,GO:0060048,Phenotype,related_to,BiologicalProcess,infores:upheno,increased heart right ventricle muscle contrac...,cardiac muscle contraction,Mus musculus,NaN,MGI,Quick_GO,Mus musculus
